# Paper 3 — Baseline Modelling and Generalization

This notebook uses the frozen cohorts and feature definitions generated in
`01_data_audit_and_cohort_definition.ipynb`.

## 1. Study objective

This notebook evaluates how photovoltaic-efficiency prediction changes as
validation becomes progressively more independent of the training chemistry.

The provenance-clean OPV-DB benchmark is evaluated under:

1. random row-wise cross-validation;
2. publication-held-out cross-validation;
3. donor–acceptor-pair-held-out cross-validation;
4. donor-held-out cross-validation; and
5. acceptor-held-out cross-validation.

The initial objective is not to maximize predictive performance. It is to
determine how strongly apparent model performance depends on the definition
of an independent test set.

All datasets and preprocessing decisions used here were frozen during the
preceding audit notebook.

> **Public reproducibility copy.** Transient API-failure cells, local machine paths, personal contact input, and non-scientific warning output were removed or redacted. The scientific workflow, analysis code, and retained numerical/figure outputs are unchanged.


In [ ]:
# ------------------------------------------------------------------
# 1.1
# Environment and project paths
# ------------------------------------------------------------------

import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd

import sklearn


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nInterim directory exists:")
print(INTERIM_DIR.exists())

Python: 3.11.15
Platform: Windows-10-10.0.22000-SP0
NumPy: 2.4.6
pandas: 3.0.5
scikit-learn: 1.9.0

Project directory:
<PROJECT_DIR>

Interim directory exists:
True


## 1.2 Frozen OPV-DB benchmark

The provenance-clean OPV-DB cohort generated during the audit stage is loaded
without further filtering or modification.

This frozen cohort forms the basis of all subsequent structure–performance and
generalization experiments.

In [ ]:
# ------------------------------------------------------------------
# 1.2
# Load frozen OPV-DB benchmark
# ------------------------------------------------------------------

opv = pd.read_csv(
    INTERIM_DIR
    / "FINAL_opvdb_primary_cohort.csv"
)


print("OPV-DB cohort dimensions:")
print(opv.shape)

print("\nUnique records:")
print(opv["id"].nunique())

print("\nSource DOIs:")
print(opv["doi_norm"].nunique())

print("\nRequired modeling fields:")

required_opv_fields = [
    "id",
    "doi_norm",
    "donor",
    "acceptor",
    "donor_graph_smiles",
    "acceptor_graph_smiles",
    "pce",
]

for col in required_opv_fields:
    print(
        f"{col:25s}",
        "FOUND" if col in opv.columns else "MISSING"
    )


print("\nPCE distribution:")

display(
    opv["pce"]
    .describe()
    .to_frame(name="PCE")
    .round(3)
)

OPV-DB cohort dimensions:
(21590, 42)

Unique records:
21590

Source DOIs:
5439

Required modeling fields:
id                        FOUND
doi_norm                  FOUND
donor                     FOUND
acceptor                  FOUND
donor_graph_smiles        FOUND
acceptor_graph_smiles     FOUND
pce                       FOUND

PCE distribution:


,PCE
count,21590.000
mean,9.690
std,5.686
min,0.000
25%,4.640
50%,9.260
75%,15.200
max,21.630


## 1.3 Frozen chemical-generalization assignments

Grouped cross-validation assignments developed during the audit are loaded
rather than regenerated.

Reusing the frozen assignments ensures that validation definitions remain
independent of subsequent model choice and prevents favorable folds from being
selected after observing predictive performance.

In [ ]:
# ------------------------------------------------------------------
# 1.3
# Load frozen grouped-CV assignments
# ------------------------------------------------------------------

grouped_cv_path = (
    INTERIM_DIR
    / "opvdb_grouped_cv_assignments.csv"
)


print(
    "Grouped-CV file exists:",
    grouped_cv_path.exists()
)


grouped_cv = pd.read_csv(
    grouped_cv_path
)


print("\nGrouped-CV dimensions:")
print(grouped_cv.shape)


print("\nColumns:")
print(grouped_cv.columns.tolist())


display(
    grouped_cv.head(10)
)

Grouped-CV file exists: True

Grouped-CV dimensions:
(21590, 9)

Columns:
['id', 'doi_norm', 'donor_graph_smiles', 'acceptor_graph_smiles', 'graph_pair', 'fold_doi_held_out', 'fold_da_pair_held_out', 'fold_donor_held_out', 'fold_acceptor_held_out']


,id,doi_norm,donor_graph_smiles,acceptor_graph_smiles,graph_pair,fold_doi_held_out,fold_da_pair_held_out,fold_donor_held_out,fold_acceptor_held_out
0,1,10.1002/aenm.201703291,CCCCCCC(CCCC)Cc1ccc(-c2c3cc(-c4c(F)cc(-c5cc6c(...,CCCCCCCCCCC(CCCCCCCC)CN1C(=O)c2ccc3c4c(c(-c5cc...,('CCCCCCC(CCCC)Cc1ccc(-c2c3cc(-c4c(F)cc(-c5cc6...,1,1,2,0
1,2,10.1002/aenm.201703291,CCCCCCC(CCCC)Cc1ccc(-c2c3cc(-c4ccc(-c5cc6c(s5)...,CCCCCCCCCCC(CCCCCCCC)CN1C(=O)c2ccc3c4c(c(-c5cc...,('CCCCCCC(CCCC)Cc1ccc(-c2c3cc(-c4ccc(-c5cc6c(s...,1,0,1,0
2,4,10.1002/aenm.201901024,CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(CC)...,CCCCCCc1ccc(C2(c3ccc(CCCCCC)cc3)c3c(sc4cc(/C=C...,('CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(C...,1,2,1,2
3,5,10.1002/adma.201706816,CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(CC)...,CCCCCCc1ccc(C2(c3ccc(CCCCCC)cc3)c3c(sc4cc(C=C5...,('CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(C...,0,1,1,0
4,7,10.1002/aenm.202100839,CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(CC)...,CCCCCCCCC1(CCCCCCCC)c2cc3c(cc2-c2sc(-c4ccc(C=C...,('CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(C...,2,0,1,1
5,8,10.1002/aenm.201901024,CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(CC)...,CCCCCCCCc1c(/C=C2\C(=O)c3cc(Cl)c(Cl)cc3C2=C(C#...,('CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(C...,1,2,1,2
6,9,10.1002/adfm.202102361,CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(CC)...,CCCCCCCCCc1c(/C=C/C=C2\C(=O)c3cc(F)c(F)cc3C2=C...,('CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(C...,2,2,1,0
7,10,10.1002/adma.201707508,CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(CC)...,CCCCCCCCOc1c2sc3c(c2c(OCCCCCCCC)c2sc4c(c12)C(c...,('CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(C...,1,0,1,0
8,11,10.1002/aenm.201600742,CCCCCCCCOc1c(OCCCCCCCC)c(-c2ccc(-c3cc4c(-c5ccc...,CCCCCCc1ccc(C2(c3ccc(CCCCCC)cc3)c3cc4c(cc3-c3s...,('CCCCCCCCOc1c(OCCCCCCCC)c(-c2ccc(-c3cc4c(-c5c...,1,0,1,2
9,12,10.1002/adma.201807577,CCCCC(CC)Cc1ccc(-c2c3cc(-c4ccc(-c5c(F)c(F)c(-c...,CCCCCCCCCCCc1c(/C=C2\C(=O)c3ccccc3C2=C(C#N)C#N...,('CCCCC(CC)Cc1ccc(-c2c3cc(-c4ccc(-c5c(F)c(F)c(...,0,2,2,1


In [ ]:
# ------------------------------------------------------------------
# 1.4
# Verify frozen cohort and CV assignment alignment
# ------------------------------------------------------------------

print("OPV cohort rows:", len(opv))
print("CV assignment rows:", len(grouped_cv))


if "id" not in grouped_cv.columns:
    print(
        "\nWARNING: no 'id' column found in grouped CV table."
    )

else:

    print(
        "\nUnique OPV IDs:",
        opv["id"].nunique()
    )

    print(
        "Unique CV IDs:",
        grouped_cv["id"].nunique()
    )


    opv_ids = set(
        opv["id"]
    )

    cv_ids = set(
        grouped_cv["id"]
    )


    print(
        "\nOPV IDs missing from CV assignments:",
        len(opv_ids - cv_ids)
    )

    print(
        "CV IDs absent from frozen OPV cohort:",
        len(cv_ids - opv_ids)
    )


    print(
        "Duplicated IDs in CV assignments:",
        grouped_cv["id"].duplicated().sum()
    )

OPV cohort rows: 21590
CV assignment rows: 21590

Unique OPV IDs: 21590
Unique CV IDs: 21590

OPV IDs missing from CV assignments: 0
CV IDs absent from frozen OPV cohort: 0
Duplicated IDs in CV assignments: 0


## 2. Validation Regimes

Model performance is evaluated under five three-fold validation regimes with
increasing forms of dataset independence.

The four grouped regimes were frozen during the data-audit stage and are
reused without modification:

1. publication-held-out;
2. donor–acceptor-pair-held-out;
3. donor-held-out; and
4. acceptor-held-out.

A shuffled random row-wise three-fold split is added as the conventional
interpolation reference.

The same molecular representation, target definition, model architecture, and
evaluation metrics will subsequently be used across all validation regimes.
Thus, changes in predictive performance can be attributed primarily to the
definition of test-set independence rather than changes in modeling procedure.

In [ ]:
# ------------------------------------------------------------------
# 2.1
# Combine frozen CV assignments with target
# and construct random three-fold reference
# ------------------------------------------------------------------

from sklearn.model_selection import KFold


cv_design = (
    grouped_cv
    .merge(
        opv[
            ["id", "pce"]
        ],
        on="id",
        how="left",
        validate="one_to_one"
    )
    .copy()
)


assert len(cv_design) == len(opv)
assert cv_design["pce"].notna().all()


# --------------------------------------------------------------
# Random row-wise 3-fold CV
# --------------------------------------------------------------

random_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


cv_design["fold_random"] = -1


for fold, (_, test_idx) in enumerate(
    random_cv.split(cv_design)
):

    cv_design.loc[
        test_idx,
        "fold_random"
    ] = fold


print(
    "Random fold values:",
    sorted(
        cv_design[
            "fold_random"
        ].unique()
    )
)

print(
    "Unassigned random rows:",
    int(
        (
            cv_design[
                "fold_random"
            ] < 0
        ).sum()
    )
)

Random fold values: [np.int64(0), np.int64(1), np.int64(2)]
Unassigned random rows: 0


In [ ]:
# ------------------------------------------------------------------
# 2.2
# Fold-size balance
# ------------------------------------------------------------------

validation_regimes = {
    "Random":
        "fold_random",

    "DOI held-out":
        "fold_doi_held_out",

    "D:A pair held-out":
        "fold_da_pair_held_out",

    "Donor held-out":
        "fold_donor_held_out",

    "Acceptor held-out":
        "fold_acceptor_held_out",
}


fold_size_rows = []


for regime_name, fold_col in validation_regimes.items():

    for fold in sorted(
        cv_design[
            fold_col
        ].unique()
    ):

        test_mask = (
            cv_design[
                fold_col
            ] == fold
        )

        fold_size_rows.append({

            "regime":
                regime_name,

            "fold":
                int(fold),

            "test_records":
                int(
                    test_mask.sum()
                ),

            "train_records":
                int(
                    (~test_mask).sum()
                ),

            "test_percent":
                100
                * test_mask.mean(),
        })


fold_size_summary = pd.DataFrame(
    fold_size_rows
)


display(
    fold_size_summary.round(2)
)

,regime,fold,test_records,train_records,test_percent
0,Random,0,7197,14393,33.33
1,Random,1,7197,14393,33.33
2,Random,2,7196,14394,33.33
3,DOI held-out,0,7197,14393,33.33
4,DOI held-out,1,7196,14394,33.33
5,DOI held-out,2,7197,14393,33.33
6,D:A pair held-out,0,7196,14394,33.33
7,D:A pair held-out,1,7197,14393,33.33
8,D:A pair held-out,2,7197,14393,33.33
9,Donor held-out,0,7197,14393,33.33


In [ ]:
# ------------------------------------------------------------------
# 2.3
# Target distribution by validation regime and fold
# ------------------------------------------------------------------

fold_target_rows = []


for regime_name, fold_col in validation_regimes.items():

    for fold in sorted(
        cv_design[
            fold_col
        ].unique()
    ):

        test = cv_design[
            cv_design[
                fold_col
            ] == fold
        ]


        fold_target_rows.append({

            "regime":
                regime_name,

            "fold":
                int(fold),

            "n":
                len(test),

            "pce_mean":
                test["pce"].mean(),

            "pce_median":
                test["pce"].median(),

            "pce_std":
                test["pce"].std(),

            "pce_min":
                test["pce"].min(),

            "pce_max":
                test["pce"].max(),
        })


fold_target_summary = pd.DataFrame(
    fold_target_rows
)


display(
    fold_target_summary.round(3)
)

,regime,fold,n,pce_mean,pce_median,pce_std,pce_min,pce_max
0,Random,0,7197,9.653,9.180,5.721,0.000,21.04
1,Random,1,7197,9.722,9.300,5.661,0.005,21.63
2,Random,2,7196,9.694,9.300,5.677,0.004,20.87
3,DOI held-out,0,7197,9.674,9.270,5.653,0.000,21.04
4,DOI held-out,1,7196,9.723,9.315,5.691,0.014,21.63
5,DOI held-out,2,7197,9.672,9.200,5.714,0.004,20.94
6,D:A pair held-out,0,7196,9.895,10.280,5.605,0.005,20.56
7,D:A pair held-out,1,7197,9.542,8.760,5.417,0.004,21.63
8,D:A pair held-out,2,7197,9.632,9.150,6.015,0.000,20.94
9,Donor held-out,0,7197,14.262,15.560,4.138,0.004,21.63


In [ ]:
# ------------------------------------------------------------------
# 2.4
# Independence check for grouped validation regimes
# ------------------------------------------------------------------

group_independence_checks = {
    "DOI held-out": (
        "fold_doi_held_out",
        "doi_norm"
    ),

    "D:A pair held-out": (
        "fold_da_pair_held_out",
        "graph_pair"
    ),

    "Donor held-out": (
        "fold_donor_held_out",
        "donor_graph_smiles"
    ),

    "Acceptor held-out": (
        "fold_acceptor_held_out",
        "acceptor_graph_smiles"
    ),
}


independence_rows = []


for regime_name, (
    fold_col,
    group_col
) in group_independence_checks.items():

    for fold in [0, 1, 2]:

        train = cv_design[
            cv_design[
                fold_col
            ] != fold
        ]

        test = cv_design[
            cv_design[
                fold_col
            ] == fold
        ]


        train_groups = set(
            train[
                group_col
            ].dropna()
        )

        test_groups = set(
            test[
                group_col
            ].dropna()
        )


        overlap = (
            train_groups
            &
            test_groups
        )


        independence_rows.append({

            "regime":
                regime_name,

            "fold":
                fold,

            "train_groups":
                len(train_groups),

            "test_groups":
                len(test_groups),

            "overlapping_groups":
                len(overlap),
        })


group_independence_summary = (
    pd.DataFrame(
        independence_rows
    )
)


display(
    group_independence_summary
)

,regime,fold,train_groups,test_groups,overlapping_groups
0,DOI held-out,0,3626,1813,0
1,DOI held-out,1,3626,1813,0
2,DOI held-out,2,3626,1813,0
3,D:A pair held-out,0,3238,1615,0
4,D:A pair held-out,1,3234,1619,0
5,D:A pair held-out,2,3234,1619,0
6,Donor held-out,0,1638,348,0
7,Donor held-out,1,1166,820,0
8,Donor held-out,2,1168,818,0
9,Acceptor held-out,0,957,479,0


In [ ]:
# ------------------------------------------------------------------
# 2.5
# Chemical and publication overlap in random CV
# ------------------------------------------------------------------

random_overlap_rows = []


overlap_variables = {
    "DOI":
        "doi_norm",

    "donor graph":
        "donor_graph_smiles",

    "acceptor graph":
        "acceptor_graph_smiles",

    "D:A graph pair":
        "graph_pair",
}


for fold in [0, 1, 2]:

    train = cv_design[
        cv_design[
            "fold_random"
        ] != fold
    ]

    test = cv_design[
        cv_design[
            "fold_random"
        ] == fold
    ]


    for identity_name, col in overlap_variables.items():

        train_values = set(
            train[col].dropna()
        )

        seen_mask = (
            test[col]
            .isin(
                train_values
            )
        )


        random_overlap_rows.append({

            "fold":
                fold,

            "identity":
                identity_name,

            "test_records":
                len(test),

            "seen_in_training":
                int(
                    seen_mask.sum()
                ),

            "percent_seen":
                100
                * seen_mask.mean(),
        })


random_overlap_summary = pd.DataFrame(
    random_overlap_rows
)


display(
    random_overlap_summary.round(2)
)

,fold,identity,test_records,seen_in_training,percent_seen
0,0,DOI,7197,6356,88.31
1,0,donor graph,7197,6796,94.43
2,0,acceptor graph,7197,6883,95.64
3,0,D:A graph pair,7197,5991,83.24
4,1,DOI,7197,6385,88.72
5,1,donor graph,7197,6824,94.82
6,1,acceptor graph,7197,6834,94.96
7,1,D:A graph pair,7197,5944,82.59
8,2,DOI,7196,6402,88.97
9,2,donor graph,7196,6717,93.34


In [ ]:
# ------------------------------------------------------------------
# Mean random-CV overlap
# ------------------------------------------------------------------

random_overlap_mean = (
    random_overlap_summary
    .groupby(
        "identity"
    )
    .agg(
        mean_percent_seen=(
            "percent_seen",
            "mean"
        ),

        min_percent_seen=(
            "percent_seen",
            "min"
        ),

        max_percent_seen=(
            "percent_seen",
            "max"
        ),
    )
    .reset_index()
)


display(
    random_overlap_mean.round(2)
)

,identity,mean_percent_seen,min_percent_seen,max_percent_seen
0,D:A graph pair,82.82,82.59,83.24
1,DOI,88.67,88.31,88.97
2,acceptor graph,95.41,94.96,95.64
3,donor graph,94.20,93.34,94.82


In [ ]:
# ------------------------------------------------------------------
# 2.6
# Freeze all five validation regimes
# ------------------------------------------------------------------

cv_design.to_csv(
    INTERIM_DIR
    / "FINAL_opvdb_five_regime_cv_assignments.csv",
    index=False
)


fold_size_summary.to_csv(
    INTERIM_DIR
    / "FINAL_opvdb_cv_fold_sizes.csv",
    index=False
)


fold_target_summary.to_csv(
    INTERIM_DIR
    / "FINAL_opvdb_cv_target_summary.csv",
    index=False
)


print(
    "Five-regime CV design saved."
)

print(
    "Records:",
    len(cv_design)
)

print(
    "Validation regimes:",
    len(validation_regimes)
)

Five-regime CV design saved.
Records: 21590
Validation regimes: 5


## 3. Controlled Molecular Representation

To isolate the effect of validation design from the effect of molecular
representation, all initial OPV-DB models use the same fixed structural
representation.

Canonical donor and acceptor molecular graphs are independently encoded using
radius-2 Morgan fingerprints with 2,048 bits per component. Donor and acceptor
fingerprints are concatenated to form a 4,096-dimensional device-level
representation.

Fingerprints are generated once for each unique canonical molecular graph and
then mapped to device records. No target information is used during
fingerprint construction.

This representation is intended as a controlled baseline rather than an
assumption that Morgan fingerprints provide an optimal description of
photovoltaic materials.

In [ ]:
# ------------------------------------------------------------------
# 3.1
# Verify canonical molecular graphs
# ------------------------------------------------------------------

graph_summary = pd.DataFrame({
    "component": [
        "donor",
        "acceptor"
    ],

    "records": [
        len(opv),
        len(opv)
    ],

    "missing_graphs": [
        opv["donor_graph_smiles"].isna().sum(),
        opv["acceptor_graph_smiles"].isna().sum(),
    ],

    "unique_graphs": [
        opv["donor_graph_smiles"].nunique(),
        opv["acceptor_graph_smiles"].nunique(),
    ],
})

display(graph_summary)

,component,records,missing_graphs,unique_graphs
0,donor,21590,0,1986
1,acceptor,21590,0,1436


In [ ]:
# ------------------------------------------------------------------
# 3.2
# Generate Morgan fingerprints for unique molecular graphs
# ------------------------------------------------------------------

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

FP_RADIUS = 2
FP_SIZE = 2048


morgan_generator = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=FP_RADIUS,
        fpSize=FP_SIZE
    )
)


def graph_to_fp_array(smiles):

    mol = Chem.MolFromSmiles(
        str(smiles)
    )

    if mol is None:
        return None

    fp = morgan_generator.GetFingerprint(
        mol
    )

    array = np.zeros(
        (FP_SIZE,),
        dtype=np.uint8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        array
    )

    return array


    # ------------------------------------------------------------------
# Unique donor/acceptor fingerprint dictionaries
# ------------------------------------------------------------------

unique_donors = (
    opv["donor_graph_smiles"]
    .dropna()
    .unique()
)

unique_acceptors = (
    opv["acceptor_graph_smiles"]
    .dropna()
    .unique()
)


donor_fp_map = {
    smiles: graph_to_fp_array(smiles)
    for smiles in unique_donors
}

acceptor_fp_map = {
    smiles: graph_to_fp_array(smiles)
    for smiles in unique_acceptors
}


donor_failures = [
    smiles
    for smiles, fp in donor_fp_map.items()
    if fp is None
]

acceptor_failures = [
    smiles
    for smiles, fp in acceptor_fp_map.items()
    if fp is None
]


print(
    "Unique donor graphs:",
    len(unique_donors)
)

print(
    "Unique acceptor graphs:",
    len(unique_acceptors)
)

print(
    "Donor fingerprint failures:",
    len(donor_failures)
)

print(
    "Acceptor fingerprint failures:",
    len(acceptor_failures)
)

Unique donor graphs: 1986
Unique acceptor graphs: 1436
Donor fingerprint failures: 0
Acceptor fingerprint failures: 0


In [ ]:
# ------------------------------------------------------------------
# 3.3
# Construct sparse concatenated donor + acceptor fingerprints
# ------------------------------------------------------------------

from scipy import sparse


donor_matrix = np.vstack(
    opv["donor_graph_smiles"]
    .map(donor_fp_map)
    .to_numpy()
)

acceptor_matrix = np.vstack(
    opv["acceptor_graph_smiles"]
    .map(acceptor_fp_map)
    .to_numpy()
)


X_opv = sparse.csr_matrix(
    np.hstack(
        [
            donor_matrix,
            acceptor_matrix
        ]
    ),
    dtype=np.float32
)


y_opv = (
    opv["pce"]
    .to_numpy(
        dtype=np.float64
    )
)


print(
    "OPV feature matrix:",
    X_opv.shape
)

print(
    "Target:",
    y_opv.shape
)

print(
    "Sparse nonzero entries:",
    f"{X_opv.nnz:,}"
)

print(
    "Matrix density:",
    round(
        X_opv.nnz
        / (
            X_opv.shape[0]
            * X_opv.shape[1]
        ),
        4
    )
)

OPV feature matrix: (21590, 4096)
Target: (21590,)
Sparse nonzero entries: 2,904,854
Matrix density: 0.0328


In [ ]:
# ------------------------------------------------------------------
# 3.4
# Representation integrity checks
# ------------------------------------------------------------------

print(
    "Missing target values:",
    int(
        np.isnan(y_opv).sum()
    )
)

print(
    "Infinite target values:",
    int(
        np.isinf(y_opv).sum()
    )
)

print(
    "Feature minimum:",
    X_opv.min()
)

print(
    "Feature maximum:",
    X_opv.max()
)


row_active_bits = np.asarray(
    X_opv.sum(axis=1)
).ravel()


representation_summary = pd.DataFrame({
    "metric": [
        "records",
        "features",
        "mean active bits",
        "median active bits",
        "minimum active bits",
        "maximum active bits",
    ],

    "value": [
        X_opv.shape[0],
        X_opv.shape[1],
        row_active_bits.mean(),
        np.median(row_active_bits),
        row_active_bits.min(),
        row_active_bits.max(),
    ]
})


display(
    representation_summary.round(2)
)

Missing target values: 0
Infinite target values: 0
Feature minimum: 0.0
Feature maximum: 1.0


,metric,value
0,records,21590.00
1,features,4096.00
2,mean active bits,134.55
3,median active bits,140.00
4,minimum active bits,10.00
5,maximum active bits,210.00


In [ ]:
# ------------------------------------------------------------------
# 3.5
# Cache molecular representation
# ------------------------------------------------------------------

opv_fp_path = (
    PROCESSED_DIR
    / "opvdb_morgan_r2_2048x2.npz"
)

opv_target_path = (
    PROCESSED_DIR
    / "opvdb_pce_target.npy"
)

opv_id_path = (
    PROCESSED_DIR
    / "opvdb_model_ids.npy"
)


sparse.save_npz(
    opv_fp_path,
    X_opv
)

np.save(
    opv_target_path,
    y_opv
)

np.save(
    opv_id_path,
    opv["id"].to_numpy()
)


print(
    "Fingerprint matrix saved:",
    opv_fp_path.exists()
)

print(
    "Target saved:",
    opv_target_path.exists()
)

print(
    "IDs saved:",
    opv_id_path.exists()
)

Fingerprint matrix saved: True
Target saved: True
IDs saved: True


## 4. Baseline Generalization Experiments

Initial predictive experiments use deliberately simple baselines to establish
the effect of validation design before model complexity or hyperparameter
optimization is introduced.

Two models are evaluated first:

1. a mean DummyRegressor, which provides a target-only reference; and
2. L2-regularized Ridge regression applied to the fixed concatenated molecular
   fingerprint representation.

Both models are evaluated under the same five frozen three-fold validation
regimes using mean absolute error (MAE), root mean squared error (RMSE), and
coefficient of determination (R²).

No validation regime receives separate feature engineering or
hyperparameter tuning.

In [ ]:
# ------------------------------------------------------------------
# 4.1
# Align feature matrix, target, IDs, and CV assignments
# ------------------------------------------------------------------

model_ids = np.load(
    PROCESSED_DIR
    / "opvdb_model_ids.npy"
)

print("Feature rows:", X_opv.shape[0])
print("Target rows :", len(y_opv))
print("Model IDs   :", len(model_ids))
print("CV rows     :", len(cv_design))


# Ensure cached matrix follows frozen OPV cohort order
assert np.array_equal(
    model_ids,
    opv["id"].to_numpy()
)


# Reorder CV assignments explicitly to model order
cv_for_model = (
    pd.DataFrame({
        "id": model_ids
    })
    .merge(
        cv_design,
        on="id",
        how="left",
        validate="one_to_one"
    )
)


assert len(cv_for_model) == X_opv.shape[0]


fold_columns = {
    "Random":
        "fold_random",

    "DOI held-out":
        "fold_doi_held_out",

    "D:A pair held-out":
        "fold_da_pair_held_out",

    "Donor held-out":
        "fold_donor_held_out",

    "Acceptor held-out":
        "fold_acceptor_held_out",
}


print("Alignment check passed.")
print("Records:", len(cv_for_model))

display(
    cv_for_model[
        [
            "id",
            *fold_columns.values()
        ]
    ].head()
)

Feature rows: 21590
Target rows : 21590
Model IDs   : 21590
CV rows     : 21590
Alignment check passed.
Records: 21590


,id,fold_random,fold_doi_held_out,fold_da_pair_held_out,fold_donor_held_out,fold_acceptor_held_out
0,1,0,1,1,2,0
1,2,2,1,0,1,0
2,4,1,1,2,1,2
3,5,0,0,1,1,0
4,7,2,2,0,1,1


In [ ]:
# ------------------------------------------------------------------
# 4.2
# Cross-validation evaluation helper
# ------------------------------------------------------------------

from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)


def evaluate_model_across_regimes(
    model,
    model_name,
    X,
    y,
    cv_table,
    regimes
):

    result_rows = []


    for regime_name, fold_col in regimes.items():

        print("\n" + "=" * 70)
        print(model_name, "—", regime_name)
        print("=" * 70)


        for fold in [0, 1, 2]:

            test_mask = (
                cv_table[
                    fold_col
                ].to_numpy()
                == fold
            )

            train_mask = ~test_mask


            X_train = X[
                train_mask
            ]

            X_test = X[
                test_mask
            ]

            y_train = y[
                train_mask
            ]

            y_test = y[
                test_mask
            ]


            estimator = clone(
                model
            )

            estimator.fit(
                X_train,
                y_train
            )

            prediction = estimator.predict(
                X_test
            )


            mae = mean_absolute_error(
                y_test,
                prediction
            )

            rmse = root_mean_squared_error(
                y_test,
                prediction
            )

            r2 = r2_score(
                y_test,
                prediction
            )


            result_rows.append({

                "model":
                    model_name,

                "regime":
                    regime_name,

                "fold":
                    fold,

                "train_n":
                    len(y_train),

                "test_n":
                    len(y_test),

                "test_pce_mean":
                    y_test.mean(),

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2,
            })


            print(
                f"Fold {fold}: "
                f"MAE={mae:.3f} | "
                f"RMSE={rmse:.3f} | "
                f"R2={r2:.3f}"
            )


    return pd.DataFrame(
        result_rows
    )

In [ ]:
# ------------------------------------------------------------------
# 4.3
# Target-only baseline
# ------------------------------------------------------------------

from sklearn.dummy import DummyRegressor


dummy_model = DummyRegressor(
    strategy="mean"
)


dummy_results = (
    evaluate_model_across_regimes(
        model=dummy_model,
        model_name="Dummy mean",
        X=X_opv,
        y=y_opv,
        cv_table=cv_for_model,
        regimes=fold_columns,
    )
)


display(
    dummy_results.round(3)
)


Dummy mean — Random
Fold 0: MAE=4.988 | RMSE=5.721 | R2=-0.000
Fold 1: MAE=4.919 | RMSE=5.660 | R2=-0.000
Fold 2: MAE=4.945 | RMSE=5.677 | R2=-0.000

Dummy mean — DOI held-out
Fold 0: MAE=4.916 | RMSE=5.653 | R2=-0.000
Fold 1: MAE=4.955 | RMSE=5.691 | R2=-0.000
Fold 2: MAE=4.982 | RMSE=5.714 | R2=-0.000

Dummy mean — D:A pair held-out
Fold 0: MAE=4.997 | RMSE=5.613 | R2=-0.003
Fold 1: MAE=4.580 | RMSE=5.421 | R2=-0.002
Fold 2: MAE=5.293 | RMSE=6.015 | R2=-0.000

Dummy mean — Donor held-out
Fold 0: MAE=7.395 | RMSE=8.010 | R2=-2.747
Fold 1: MAE=4.731 | RMSE=5.533 | R2=-0.119
Fold 2: MAE=5.808 | RMSE=6.660 | R2=-1.361

Dummy mean — Acceptor held-out
Fold 0: MAE=5.052 | RMSE=5.630 | R2=-0.006
Fold 1: MAE=6.091 | RMSE=6.799 | R2=-0.175
Fold 2: MAE=4.517 | RMSE=5.314 | R2=-0.502


,model,regime,fold,train_n,test_n,test_pce_mean,MAE,RMSE,R2
0,Dummy mean,Random,0,14393,7197,9.653,4.988,5.721,-0.000
1,Dummy mean,Random,1,14393,7197,9.722,4.919,5.660,-0.000
2,Dummy mean,Random,2,14394,7196,9.694,4.945,5.677,-0.000
3,Dummy mean,DOI held-out,0,14393,7197,9.674,4.916,5.653,-0.000
4,Dummy mean,DOI held-out,1,14394,7196,9.723,4.955,5.691,-0.000
5,Dummy mean,DOI held-out,2,14393,7197,9.672,4.982,5.714,-0.000
6,Dummy mean,D:A pair held-out,0,14394,7196,9.895,4.997,5.613,-0.003
7,Dummy mean,D:A pair held-out,1,14393,7197,9.542,4.580,5.421,-0.002
8,Dummy mean,D:A pair held-out,2,14393,7197,9.632,5.293,6.015,-0.000
9,Dummy mean,Donor held-out,0,14393,7197,14.262,7.395,8.010,-2.747


In [ ]:
# ------------------------------------------------------------------
# 4.4
# Linear molecular baseline
# ------------------------------------------------------------------

from sklearn.linear_model import Ridge


ridge_model = Ridge(
    alpha=1.0,
    fit_intercept=True,
    solver="lsqr"
)


ridge_results = (
    evaluate_model_across_regimes(
        model=ridge_model,
        model_name="Ridge",
        X=X_opv,
        y=y_opv,
        cv_table=cv_for_model,
        regimes=fold_columns,
    )
)


display(
    ridge_results.round(3)
)


Ridge — Random
Fold 0: MAE=1.888 | RMSE=2.649 | R2=0.786
Fold 1: MAE=1.879 | RMSE=2.583 | R2=0.792
Fold 2: MAE=1.874 | RMSE=2.586 | R2=0.792

Ridge — DOI held-out
Fold 0: MAE=2.131 | RMSE=2.922 | R2=0.733
Fold 1: MAE=2.173 | RMSE=3.024 | R2=0.718
Fold 2: MAE=2.175 | RMSE=2.961 | R2=0.731

Ridge — D:A pair held-out
Fold 0: MAE=2.391 | RMSE=3.121 | R2=0.690
Fold 1: MAE=2.140 | RMSE=2.854 | R2=0.722
Fold 2: MAE=2.133 | RMSE=2.926 | R2=0.763

Ridge — Donor held-out
Fold 0: MAE=5.999 | RMSE=6.506 | R2=-1.472
Fold 1: MAE=2.296 | RMSE=3.137 | R2=0.640
Fold 2: MAE=2.394 | RMSE=3.184 | R2=0.460

Ridge — Acceptor held-out
Fold 0: MAE=2.362 | RMSE=3.255 | R2=0.664
Fold 1: MAE=2.596 | RMSE=3.354 | R2=0.714
Fold 2: MAE=2.950 | RMSE=3.698 | R2=0.273


,model,regime,fold,train_n,test_n,test_pce_mean,MAE,RMSE,R2
0,Ridge,Random,0,14393,7197,9.653,1.888,2.649,0.786
1,Ridge,Random,1,14393,7197,9.722,1.879,2.583,0.792
2,Ridge,Random,2,14394,7196,9.694,1.874,2.586,0.792
3,Ridge,DOI held-out,0,14393,7197,9.674,2.131,2.922,0.733
4,Ridge,DOI held-out,1,14394,7196,9.723,2.173,3.024,0.718
5,Ridge,DOI held-out,2,14393,7197,9.672,2.175,2.961,0.731
6,Ridge,D:A pair held-out,0,14394,7196,9.895,2.391,3.121,0.690
7,Ridge,D:A pair held-out,1,14393,7197,9.542,2.140,2.854,0.722
8,Ridge,D:A pair held-out,2,14393,7197,9.632,2.133,2.926,0.763
9,Ridge,Donor held-out,0,14393,7197,14.262,5.999,6.506,-1.472


In [ ]:
# ------------------------------------------------------------------
# 4.5
# Aggregate baseline performance
# ------------------------------------------------------------------

baseline_fold_results = pd.concat(
    [
        dummy_results,
        ridge_results
    ],
    ignore_index=True
)


baseline_summary = (
    baseline_fold_results
    .groupby(
        ["model", "regime"]
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
    )
    .reset_index()
)


display(
    baseline_summary.round(3)
)

,model,regime,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,Dummy mean,Acceptor held-out,5.220,0.800,5.914,0.782,-0.228,0.252
1,Dummy mean,D:A pair held-out,4.957,0.359,5.683,0.303,-0.002,0.001
2,Dummy mean,DOI held-out,4.951,0.033,5.686,0.031,-0.000,0.000
3,Dummy mean,Donor held-out,5.978,1.340,6.734,1.240,-1.409,1.315
4,Dummy mean,Random,4.951,0.034,5.686,0.031,-0.000,0.000
5,Ridge,Acceptor held-out,2.636,0.296,3.436,0.232,0.550,0.242
6,Ridge,D:A pair held-out,2.221,0.147,2.967,0.138,0.725,0.037
7,Ridge,DOI held-out,2.160,0.025,2.969,0.052,0.727,0.008
8,Ridge,Donor held-out,3.563,2.110,4.276,1.931,-0.124,1.171
9,Ridge,Random,1.881,0.007,2.606,0.037,0.790,0.004


In [ ]:
# ------------------------------------------------------------------
# 4.6
# Generalization gap relative to random CV
# ------------------------------------------------------------------

ridge_summary = (
    baseline_summary[
        baseline_summary[
            "model"
        ] == "Ridge"
    ]
    .copy()
)


random_ridge = (
    ridge_summary[
        ridge_summary[
            "regime"
        ] == "Random"
    ]
    .iloc[0]
)


ridge_summary[
    "MAE_increase_vs_random"
] = (
    ridge_summary[
        "MAE_mean"
    ]
    -
    random_ridge[
        "MAE_mean"
    ]
)


ridge_summary[
    "RMSE_increase_vs_random"
] = (
    ridge_summary[
        "RMSE_mean"
    ]
    -
    random_ridge[
        "RMSE_mean"
    ]
)


ridge_summary[
    "R2_drop_vs_random"
] = (
    random_ridge[
        "R2_mean"
    ]
    -
    ridge_summary[
        "R2_mean"
    ]
)


display(
    ridge_summary[
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
            "R2_mean",
            "MAE_increase_vs_random",
            "RMSE_increase_vs_random",
            "R2_drop_vs_random",
        ]
    ]
    .round(3)
)

,regime,MAE_mean,RMSE_mean,R2_mean,MAE_increase_vs_random,RMSE_increase_vs_random,R2_drop_vs_random
5,Acceptor held-out,2.636,3.436,0.550,0.755,0.830,0.240
6,D:A pair held-out,2.221,2.967,0.725,0.341,0.361,0.065
7,DOI held-out,2.160,2.969,0.727,0.279,0.363,0.063
8,Donor held-out,3.563,4.276,-0.124,1.682,1.670,0.914
9,Random,1.881,2.606,0.790,0.000,0.000,0.000


In [ ]:
# ------------------------------------------------------------------
# 4.7
# Save baseline model results
# ------------------------------------------------------------------

baseline_fold_results.to_csv(
    PROCESSED_DIR
    / "opvdb_baseline_fold_results.csv",
    index=False
)


baseline_summary.to_csv(
    PROCESSED_DIR
    / "opvdb_baseline_summary.csv",
    index=False
)


ridge_summary.to_csv(
    PROCESSED_DIR
    / "opvdb_ridge_generalization_gap.csv",
    index=False
)


print("Baseline results saved.")

Baseline results saved.


### 4.8 Baseline generalization pattern

The first molecular baseline reveals a progressive deterioration in predictive
performance as validation becomes more chemically independent.

Publication-held-out and donor–acceptor-pair-held-out performance remains
relatively close to random cross-validation, whereas acceptor-held-out and
especially donor-held-out validation produces substantially larger errors.

The donor-held-out result is highly heterogeneous across folds. Therefore,
before introducing a more complex model, the chemical composition of the
difficult donor fold is examined to determine whether the apparent failure is
associated with a particular dominant donor domain or with uniformly poor
unseen-donor prediction.

In [ ]:
# ------------------------------------------------------------------
# 4.8
# Relative performance and skill versus Dummy baseline
# ------------------------------------------------------------------

dummy_summary = (
    baseline_summary[
        baseline_summary["model"] == "Dummy mean"
    ][
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
        ]
    ]
    .rename(
        columns={
            "MAE_mean": "dummy_MAE",
            "RMSE_mean": "dummy_RMSE",
        }
    )
)


ridge_interpretation = (
    ridge_summary
    .merge(
        dummy_summary,
        on="regime",
        how="left",
        validate="one_to_one"
    )
)


ridge_interpretation[
    "MAE_percent_increase_vs_random"
] = (
    100
    * ridge_interpretation[
        "MAE_increase_vs_random"
    ]
    / random_ridge["MAE_mean"]
)


ridge_interpretation[
    "MAE_skill_vs_dummy"
] = (
    1
    -
    ridge_interpretation["MAE_mean"]
    /
    ridge_interpretation["dummy_MAE"]
)


ridge_interpretation[
    "RMSE_skill_vs_dummy"
] = (
    1
    -
    ridge_interpretation["RMSE_mean"]
    /
    ridge_interpretation["dummy_RMSE"]
)


display(
    ridge_interpretation[
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
            "R2_mean",
            "R2_std",
            "MAE_percent_increase_vs_random",
            "MAE_skill_vs_dummy",
            "RMSE_skill_vs_dummy",
        ]
    ].round(3)
)

,regime,MAE_mean,RMSE_mean,R2_mean,R2_std,MAE_percent_increase_vs_random,MAE_skill_vs_dummy,RMSE_skill_vs_dummy
0,Acceptor held-out,2.636,3.436,0.550,0.242,40.159,0.495,0.419
1,D:A pair held-out,2.221,2.967,0.725,0.037,18.109,0.552,0.478
2,DOI held-out,2.160,2.969,0.727,0.008,14.838,0.564,0.478
3,Donor held-out,3.563,4.276,-0.124,1.171,89.444,0.404,0.365
4,Random,1.881,2.606,0.790,0.004,0.000,0.620,0.542


## 5. Chemical-Domain Diagnosis of Donor-Held-Out Validation

Donor-held-out validation exhibits substantially greater fold-to-fold
variability than the other validation regimes.

The difficult fold is therefore examined at the material-group level before
model complexity is increased.

This analysis tests whether poor donor-held-out performance reflects a
general inability to predict unseen donors or a particularly challenging
chemical domain that dominates one validation fold.

In [ ]:
# ------------------------------------------------------------------
# 5.1
# Donor-held-out train/test target shift
# ------------------------------------------------------------------

donor_fold_shift_rows = []


for fold in [0, 1, 2]:

    test_mask = (
        cv_for_model[
            "fold_donor_held_out"
        ].eq(fold)
    )

    train_mask = ~test_mask


    y_train_fold = y_opv[
        train_mask.to_numpy()
    ]

    y_test_fold = y_opv[
        test_mask.to_numpy()
    ]


    donor_fold_shift_rows.append({

        "fold":
            fold,

        "train_n":
            len(y_train_fold),

        "test_n":
            len(y_test_fold),

        "train_mean":
            y_train_fold.mean(),

        "test_mean":
            y_test_fold.mean(),

        "mean_shift":
            (
                y_test_fold.mean()
                - y_train_fold.mean()
            ),

        "train_median":
            np.median(y_train_fold),

        "test_median":
            np.median(y_test_fold),

        "median_shift":
            (
                np.median(y_test_fold)
                - np.median(y_train_fold)
            ),
    })


donor_fold_shift = pd.DataFrame(
    donor_fold_shift_rows
)

display(
    donor_fold_shift.round(3)
)

,fold,train_n,test_n,train_mean,test_mean,mean_shift,train_median,test_median,median_shift
0,0,14393,7197,7.403,14.262,6.858,6.64,15.56,8.92
1,1,14393,7197,10.290,8.488,-1.802,10.66,7.71,-2.95
2,2,14394,7196,11.375,6.318,-5.056,12.00,5.41,-6.59


In [ ]:
# ------------------------------------------------------------------
# 5.2
# Donor composition of each held-out fold
# ------------------------------------------------------------------

donor_fold_metadata = (
    cv_for_model[
        [
            "id",
            "fold_donor_held_out",
        ]
    ]
    .merge(
        opv[
            [
                "id",
                "donor",
                "donor_canonical",
                "donor_graph_smiles",
                "pce",
            ]
        ],
        on="id",
        how="left",
        validate="one_to_one"
    )
)


def representative_label(series):

    mode = series.dropna().mode()

    if len(mode):
        return mode.iloc[0]

    nonmissing = series.dropna()

    if len(nonmissing):
        return nonmissing.iloc[0]

    return "unknown"


donor_group_summary = (
    donor_fold_metadata
    .groupby(
        [
            "fold_donor_held_out",
            "donor_graph_smiles",
        ],
        dropna=False
    )
    .agg(
        representative_donor=(
            "donor_canonical",
            representative_label
        ),

        records=(
            "id",
            "size"
        ),

        pce_mean=(
            "pce",
            "mean"
        ),

        pce_median=(
            "pce",
            "median"
        ),

        pce_min=(
            "pce",
            "min"
        ),

        pce_max=(
            "pce",
            "max"
        ),
    )
    .reset_index()
)



# ------------------------------------------------------------------
# 5.3
# Rank donor groups within each test fold
# ------------------------------------------------------------------

fold_totals = (
    donor_group_summary
    .groupby(
        "fold_donor_held_out"
    )["records"]
    .transform("sum")
)


donor_group_summary[
    "percent_of_test_fold"
] = (
    100
    * donor_group_summary["records"]
    / fold_totals
)


donor_group_summary[
    "rank_within_fold"
] = (
    donor_group_summary
    .groupby(
        "fold_donor_held_out"
    )["records"]
    .rank(
        method="first",
        ascending=False
    )
)


top_donors_by_fold = (
    donor_group_summary[
        donor_group_summary[
            "rank_within_fold"
        ] <= 10
    ]
    .sort_values(
        [
            "fold_donor_held_out",
            "rank_within_fold",
        ]
    )
)


display(
    top_donors_by_fold[
        [
            "fold_donor_held_out",
            "rank_within_fold",
            "representative_donor",
            "records",
            "percent_of_test_fold",
            "pce_mean",
            "pce_median",
            "pce_min",
            "pce_max",
        ]
    ].round(2)
)

,fold_donor_held_out,rank_within_fold,representative_donor,records,percent_of_test_fold,pce_mean,pce_median,pce_min,pce_max
343,0,1.0,PM6,6807,94.58,14.73,15.71,0.00,21.63
1,0,2.0,PdC8ThDT,2,0.03,8.15,8.15,7.76,8.54
10,0,3.0,PBDTTF-FTTE,2,0.03,8.27,8.27,7.44,9.10
19,0,4.0,PENTBT,2,0.03,2.02,2.02,2.01,2.04
23,0,5.0,PBDTT-S-TT-CF,2,0.03,9.58,9.58,9.58,9.58
28,0,6.0,PBTFB-C2C6,2,0.03,1.66,1.66,1.52,1.80
32,0,7.0,PCl(4)BDB-T,2,0.03,12.33,12.33,12.33,12.33
35,0,8.0,PTBFTPD,2,0.03,3.66,3.66,3.00,4.33
45,0,9.0,PT4Si-BDD,2,0.03,4.67,4.67,4.46,4.88
78,0,10.0,P4T2F-BO,2,0.03,5.75,5.75,5.30,6.20


In [ ]:
# ------------------------------------------------------------------
# 5.4
# Concentration of donor-held-out folds
# ------------------------------------------------------------------

donor_concentration_rows = []


for fold in [0, 1, 2]:

    fold_groups = (
        donor_group_summary[
            donor_group_summary[
                "fold_donor_held_out"
            ] == fold
        ]
        .sort_values(
            "records",
            ascending=False
        )
    )


    total = fold_groups[
        "records"
    ].sum()


    donor_concentration_rows.append({

        "fold":
            fold,

        "unique_donors":
            len(fold_groups),

        "largest_donor_records":
            fold_groups.iloc[0][
                "records"
            ],

        "largest_donor_percent":
            100
            * fold_groups.iloc[0][
                "records"
            ]
            / total,

        "top_5_donor_percent":
            100
            * fold_groups.head(5)[
                "records"
            ].sum()
            / total,
    })


donor_fold_concentration = (
    pd.DataFrame(
        donor_concentration_rows
    )
)


display(
    donor_fold_concentration.round(2)
)

,fold,unique_donors,largest_donor_records,largest_donor_percent,top_5_donor_percent
0,0,348,6807,94.58,94.69
1,1,820,1571,21.83,50.22
2,2,818,1553,21.58,50.22


In [ ]:
# ------------------------------------------------------------------
# 5.5
# Save donor-domain diagnosis
# ------------------------------------------------------------------

donor_fold_shift.to_csv(
    PROCESSED_DIR
    / "opvdb_donor_fold_target_shift.csv",
    index=False
)

top_donors_by_fold.to_csv(
    PROCESSED_DIR
    / "opvdb_top_donors_by_fold.csv",
    index=False
)

donor_fold_concentration.to_csv(
    PROCESSED_DIR
    / "opvdb_donor_fold_concentration.csv",
    index=False
)

ridge_interpretation.to_csv(
    PROCESSED_DIR
    / "opvdb_ridge_interpretation.csv",
    index=False
)

print("Donor-domain diagnosis saved.")

Donor-domain diagnosis saved.


### 5.5 Donor-domain diagnosis

Donor-held-out validation is highly heterogeneous across folds.

The difficult donor fold is dominated by PM6, which accounts for 6,807 of
7,197 test records (94.58%) and 31.53% of the complete provenance-clean
OPV-DB cohort. Because all records belonging to a donor must remain in the
same partition, this concentration is an intrinsic consequence of strict
donor-held-out validation rather than a fold-construction error.

Removal of the PM6 domain produces a pronounced target-distribution shift:
mean test PCE exceeds the corresponding training mean by 6.858 percentage
points, while the median shift is 8.920 percentage points. Ridge regression
therefore performs poorly on this fold despite retaining useful predictive
performance on the other two unseen-donor folds.

Donor-held-out performance is consequently reported both as an aggregate
three-fold result and at the individual-fold level. The dominant PM6 holdout
is interpreted as a particularly demanding chemical-domain extrapolation
test rather than evidence of uniformly poor unseen-donor prediction.

## 6. Nonlinear Molecular Baseline

A nonlinear tree-ensemble baseline is evaluated using the same frozen
4,096-bit molecular representation and the same five validation regimes.

The purpose of this experiment is not hyperparameter optimization. It tests
whether the generalization pattern observed with Ridge regression is primarily
a consequence of linear model capacity or persists under a nonlinear
structure–performance mapping.

Extra Trees regression is used as a reproducible untuned ensemble baseline.
If nonlinear modeling improves random and interpolation-oriented validation
but fails to recover the difficult unseen-donor domain, this would support
chemical-domain extrapolation rather than simple model underfitting as the
dominant limitation.

In [ ]:
# ------------------------------------------------------------------
# 6.1
# Untuned nonlinear molecular baseline
# ------------------------------------------------------------------

from sklearn.ensemble import ExtraTreesRegressor
import time


extra_trees_model = ExtraTreesRegressor(
    n_estimators=200,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)


start_time = time.perf_counter()


extra_trees_results = (
    evaluate_model_across_regimes(
        model=extra_trees_model,
        model_name="Extra Trees",
        X=X_opv,
        y=y_opv,
        cv_table=cv_for_model,
        regimes=fold_columns,
    )
)


elapsed_minutes = (
    time.perf_counter()
    - start_time
) / 60


print(
    "\nTotal Extra Trees runtime:",
    round(elapsed_minutes, 2),
    "minutes"
)


display(
    extra_trees_results.round(3)
)


Extra Trees — Random
Fold 0: MAE=1.758 | RMSE=2.494 | R2=0.810
Fold 1: MAE=1.739 | RMSE=2.453 | R2=0.812
Fold 2: MAE=1.723 | RMSE=2.422 | R2=0.818

Extra Trees — DOI held-out
Fold 0: MAE=1.907 | RMSE=2.608 | R2=0.787
Fold 1: MAE=1.929 | RMSE=2.701 | R2=0.775
Fold 2: MAE=1.967 | RMSE=2.702 | R2=0.776

Extra Trees — D:A pair held-out
Fold 0: MAE=2.036 | RMSE=2.818 | R2=0.747
Fold 1: MAE=2.127 | RMSE=2.742 | R2=0.744
Fold 2: MAE=1.968 | RMSE=2.667 | R2=0.803

Extra Trees — Donor held-out
Fold 0: MAE=4.079 | RMSE=4.581 | R2=-0.226
Fold 1: MAE=2.353 | RMSE=2.961 | R2=0.680
Fold 2: MAE=1.967 | RMSE=2.627 | R2=0.633

Extra Trees — Acceptor held-out
Fold 0: MAE=2.125 | RMSE=3.111 | R2=0.693
Fold 1: MAE=2.247 | RMSE=2.833 | R2=0.796
Fold 2: MAE=2.080 | RMSE=2.681 | R2=0.618

Total Extra Trees runtime: 41.03 minutes


,model,regime,fold,train_n,test_n,test_pce_mean,MAE,RMSE,R2
0,Extra Trees,Random,0,14393,7197,9.653,1.758,2.494,0.810
1,Extra Trees,Random,1,14393,7197,9.722,1.739,2.453,0.812
2,Extra Trees,Random,2,14394,7196,9.694,1.723,2.422,0.818
3,Extra Trees,DOI held-out,0,14393,7197,9.674,1.907,2.608,0.787
4,Extra Trees,DOI held-out,1,14394,7196,9.723,1.929,2.701,0.775
5,Extra Trees,DOI held-out,2,14393,7197,9.672,1.967,2.702,0.776
6,Extra Trees,D:A pair held-out,0,14394,7196,9.895,2.036,2.818,0.747
7,Extra Trees,D:A pair held-out,1,14393,7197,9.542,2.127,2.742,0.744
8,Extra Trees,D:A pair held-out,2,14393,7197,9.632,1.968,2.667,0.803
9,Extra Trees,Donor held-out,0,14393,7197,14.262,4.079,4.581,-0.226


In [ ]:
# ------------------------------------------------------------------
# 6.2
# Dummy vs Ridge vs nonlinear baseline
# ------------------------------------------------------------------

all_baseline_results = pd.concat(
    [
        dummy_results,
        ridge_results,
        extra_trees_results,
    ],
    ignore_index=True
)


model_comparison_summary = (
    all_baseline_results
    .groupby(
        ["model", "regime"]
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
    )
    .reset_index()
)


display(
    model_comparison_summary.round(3)
)

,model,regime,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,Dummy mean,Acceptor held-out,5.220,0.800,5.914,0.782,-0.228,0.252
1,Dummy mean,D:A pair held-out,4.957,0.359,5.683,0.303,-0.002,0.001
2,Dummy mean,DOI held-out,4.951,0.033,5.686,0.031,-0.000,0.000
3,Dummy mean,Donor held-out,5.978,1.340,6.734,1.240,-1.409,1.315
4,Dummy mean,Random,4.951,0.034,5.686,0.031,-0.000,0.000
5,Extra Trees,Acceptor held-out,2.151,0.087,2.875,0.218,0.702,0.090
6,Extra Trees,D:A pair held-out,2.043,0.080,2.742,0.076,0.765,0.033
7,Extra Trees,DOI held-out,1.934,0.030,2.670,0.054,0.779,0.007
8,Extra Trees,Donor held-out,2.800,1.124,3.390,1.045,0.362,0.510
9,Extra Trees,Random,1.740,0.017,2.456,0.036,0.813,0.004


In [ ]:
# ------------------------------------------------------------------
# 6.3
# R2 comparison table
# ------------------------------------------------------------------

r2_comparison = (
    model_comparison_summary
    .pivot(
        index="regime",
        columns="model",
        values="R2_mean"
    )
)


display(
    r2_comparison.round(3)
)

model,Dummy mean,Extra Trees,Ridge
regime,,,
Acceptor held-out,-0.228,0.702,0.550
D:A pair held-out,-0.002,0.765,0.725
DOI held-out,-0.000,0.779,0.727
Donor held-out,-1.409,0.362,-0.124
Random,-0.000,0.813,0.790


In [ ]:
# ------------------------------------------------------------------
# 6.4
# MAE comparison table
# ------------------------------------------------------------------

mae_comparison = (
    model_comparison_summary
    .pivot(
        index="regime",
        columns="model",
        values="MAE_mean"
    )
)


display(
    mae_comparison.round(3)
)

model,Dummy mean,Extra Trees,Ridge
regime,,,
Acceptor held-out,5.220,2.151,2.636
D:A pair held-out,4.957,2.043,2.221
DOI held-out,4.951,1.934,2.160
Donor held-out,5.978,2.800,3.563
Random,4.951,1.740,1.881


In [ ]:
# ------------------------------------------------------------------
# 6.5
# Ridge vs Extra Trees under strict donor holdout
# ------------------------------------------------------------------

donor_model_comparison = (
    all_baseline_results[
        (
            all_baseline_results[
                "regime"
            ] == "Donor held-out"
        )
        &
        (
            all_baseline_results[
                "model"
            ].isin(
                [
                    "Ridge",
                    "Extra Trees",
                ]
            )
        )
    ][
        [
            "model",
            "fold",
            "test_pce_mean",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .sort_values(
        ["fold", "model"]
    )
)


display(
    donor_model_comparison.round(3)
)

,model,fold,test_pce_mean,MAE,RMSE,R2
39,Extra Trees,0,14.262,4.079,4.581,-0.226
24,Ridge,0,14.262,5.999,6.506,-1.472
40,Extra Trees,1,8.488,2.353,2.961,0.680
25,Ridge,1,8.488,2.296,3.137,0.640
41,Extra Trees,2,6.318,1.967,2.627,0.633
26,Ridge,2,6.318,2.394,3.184,0.460


In [ ]:
# ------------------------------------------------------------------
# 6.6
# Save nonlinear-baseline results
# ------------------------------------------------------------------

extra_trees_results.to_csv(
    PROCESSED_DIR
    / "opvdb_extra_trees_fold_results.csv",
    index=False
)

model_comparison_summary.to_csv(
    PROCESSED_DIR
    / "opvdb_initial_model_comparison.csv",
    index=False
)

donor_model_comparison.to_csv(
    PROCESSED_DIR
    / "opvdb_donor_model_comparison.csv",
    index=False
)

print("Nonlinear baseline results saved.")

Nonlinear baseline results saved.


### 6.7 Nonlinear-baseline interpretation

Nonlinear modeling improves predictive performance under every validation
regime, but the magnitude of improvement depends strongly on the degree of
chemical independence.

Extra Trees provides only a modest improvement over Ridge under random
row-wise validation, whereas substantially larger gains are observed for
acceptor- and donor-held-out validation. This indicates that nonlinear
structure–performance relationships contribute to the apparent
generalization limitation.

Nevertheless, model capacity does not fully resolve the most difficult
chemical-domain shift. In the PM6-dominated donor-held-out fold, Extra Trees
reduces MAE from 5.999 to 4.079 percentage points and improves R² from -1.472
to -0.226, but predictive performance remains poor relative to the other
unseen-donor folds.

The donor-generalization limitation therefore reflects both model capacity
and genuine domain extrapolation. Subsequent analysis examines whether
prediction error is related to the structural distance between held-out
donors and the nearest donor chemistry available during training.

In [ ]:
all_baseline_results.to_csv(
    PROCESSED_DIR / "opvdb_all_initial_baseline_results.csv",
    index=False
)

model_comparison_summary.to_csv(
    PROCESSED_DIR / "opvdb_initial_model_comparison.csv",
    index=False
)

print("Initial baseline comparison saved.")

Initial baseline comparison saved.


## 7. Chemical-Distance Dependence of Donor Generalization

The donor-held-out experiments reveal substantial variation in predictive
difficulty across chemical domains.

To determine whether this variation is associated with structural novelty,
each held-out donor is compared with the most similar donor available in its
corresponding training fold.

Maximum Morgan-fingerprint Tanimoto similarity to the training donor set is
used as an applicability-domain measure. Out-of-fold prediction error is then
examined as a function of this nearest-training similarity.

This analysis tests whether poor prospective performance is systematically
associated with increasing chemical distance rather than being attributable
solely to target-distribution shift or model choice.

In [ ]:
# ------------------------------------------------------------------
# 7.1
# Donor-held-out out-of-fold Extra Trees predictions
# ------------------------------------------------------------------

donor_oof_prediction = np.full(
    len(y_opv),
    np.nan,
    dtype=float
)


for fold in [0, 1, 2]:

    print(f"Fitting donor-held-out fold {fold}...")

    test_mask = (
        cv_for_model[
            "fold_donor_held_out"
        ].to_numpy()
        == fold
    )

    train_mask = ~test_mask


    model = ExtraTreesRegressor(
        n_estimators=200,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )


    model.fit(
        X_opv[train_mask],
        y_opv[train_mask]
    )


    donor_oof_prediction[
        test_mask
    ] = model.predict(
        X_opv[test_mask]
    )


print(
    "Missing OOF predictions:",
    np.isnan(
        donor_oof_prediction
    ).sum()
)

Fitting donor-held-out fold 0...
Fitting donor-held-out fold 1...
Fitting donor-held-out fold 2...
Missing OOF predictions: 0


In [ ]:
# ------------------------------------------------------------------
# 7.2
# RDKit fingerprint objects for donor similarity analysis
# ------------------------------------------------------------------

donor_rdkit_fp_map = {}


for smiles in unique_donors:

    mol = Chem.MolFromSmiles(
        str(smiles)
    )

    donor_rdkit_fp_map[
        smiles
    ] = (
        morgan_generator
        .GetFingerprint(mol)
    )


print(
    "Donor fingerprints available:",
    len(donor_rdkit_fp_map)
)

Donor fingerprints available: 1986


In [ ]:
# ------------------------------------------------------------------
# Maximum Tanimoto similarity to training donors
# ------------------------------------------------------------------

nearest_training_donor_similarity = {}


for fold in [0, 1, 2]:

    train_donors = (
        cv_for_model.loc[
            cv_for_model[
                "fold_donor_held_out"
            ] != fold,
            "donor_graph_smiles"
        ]
        .drop_duplicates()
        .tolist()
    )


    test_donors = (
        cv_for_model.loc[
            cv_for_model[
                "fold_donor_held_out"
            ] == fold,
            "donor_graph_smiles"
        ]
        .drop_duplicates()
        .tolist()
    )


    train_fps = [
        donor_rdkit_fp_map[
            smiles
        ]
        for smiles in train_donors
    ]


    print(
        f"Fold {fold}:",
        len(test_donors),
        "held-out donors vs",
        len(train_donors),
        "training donors"
    )


    for smiles in test_donors:

        similarities = (
            DataStructs
            .BulkTanimotoSimilarity(
                donor_rdkit_fp_map[
                    smiles
                ],
                train_fps
            )
        )


        nearest_training_donor_similarity[
            (fold, smiles)
        ] = max(similarities)

Fold 0: 348 held-out donors vs 1638 training donors
Fold 1: 820 held-out donors vs 1166 training donors
Fold 2: 818 held-out donors vs 1168 training donors


In [ ]:
# ------------------------------------------------------------------
# 7.3
# Combine predictions, structural novelty, and observed PCE
# ------------------------------------------------------------------

donor_ad = (
    cv_for_model[
        [
            "id",
            "fold_donor_held_out",
            "donor_graph_smiles",
        ]
    ]
    .copy()
)


donor_ad[
    "observed_pce"
] = y_opv


donor_ad[
    "predicted_pce"
] = donor_oof_prediction


donor_ad[
    "absolute_error"
] = np.abs(
    donor_ad[
        "observed_pce"
    ]
    -
    donor_ad[
        "predicted_pce"
    ]
)


donor_ad[
    "nearest_training_donor_tanimoto"
] = [
    nearest_training_donor_similarity[
        (
            fold,
            smiles
        )
    ]
    for fold, smiles
    in zip(
        donor_ad[
            "fold_donor_held_out"
        ],
        donor_ad[
            "donor_graph_smiles"
        ],
    )
]


print(
    "Records:",
    len(donor_ad)
)

print(
    "Missing similarity:",
    donor_ad[
        "nearest_training_donor_tanimoto"
    ].isna().sum()
)


display(
    donor_ad[
        [
            "observed_pce",
            "predicted_pce",
            "absolute_error",
            "nearest_training_donor_tanimoto",
        ]
    ]
    .describe()
    .round(3)
)

Records: 21590
Missing similarity: 0


,observed_pce,predicted_pce,absolute_error,nearest_training_donor_tanimoto
count,21590.000,21590.000,21590.000,21590.000
mean,9.690,8.331,2.800,0.683
std,5.686,3.631,2.092,0.184
min,0.000,0.790,0.000,0.064
25%,4.640,4.876,1.087,0.512
50%,9.260,9.052,2.343,0.717
75%,15.200,11.064,4.285,0.848
max,21.630,16.773,18.680,1.000


In [ ]:
# ------------------------------------------------------------------
# 7.4
# Error and chemical distance at donor-group level
# ------------------------------------------------------------------

donor_ad_group = (
    donor_ad
    .merge(
        opv[
            [
                "id",
                "donor_canonical",
            ]
        ],
        on="id",
        how="left",
        validate="one_to_one"
    )
    .groupby(
        [
            "fold_donor_held_out",
            "donor_graph_smiles",
        ]
    )
    .agg(
        representative_donor=(
            "donor_canonical",
            representative_label
        ),

        records=(
            "id",
            "size"
        ),

        nearest_training_tanimoto=(
            "nearest_training_donor_tanimoto",
            "first"
        ),

        observed_pce_mean=(
            "observed_pce",
            "mean"
        ),

        MAE=(
            "absolute_error",
            "mean"
        ),
    )
    .reset_index()
)


display(
    donor_ad_group
    .sort_values(
        "records",
        ascending=False
    )
    .head(25)
    .round(3)
)

,fold_donor_held_out,donor_graph_smiles,representative_donor,records,nearest_training_tanimoto,observed_pce_mean,MAE
343,0,COC(=O)C(C)(C)c1cc(C)cc(-c2sc3c4sc5cc(/C=C6/C(...,PM6,6807,0.512,14.730,4.195
386,1,CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(CC)...,PTB7-Th,1571,0.848,7.594,1.826
1934,2,CCCCCCc1ccsc1,P3HT,1553,0.436,3.024,1.273
1263,2,CCCCC(CC)Cc1ccc(-c2c3cc(-c4ccc(-c5sc(-c6cccs6)...,PBDB-T,1406,0.944,9.033,1.979
530,1,CCCCCCC(CCCC)Cc1csc(-c2cc3c4nsnc4c4cc(-c5cc(CC...,D18,956,0.807,17.214,4.348
384,1,CCCCC(CC)COC(=O)c1sc2c(-c3cc4c(OCC(CC)CCCC)c5s...,PTB7,627,0.757,5.890,2.146
1827,2,CCCCCCCCc1cc(-c2c(-c3cccs3)sc3c(-c4cc(CCCCCCCC...,P1,296,0.548,3.966,1.909
1082,1,CCCCCCc1cc(C)sc1C,P3HT,263,1.000,3.072,1.755
1388,2,CCCCCCCCC(CCCCCC)COc1cnc2c(-c3cccs3)c(F)c(F)cc2n1,PTQ10,199,0.862,11.856,3.360
529,1,CCCCCCC(CCCC)Cc1csc(-c2cc3c4nsnc4c4cc(-c5cc(CC...,D18-Cl,197,0.807,16.477,4.231


In [ ]:
pm6_ad = (
    donor_ad_group[
        donor_ad_group[
            "representative_donor"
        ].astype(str)
        .str.upper()
        .eq("PM6")
    ]
)

display(
    pm6_ad.round(3)
)

,fold_donor_held_out,donor_graph_smiles,representative_donor,records,nearest_training_tanimoto,observed_pce_mean,MAE
343,0,COC(=O)C(C)(C)c1cc(C)cc(-c2sc3c4sc5cc(/C=C6/C(...,PM6,6807,0.512,14.73,4.195


### 7.6 Chemical distance versus performance-domain shift

Nearest-neighbor molecular similarity alone does not appear sufficient to
explain unseen-donor prediction difficulty.

Some chemically distant donors are predicted accurately, whereas several
donors with relatively high similarity to the training chemistry retain large
prediction errors.

The analysis is therefore extended to distinguish two forms of extrapolation:

1. structural-domain shift, represented by nearest-training-donor Tanimoto
   similarity; and

2. performance-domain shift, represented by the difference between the mean
   PCE of each held-out donor and the mean PCE available in its corresponding
   training fold.

Associations with donor-level prediction error are evaluated without weighting
all device records equally, because highly represented donors such as PM6
would otherwise dominate the analysis.

In [ ]:
# ------------------------------------------------------------------
# 7.6A
# Fold-specific performance-domain shift
# ------------------------------------------------------------------

training_pce_reference = {}

for fold in [0, 1, 2]:

    train_mask = (
        cv_for_model[
            "fold_donor_held_out"
        ].to_numpy()
        != fold
    )

    training_pce_reference[fold] = {
        "train_mean":
            y_opv[train_mask].mean(),

        "train_median":
            np.median(
                y_opv[train_mask]
            ),
    }


donor_ad_group[
    "training_pce_mean"
] = (
    donor_ad_group[
        "fold_donor_held_out"
    ].map(
        lambda fold:
            training_pce_reference[
                fold
            ]["train_mean"]
    )
)


donor_ad_group[
    "signed_pce_shift"
] = (
    donor_ad_group[
        "observed_pce_mean"
    ]
    -
    donor_ad_group[
        "training_pce_mean"
    ]
)


donor_ad_group[
    "absolute_pce_shift"
] = (
    donor_ad_group[
        "signed_pce_shift"
    ].abs()
)


display(
    donor_ad_group[
        [
            "fold_donor_held_out",
            "representative_donor",
            "records",
            "nearest_training_tanimoto",
            "training_pce_mean",
            "observed_pce_mean",
            "signed_pce_shift",
            "absolute_pce_shift",
            "MAE",
        ]
    ]
    .sort_values(
        "records",
        ascending=False
    )
    .head(30)
    .round(3)
)

,fold_donor_held_out,representative_donor,records,nearest_training_tanimoto,training_pce_mean,observed_pce_mean,signed_pce_shift,absolute_pce_shift,MAE
343,0,PM6,6807,0.512,7.403,14.730,7.327,7.327,4.195
386,1,PTB7-Th,1571,0.848,10.290,7.594,-2.696,2.696,1.826
1934,2,P3HT,1553,0.436,11.375,3.024,-8.351,8.351,1.273
1263,2,PBDB-T,1406,0.944,11.375,9.033,-2.341,2.341,1.979
530,1,D18,956,0.807,10.290,17.214,6.923,6.923,4.348
384,1,PTB7,627,0.757,10.290,5.890,-4.400,4.400,2.146
1827,2,P1,296,0.548,11.375,3.966,-7.409,7.409,1.909
1082,1,P3HT,263,1.000,10.290,3.072,-7.219,7.219,1.755
1388,2,PTQ10,199,0.862,11.375,11.856,0.481,0.481,3.360
529,1,D18-Cl,197,0.807,10.290,16.477,6.187,6.187,4.231


In [ ]:
# ------------------------------------------------------------------
# 7.6B
# Donor-level association with prediction error
# ------------------------------------------------------------------

from scipy.stats import (
    pearsonr,
    spearmanr,
)


analysis_cols = [
    "nearest_training_tanimoto",
    "absolute_pce_shift",
    "MAE",
]

donor_corr_data = (
    donor_ad_group[
        analysis_cols
    ]
    .dropna()
    .copy()
)


similarity_pearson = pearsonr(
    donor_corr_data[
        "nearest_training_tanimoto"
    ],
    donor_corr_data[
        "MAE"
    ]
)

similarity_spearman = spearmanr(
    donor_corr_data[
        "nearest_training_tanimoto"
    ],
    donor_corr_data[
        "MAE"
    ]
)


shift_pearson = pearsonr(
    donor_corr_data[
        "absolute_pce_shift"
    ],
    donor_corr_data[
        "MAE"
    ]
)

shift_spearman = spearmanr(
    donor_corr_data[
        "absolute_pce_shift"
    ],
    donor_corr_data[
        "MAE"
    ]
)


association_summary = pd.DataFrame({
    "predictor": [
        "nearest-training donor Tanimoto",
        "absolute donor PCE shift",
    ],

    "pearson_r": [
        similarity_pearson.statistic,
        shift_pearson.statistic,
    ],

    "pearson_p": [
        similarity_pearson.pvalue,
        shift_pearson.pvalue,
    ],

    "spearman_rho": [
        similarity_spearman.statistic,
        shift_spearman.statistic,
    ],

    "spearman_p": [
        similarity_spearman.pvalue,
        shift_spearman.pvalue,
    ],
})


display(
    association_summary.round(4)
)

,predictor,pearson_r,pearson_p,spearman_rho,spearman_p
0,nearest-training donor Tanimoto,-0.1103,0.0000,-0.1248,0.0000
1,absolute donor PCE shift,0.0392,0.0809,-0.0238,0.2892


In [ ]:
# ------------------------------------------------------------------
# 7.6C
# Joint donor-level explanation of MAE
# ------------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression


joint_data = (
    donor_ad_group[
        [
            "nearest_training_tanimoto",
            "absolute_pce_shift",
            "MAE",
        ]
    ]
    .dropna()
    .copy()
)


X_explain = joint_data[
    [
        "nearest_training_tanimoto",
        "absolute_pce_shift",
    ]
].to_numpy()


y_explain = (
    joint_data[
        "MAE"
    ].to_numpy()
)


scaler = StandardScaler()

X_explain_scaled = scaler.fit_transform(
    X_explain
)


explanation_model = LinearRegression()

explanation_model.fit(
    X_explain_scaled,
    y_explain
)


joint_explanation = pd.DataFrame({

    "predictor": [
        "nearest-training Tanimoto",
        "absolute PCE shift",
    ],

    "standardized_coefficient":
        explanation_model.coef_,
})


print(
    "Joint donor-level R2:",
    round(
        explanation_model.score(
            X_explain_scaled,
            y_explain
        ),
        3
    )
)


display(
    joint_explanation.round(3)
)

Joint donor-level R2: 0.012


,predictor,standardized_coefficient
0,nearest-training Tanimoto,-0.162
1,absolute PCE shift,0.026


In [ ]:
# ------------------------------------------------------------------
# 7.6D
# Error by nearest-training similarity
# ------------------------------------------------------------------

donor_ad_group[
    "similarity_bin"
] = pd.cut(
    donor_ad_group[
        "nearest_training_tanimoto"
    ],
    bins=[
        0.0,
        0.4,
        0.6,
        0.8,
        1.000001,
    ],
    labels=[
        "<0.40",
        "0.40–0.60",
        "0.60–0.80",
        ">0.80",
    ],
    include_lowest=True,
)


similarity_bin_summary = (
    donor_ad_group
    .groupby(
        "similarity_bin",
        observed=True
    )
    .agg(
        donor_groups=(
            "donor_graph_smiles",
            "size"
        ),

        median_MAE=(
            "MAE",
            "median"
        ),

        mean_MAE=(
            "MAE",
            "mean"
        ),

        median_absolute_pce_shift=(
            "absolute_pce_shift",
            "median"
        ),
    )
    .reset_index()
)


display(
    similarity_bin_summary.round(3)
)

,similarity_bin,donor_groups,median_MAE,mean_MAE,median_absolute_pce_shift
0,<0.40,28,2.061,2.634,7.323
1,0.40–0.60,189,1.903,2.154,6.913
2,0.60–0.80,824,1.746,2.009,5.754
3,>0.80,945,1.491,1.804,4.305


In [ ]:
# ------------------------------------------------------------------
# 7.6E
# Highest-error unseen donor groups
# ------------------------------------------------------------------

hardest_donors = (
    donor_ad_group[
        [
            "fold_donor_held_out",
            "representative_donor",
            "records",
            "nearest_training_tanimoto",
            "observed_pce_mean",
            "training_pce_mean",
            "signed_pce_shift",
            "absolute_pce_shift",
            "MAE",
        ]
    ]
    .sort_values(
        "MAE",
        ascending=False
    )
    .head(30)
)


display(
    hardest_donors.round(3)
)

,fold_donor_held_out,representative_donor,records,nearest_training_tanimoto,observed_pce_mean,training_pce_mean,signed_pce_shift,absolute_pce_shift,MAE
1182,2,PDTPDTBT,3,0.571,20.017,11.375,8.642,8.642,17.657
1022,1,L-MD,1,0.787,0.290,10.290,-10.000,10.000,13.288
242,0,PTzBI-pF,1,0.811,1.400,7.403,-6.003,6.003,13.044
43,0,OPz4,1,0.819,0.100,7.403,-7.303,7.303,11.766
621,1,PBTATBT-2f,1,0.885,2.610,10.290,-7.680,7.680,10.866
1293,2,PTz3Cl,1,0.645,4.070,11.375,-7.305,7.305,10.471
46,0,PBDB-Si,1,0.837,2.960,7.403,-4.443,4.443,9.619
340,0,PBDB-T-4Cl,1,0.602,1.040,7.403,-6.363,6.363,9.587
867,1,Q7,1,0.849,0.480,10.290,-9.810,9.810,9.575
61,0,PETVT-FT,1,0.712,1.920,7.403,-5.483,5.483,9.205


### 7.7 Sensitivity to donor-group support

Donor-level error estimates vary substantially in statistical reliability
because the number of device records associated with each held-out donor is
highly uneven.

Several of the highest donor-level MAEs arise from materials represented by
only one or a few device records, whereas major donor domains such as PM6 are
represented by thousands of observations.

The chemical-distance analysis is therefore repeated across minimum
donor-support thresholds. This tests whether the observed relationship between
nearest-training structural similarity and prediction error persists when
poorly supported donor-level error estimates are excluded.

In [ ]:
# ------------------------------------------------------------------
# 7.7A
# Similarity-error relationship across donor-support thresholds
# ------------------------------------------------------------------

support_thresholds = [
    1,
    2,
    5,
    10,
    20,
    50,
]


support_sensitivity_rows = []


for minimum_records in support_thresholds:

    subset = (
        donor_ad_group[
            donor_ad_group[
                "records"
            ] >= minimum_records
        ]
        .dropna(
            subset=[
                "nearest_training_tanimoto",
                "absolute_pce_shift",
                "MAE",
            ]
        )
        .copy()
    )


    if len(subset) < 3:
        continue


    sim_pearson = pearsonr(
        subset[
            "nearest_training_tanimoto"
        ],
        subset["MAE"]
    )

    sim_spearman = spearmanr(
        subset[
            "nearest_training_tanimoto"
        ],
        subset["MAE"]
    )


    shift_pearson = pearsonr(
        subset[
            "absolute_pce_shift"
        ],
        subset["MAE"]
    )

    shift_spearman = spearmanr(
        subset[
            "absolute_pce_shift"
        ],
        subset["MAE"]
    )


    support_sensitivity_rows.append({

        "minimum_records":
            minimum_records,

        "donor_groups":
            len(subset),

        "covered_device_records":
            int(
                subset["records"].sum()
            ),

        "similarity_pearson_r":
            sim_pearson.statistic,

        "similarity_pearson_p":
            sim_pearson.pvalue,

        "similarity_spearman_rho":
            sim_spearman.statistic,

        "similarity_spearman_p":
            sim_spearman.pvalue,

        "pce_shift_pearson_r":
            shift_pearson.statistic,

        "pce_shift_pearson_p":
            shift_pearson.pvalue,

        "pce_shift_spearman_rho":
            shift_spearman.statistic,

        "pce_shift_spearman_p":
            shift_spearman.pvalue,
    })


support_sensitivity = pd.DataFrame(
    support_sensitivity_rows
)


display(
    support_sensitivity.round(4)
)

,minimum_records,donor_groups,covered_device_records,similarity_pearson_r,similarity_pearson_p,similarity_spearman_rho,similarity_spearman_p,pce_shift_pearson_r,pce_shift_pearson_p,pce_shift_spearman_rho,pce_shift_spearman_p
0,1,1986,21590,-0.1103,0.0000,-0.1248,0.0000,0.0392,0.0809,-0.0238,0.2892
1,2,1074,20678,-0.0880,0.0039,-0.0870,0.0043,-0.1190,0.0001,-0.1679,0.0000
2,5,336,18746,-0.0705,0.1975,-0.1177,0.0311,-0.2552,0.0000,-0.3181,0.0000
3,10,132,17472,-0.0044,0.9600,-0.0122,0.8896,-0.2025,0.0199,-0.2741,0.0015
4,20,59,16508,-0.0203,0.8784,-0.0549,0.6797,-0.1191,0.3688,-0.2449,0.0616
5,50,27,15553,0.1295,0.5197,-0.0394,0.8453,-0.1524,0.4480,-0.3260,0.0970


In [ ]:
# ------------------------------------------------------------------
# 7.7B
# Coverage retained by support threshold
# ------------------------------------------------------------------

support_sensitivity[
    "percent_device_records_retained"
] = (
    100
    * support_sensitivity[
        "covered_device_records"
    ]
    / len(opv)
)


display(
    support_sensitivity[
        [
            "minimum_records",
            "donor_groups",
            "covered_device_records",
            "percent_device_records_retained",
            "similarity_spearman_rho",
            "similarity_spearman_p",
            "pce_shift_spearman_rho",
            "pce_shift_spearman_p",
        ]
    ]
    .round(3)
)

,minimum_records,donor_groups,covered_device_records,percent_device_records_retained,similarity_spearman_rho,similarity_spearman_p,pce_shift_spearman_rho,pce_shift_spearman_p
0,1,1986,21590,100.000,-0.125,0.000,-0.024,0.289
1,2,1074,20678,95.776,-0.087,0.004,-0.168,0.000
2,5,336,18746,86.827,-0.118,0.031,-0.318,0.000
3,10,132,17472,80.926,-0.012,0.890,-0.274,0.001
4,20,59,16508,76.461,-0.055,0.680,-0.245,0.062
5,50,27,15553,72.038,-0.039,0.845,-0.326,0.097


In [ ]:
donor_ad_group.to_csv(
    PROCESSED_DIR
    / "opvdb_donor_applicability_domain.csv",
    index=False
)

association_summary.to_csv(
    PROCESSED_DIR
    / "opvdb_donor_similarity_error_association.csv",
    index=False
)

similarity_bin_summary.to_csv(
    PROCESSED_DIR
    / "opvdb_donor_similarity_bins.csv",
    index=False
)

support_sensitivity.to_csv(
    PROCESSED_DIR
    / "opvdb_donor_support_sensitivity.csv",
    index=False
)

print("Donor applicability-domain analysis saved.")

Donor applicability-domain analysis saved.


### 7.8 Applicability-domain conclusion

Nearest-training donor similarity shows a weak inverse association with
donor-level prediction error when all donor groups are included. However,
this relationship is not robust to minimum-support filtering.

The similarity–error association remains weakly detectable for donor groups
represented by at least five records, but disappears when analysis is
restricted to donors represented by ten or more observations. Thus, the
overall statistical association is partly influenced by the large number of
poorly supported donor groups for which a donor-level MAE is estimated from
only one or a few devices.

Absolute PCE-domain displacement likewise does not provide a consistent
general explanation for unseen-donor error. Its association with MAE changes
with donor-support threshold and, where significant, is frequently opposite
to the simple expectation that larger target-domain shifts should necessarily
produce larger prediction errors.

The PM6 holdout remains a well-supported and particularly difficult
extrapolation domain, but its failure cannot be generalized into a universal
fingerprint-distance or target-shift rule.

Consequently, no fixed Tanimoto applicability-domain threshold is imposed.
Subsequent modeling treats chemical generalization as a multidimensional
problem rather than assuming that nearest-neighbor structural similarity
alone determines predictive reliability.

In [ ]:
support_sensitivity.to_csv(
    PROCESSED_DIR
    / "opvdb_donor_support_sensitivity.csv",
    index=False
)

print("OPV-DB applicability-domain branch complete.")

OPV-DB applicability-domain branch complete.


# 8. Wen–Zhang–Ma Structure–Processing–Performance Benchmark

The second modeling branch evaluates the physically harmonized
Wen–Zhang–Ma global dataset.

Three frozen feature representations are compared:

1. processing-only;
2. molecular-structure-only; and
3. combined structure + processing.

The objective is to determine whether explicit processing information adds
predictive value beyond molecular structure and whether any improvement
persists when molecular systems are held out from training.

The 994-condition physical cohort and all feature definitions were frozen
during the data-audit stage and are loaded without further feature-selection
changes.

In [ ]:
# ------------------------------------------------------------------
# 8.1
# Load frozen Wen/Ma modeling cohort and feature definitions
# ------------------------------------------------------------------

wen = pd.read_csv(
    INTERIM_DIR
    / "FINAL_wen_primary_physical_cohort.csv"
)

wen_feature_manifest = pd.read_csv(
    INTERIM_DIR
    / "FINAL_wen_feature_manifest.csv"
)


print("Wen/Ma cohort:")
print(wen.shape)

print("\nFeature manifest:")
print(wen_feature_manifest.shape)

print("\nFirst cohort columns:")
print(wen.columns[:30].tolist())

print("\nManifest columns:")
print(wen_feature_manifest.columns.tolist())

Wen/Ma cohort:
(994, 2104)

Feature manifest:
(2028, 4)

First cohort columns:
['Name_Donor', 'Smiles_Donor', 'Name_Acceptor', 'Smiles_Acceptor', 'D_A_Weight_Ratio', 'Blend_Concentration (mg/ml)', 'Solvent_DipoleMoment (Debye)', 'Solvent_EnergyGap (eV)', 'Solvent_Polarizability (a.u.)', 'Additive_MeltingPoint (℃)', 'Additive_BoilingPoint  (℃)', 'Additive_Density (g/cm3)', 'Additive_MolecularWeight', 'Additive_DipoleMoment  (Debye)', 'Additive_EnergyGap (eV)', 'Additive_Polarizability (a.u.)', 'Additive_Volume_Ratio (vol%)', 'Spin_Coating_Rate (rpm)', 'Annealing_Temperature(℃)', 'Annealing_Time (min)', 'Active_Layer_Thickness (nm)', 'E_DH-1 (eV)', 'E_DH (eV)', 'E_DL (eV)', 'E_DL+1 (eV)', 'E_AH-1 (eV)', 'E_AH (eV)', 'E_AL (eV)', 'E_AL+1 (eV)', 'delta_E_DL_DH (eV)']

Manifest columns:
['feature', 'in_processing_panel', 'in_structure_panel', 'in_full_panel']


In [ ]:
# ------------------------------------------------------------------
# 8.2
# Recover frozen processing / structure / full panels
# ------------------------------------------------------------------

processing_features = (
    wen_feature_manifest.loc[
        wen_feature_manifest["in_processing_panel"].astype(bool),
        "feature"
    ]
    .tolist()
)

structure_features = (
    wen_feature_manifest.loc[
        wen_feature_manifest["in_structure_panel"].astype(bool),
        "feature"
    ]
    .tolist()
)

full_features = (
    wen_feature_manifest.loc[
        wen_feature_manifest["in_full_panel"].astype(bool),
        "feature"
    ]
    .tolist()
)


print("Processing features:", len(processing_features))
print("Structure features :", len(structure_features))
print("Full features      :", len(full_features))


missing_features = [
    col
    for col in full_features
    if col not in wen.columns
]

print("\nMissing frozen features:", missing_features)

Processing features: 20
Structure features : 2008
Full features      : 2028

Missing frozen features: []


In [ ]:
# ------------------------------------------------------------------
# 8.3
# Verify target and validation-group fields
# ------------------------------------------------------------------

candidate_target_fields = [
    "PCE (%)",
    "pce",
]

print("Target candidates:")

for col in candidate_target_fields:
    print(
        col,
        "FOUND" if col in wen.columns else "not found"
    )


group_fields = [
    "donor_name_norm",
    "acceptor_name_norm",
    "nominal_DA_group",
]

print("\nGrouping fields:")

for col in group_fields:
    print(
        col,
        "FOUND" if col in wen.columns else "not found"
    )

Target candidates:
PCE (%) FOUND
pce not found

Grouping fields:
donor_name_norm FOUND
acceptor_name_norm FOUND
nominal_DA_group FOUND


In [ ]:
# ------------------------------------------------------------------
# 8.4
# Group-size concentration in the Wen/Ma primary cohort
# ------------------------------------------------------------------

group_size_rows = []


for label, col in [
    ("donor", "donor_name_norm"),
    ("acceptor", "acceptor_name_norm"),
    ("D:A pair", "nominal_DA_group"),
]:

    counts = (
        wen[col]
        .value_counts()
    )

    group_size_rows.append({

        "grouping":
            label,

        "unique_groups":
            len(counts),

        "largest_group_records":
            int(counts.iloc[0]),

        "largest_group_percent":
            100 * counts.iloc[0] / len(wen),

        "top_5_percent":
            100 * counts.head(5).sum() / len(wen),

        "singleton_groups":
            int((counts == 1).sum()),
    })


wen_group_concentration = pd.DataFrame(
    group_size_rows
)

display(
    wen_group_concentration.round(2)
)

,grouping,unique_groups,largest_group_records,largest_group_percent,top_5_percent,singleton_groups
0,donor,60,257,25.86,60.97,10
1,acceptor,181,134,13.48,22.54,76
2,D:A pair,247,19,1.91,8.35,124


## 8.5 Validation Regimes for Structure–Processing Modeling

The Wen–Zhang–Ma benchmark is evaluated under four three-fold validation
regimes:

1. random row-wise cross-validation;
2. donor–acceptor-pair-held-out cross-validation;
3. donor-held-out cross-validation; and
4. acceptor-held-out cross-validation.

Grouped folds are constructed using a record-balanced greedy assignment that
does not use PCE values. Groups are ordered by size and assigned to the
currently smallest fold, with random tie-breaking controlled by a fixed seed.

This design preserves strict group independence while avoiding unnecessary
imbalance in test-set size.

The same frozen folds are subsequently used for processing-only,
structure-only, and combined structure–processing models.

In [ ]:
# ------------------------------------------------------------------
# 8.5A
# Record-balanced group assignment
# ------------------------------------------------------------------

def make_balanced_group_folds(
    groups,
    n_splits=3,
    random_state=42
):
    """
    Assign complete groups to folds while approximately balancing
    the number of records per fold.

    No target values are used.
    """

    groups = pd.Series(groups).reset_index(drop=True)

    group_counts = (
        groups
        .value_counts(dropna=False)
        .rename_axis("group")
        .reset_index(name="records")
    )

    rng = np.random.default_rng(
        random_state
    )

    # Random number only breaks ties between equally sized groups
    group_counts["_tie"] = rng.random(
        len(group_counts)
    )

    group_counts = (
        group_counts
        .sort_values(
            ["records", "_tie"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )


    fold_sizes = np.zeros(
        n_splits,
        dtype=int
    )

    group_to_fold = {}


    for _, row in group_counts.iterrows():

        # Assign next group to currently smallest fold
        smallest_fold = int(
            np.argmin(fold_sizes)
        )

        group_to_fold[
            row["group"]
        ] = smallest_fold

        fold_sizes[
            smallest_fold
        ] += int(
            row["records"]
        )


    assignments = (
        groups
        .map(group_to_fold)
        .astype(int)
        .to_numpy()
    )


    return assignments, fold_sizes

In [ ]:
# ------------------------------------------------------------------
# 8.5B
# Freeze random and grouped three-fold assignments
# ------------------------------------------------------------------

from sklearn.model_selection import KFold


wen_cv = wen[
    [
        "Name_Donor",
        "Name_Acceptor",
        "donor_name_norm",
        "acceptor_name_norm",
        "nominal_DA_group",
        "PCE (%)",
    ]
].copy()


# --------------------------------------------------------------
# Random reference
# --------------------------------------------------------------

random_kfold = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

wen_cv["fold_random"] = -1

for fold, (_, test_idx) in enumerate(
    random_kfold.split(wen_cv)
):

    wen_cv.loc[
        test_idx,
        "fold_random"
    ] = fold


# --------------------------------------------------------------
# Strict grouped regimes
# --------------------------------------------------------------

(
    wen_cv["fold_da_pair_held_out"],
    pair_fold_sizes
) = make_balanced_group_folds(
    wen_cv["nominal_DA_group"],
    n_splits=3,
    random_state=42
)


(
    wen_cv["fold_donor_held_out"],
    donor_fold_sizes
) = make_balanced_group_folds(
    wen_cv["donor_name_norm"],
    n_splits=3,
    random_state=42
)


(
    wen_cv["fold_acceptor_held_out"],
    acceptor_fold_sizes
) = make_balanced_group_folds(
    wen_cv["acceptor_name_norm"],
    n_splits=3,
    random_state=42
)


print("Pair fold sizes    :", pair_fold_sizes)
print("Donor fold sizes   :", donor_fold_sizes)
print("Acceptor fold sizes:", acceptor_fold_sizes)

Pair fold sizes    : [332 331 331]
Donor fold sizes   : [332 331 331]
Acceptor fold sizes: [332 331 331]


In [ ]:
# ------------------------------------------------------------------
# 8.6
# Record balance across Wen/Ma validation regimes
# ------------------------------------------------------------------

wen_validation_regimes = {

    "Random":
        "fold_random",

    "D:A pair held-out":
        "fold_da_pair_held_out",

    "Donor held-out":
        "fold_donor_held_out",

    "Acceptor held-out":
        "fold_acceptor_held_out",
}


wen_fold_size_rows = []


for regime_name, fold_col in (
    wen_validation_regimes.items()
):

    for fold in [0, 1, 2]:

        test_mask = (
            wen_cv[fold_col] == fold
        )

        wen_fold_size_rows.append({

            "regime":
                regime_name,

            "fold":
                fold,

            "train_records":
                int((~test_mask).sum()),

            "test_records":
                int(test_mask.sum()),

            "test_percent":
                100 * test_mask.mean(),
        })


wen_fold_size_summary = pd.DataFrame(
    wen_fold_size_rows
)


display(
    wen_fold_size_summary.round(2)
)

,regime,fold,train_records,test_records,test_percent
0,Random,0,662,332,33.4
1,Random,1,663,331,33.3
2,Random,2,663,331,33.3
3,D:A pair held-out,0,662,332,33.4
4,D:A pair held-out,1,663,331,33.3
5,D:A pair held-out,2,663,331,33.3
6,Donor held-out,0,662,332,33.4
7,Donor held-out,1,663,331,33.3
8,Donor held-out,2,663,331,33.3
9,Acceptor held-out,0,662,332,33.4


In [ ]:
# ------------------------------------------------------------------
# 8.7
# Group-overlap sanity checks
# ------------------------------------------------------------------

wen_group_checks = {

    "D:A pair held-out": (
        "fold_da_pair_held_out",
        "nominal_DA_group"
    ),

    "Donor held-out": (
        "fold_donor_held_out",
        "donor_name_norm"
    ),

    "Acceptor held-out": (
        "fold_acceptor_held_out",
        "acceptor_name_norm"
    ),
}


wen_independence_rows = []


for regime_name, (
    fold_col,
    group_col
) in wen_group_checks.items():

    for fold in [0, 1, 2]:

        train = wen_cv[
            wen_cv[fold_col] != fold
        ]

        test = wen_cv[
            wen_cv[fold_col] == fold
        ]


        overlap = (
            set(train[group_col].dropna())
            &
            set(test[group_col].dropna())
        )


        wen_independence_rows.append({

            "regime":
                regime_name,

            "fold":
                fold,

            "train_groups":
                train[group_col].nunique(),

            "test_groups":
                test[group_col].nunique(),

            "overlapping_groups":
                len(overlap),
        })


wen_independence_summary = pd.DataFrame(
    wen_independence_rows
)


display(
    wen_independence_summary
)

,regime,fold,train_groups,test_groups,overlapping_groups
0,D:A pair held-out,0,164,83,0
1,D:A pair held-out,1,165,82,0
2,D:A pair held-out,2,165,82,0
3,Donor held-out,0,43,17,0
4,Donor held-out,1,42,18,0
5,Donor held-out,2,35,25,0
6,Acceptor held-out,0,126,55,0
7,Acceptor held-out,1,118,63,0
8,Acceptor held-out,2,118,63,0


In [ ]:
# ------------------------------------------------------------------
# 8.8
# PCE distribution by validation fold
# ------------------------------------------------------------------

wen_target_rows = []


for regime_name, fold_col in (
    wen_validation_regimes.items()
):

    for fold in [0, 1, 2]:

        test = wen_cv[
            wen_cv[fold_col] == fold
        ]


        wen_target_rows.append({

            "regime":
                regime_name,

            "fold":
                fold,

            "n":
                len(test),

            "pce_mean":
                test["PCE (%)"].mean(),

            "pce_median":
                test["PCE (%)"].median(),

            "pce_std":
                test["PCE (%)"].std(),

            "pce_min":
                test["PCE (%)"].min(),

            "pce_max":
                test["PCE (%)"].max(),
        })


wen_fold_target_summary = pd.DataFrame(
    wen_target_rows
)


display(
    wen_fold_target_summary.round(3)
)

,regime,fold,n,pce_mean,pce_median,pce_std,pce_min,pce_max
0,Random,0,332,9.899,10.545,4.651,0.20,18.46
1,Random,1,331,9.925,10.870,4.504,0.19,18.51
2,Random,2,331,10.075,10.980,4.533,0.10,19.06
3,D:A pair held-out,0,332,10.204,11.200,4.376,0.60,19.06
4,D:A pair held-out,1,331,9.916,10.660,4.131,0.10,18.46
5,D:A pair held-out,2,331,9.778,10.430,5.116,0.23,18.51
6,Donor held-out,0,332,12.215,12.760,3.838,0.23,19.06
7,Donor held-out,1,331,8.028,8.300,4.140,0.28,17.91
8,Donor held-out,2,331,9.649,10.680,4.655,0.10,18.51
9,Acceptor held-out,0,332,9.890,11.225,4.844,0.23,17.91


In [ ]:
# ------------------------------------------------------------------
# 8.9
# Freeze Wen/Ma validation design
# ------------------------------------------------------------------

wen_cv.to_csv(
    INTERIM_DIR
    / "FINAL_wen_four_regime_cv_assignments.csv",
    index=False
)


wen_fold_size_summary.to_csv(
    INTERIM_DIR
    / "FINAL_wen_cv_fold_sizes.csv",
    index=False
)


wen_fold_target_summary.to_csv(
    INTERIM_DIR
    / "FINAL_wen_cv_target_summary.csv",
    index=False
)


print(
    "Wen/Ma four-regime validation design saved."
)

print(
    "Records:",
    len(wen_cv)
)

print(
    "Validation regimes:",
    len(wen_validation_regimes)
)

Wen/Ma four-regime validation design saved.
Records: 994
Validation regimes: 4


## 8.10 Frozen Structure and Processing Model Matrices

Three feature panels are reconstructed from the frozen audit manifest:
processing-only, structure-only, and combined structure–processing.

The six annealed conditions with unresolved duration retain their audit flag
but use zero as a numerical placeholder for `Annealing_Time_Physical` in the
model matrix. No other missing-value imputation is introduced.

The same observations, target values, validation folds, and model
hyperparameters are used for all three feature panels so that performance
differences can be attributed to the information supplied to the model.

In [ ]:
# ------------------------------------------------------------------
# 8.10
# Construct the three frozen Wen/Ma model matrices
# ------------------------------------------------------------------

X_wen_processing_df = (
    wen[processing_features]
    .apply(pd.to_numeric, errors="coerce")
    .copy()
)

X_wen_structure_df = (
    wen[structure_features]
    .apply(pd.to_numeric, errors="coerce")
    .copy()
)

X_wen_full_df = (
    wen[full_features]
    .apply(pd.to_numeric, errors="coerce")
    .copy()
)


# --------------------------------------------------------------
# Apply the audit-frozen unresolved-duration convention
# --------------------------------------------------------------

for X in [
    X_wen_processing_df,
    X_wen_full_df,
]:

    X["Annealing_Time_Physical"] = (
        X["Annealing_Time_Physical"]
        .fillna(0.0)
    )


y_wen = (
    pd.to_numeric(
        wen["PCE (%)"],
        errors="coerce"
    )
    .to_numpy(dtype=np.float64)
)


matrix_check_rows = []

for name, X in [
    ("processing", X_wen_processing_df),
    ("structure", X_wen_structure_df),
    ("full", X_wen_full_df),
]:

    matrix_check_rows.append({
        "panel": name,
        "records": len(X),
        "features": X.shape[1],
        "missing_cells": int(X.isna().sum().sum()),
        "infinite_cells": int(
            np.isinf(
                X.to_numpy(dtype=float)
            ).sum()
        ),
    })


wen_matrix_check = pd.DataFrame(
    matrix_check_rows
)

display(wen_matrix_check)

print(
    "Missing target values:",
    np.isnan(y_wen).sum()
)

,panel,records,features,missing_cells,infinite_cells
0,processing,994,20,0,0
1,structure,994,2008,0,0
2,full,994,2028,0,0


Missing target values: 0


In [ ]:
# ------------------------------------------------------------------
# 8.10B
# Final numerical arrays
# ------------------------------------------------------------------

X_wen_processing = (
    X_wen_processing_df
    .to_numpy(dtype=np.float32)
)

X_wen_structure = (
    X_wen_structure_df
    .to_numpy(dtype=np.float32)
)

X_wen_full = (
    X_wen_full_df
    .to_numpy(dtype=np.float32)
)


print("Processing:", X_wen_processing.shape)
print("Structure :", X_wen_structure.shape)
print("Full      :", X_wen_full.shape)
print("Target    :", y_wen.shape)

Processing: (994, 20)
Structure : (994, 2008)
Full      : (994, 2028)
Target    : (994,)


In [ ]:
# ------------------------------------------------------------------
# 8.11
# Wen/Ma target-only reference
# ------------------------------------------------------------------

wen_dummy_results = (
    evaluate_model_across_regimes(
        model=DummyRegressor(
            strategy="mean"
        ),
        model_name="Dummy mean",
        X=X_wen_processing,
        y=y_wen,
        cv_table=wen_cv,
        regimes=wen_validation_regimes,
    )
)


display(
    wen_dummy_results.round(3)
)


Dummy mean — Random
Fold 0: MAE=3.858 | RMSE=4.645 | R2=-0.000
Fold 1: MAE=3.809 | RMSE=4.497 | R2=-0.000
Fold 2: MAE=3.812 | RMSE=4.529 | R2=-0.001

Dummy mean — D:A pair held-out
Fold 0: MAE=3.695 | RMSE=4.384 | R2=-0.007
Fold 1: MAE=3.384 | RMSE=4.125 | R2=-0.000
Fold 2: MAE=4.417 | RMSE=5.117 | R2=-0.003

Dummy mean — Donor held-out
Fold 0: MAE=4.454 | RMSE=5.107 | R2=-0.776
Fold 1: MAE=4.051 | RMSE=5.053 | R2=-0.494
Fold 2: MAE=3.983 | RMSE=4.672 | R2=-0.010

Dummy mean — Acceptor held-out
Fold 0: MAE=4.039 | RMSE=4.838 | R2=-0.001
Fold 1: MAE=3.486 | RMSE=4.076 | R2=-0.054
Fold 2: MAE=4.049 | RMSE=4.831 | R2=-0.049


,model,regime,fold,train_n,test_n,test_pce_mean,MAE,RMSE,R2
0,Dummy mean,Random,0,662,332,9.899,3.858,4.645,-0.000
1,Dummy mean,Random,1,663,331,9.925,3.809,4.497,-0.000
2,Dummy mean,Random,2,663,331,10.075,3.812,4.529,-0.001
3,Dummy mean,D:A pair held-out,0,662,332,10.204,3.695,4.384,-0.007
4,Dummy mean,D:A pair held-out,1,663,331,9.916,3.384,4.125,-0.000
5,Dummy mean,D:A pair held-out,2,663,331,9.778,4.417,5.117,-0.003
6,Dummy mean,Donor held-out,0,662,332,12.215,4.454,5.107,-0.776
7,Dummy mean,Donor held-out,1,663,331,8.028,4.051,5.053,-0.494
8,Dummy mean,Donor held-out,2,663,331,9.649,3.983,4.672,-0.010
9,Dummy mean,Acceptor held-out,0,662,332,9.890,4.039,4.838,-0.001


## 8.12 Nonlinear Structure–Processing Comparison

Extra Trees regression is used as the primary nonlinear comparison across the
three frozen feature panels.

A fixed `max_features=0.5` fraction is used rather than a square-root rule so
that each feature panel exposes the same proportion of its available
information to candidate splits. No panel-specific hyperparameter tuning is
performed.

The experiment therefore tests the incremental predictive information carried
by processing variables rather than differences caused by validation folds or
model selection.

In [ ]:
# ------------------------------------------------------------------
# 8.12
# Extra Trees across all feature panels and validation regimes
# ------------------------------------------------------------------

import time


wen_feature_panels = {
    "Processing only":
        X_wen_processing,

    "Structure only":
        X_wen_structure,

    "Structure + processing":
        X_wen_full,
}


wen_panel_result_frames = []


for panel_name, X_panel in (
    wen_feature_panels.items()
):

    print("\n" + "#" * 80)
    print(panel_name)
    print("#" * 80)

    start_time = time.perf_counter()


    model = ExtraTreesRegressor(
        n_estimators=200,
        max_features=0.5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )


    panel_results = (
        evaluate_model_across_regimes(
            model=model,
            model_name="Extra Trees",
            X=X_panel,
            y=y_wen,
            cv_table=wen_cv,
            regimes=wen_validation_regimes,
        )
    )


    panel_results[
        "feature_panel"
    ] = panel_name


    wen_panel_result_frames.append(
        panel_results
    )


    elapsed = (
        time.perf_counter()
        - start_time
    ) / 60


    print(
        f"\n{panel_name} runtime:",
        round(elapsed, 2),
        "minutes"
    )


wen_panel_results = pd.concat(
    wen_panel_result_frames,
    ignore_index=True
)


################################################################################
Processing only
################################################################################

Extra Trees — Random
Fold 0: MAE=1.725 | RMSE=2.319 | R2=0.751
Fold 1: MAE=1.856 | RMSE=2.756 | R2=0.625
Fold 2: MAE=1.783 | RMSE=2.399 | R2=0.719

Extra Trees — D:A pair held-out
Fold 0: MAE=2.575 | RMSE=3.205 | R2=0.462
Fold 1: MAE=3.149 | RMSE=4.040 | R2=0.041
Fold 2: MAE=2.512 | RMSE=3.274 | R2=0.589

Extra Trees — Donor held-out
Fold 0: MAE=3.186 | RMSE=3.949 | R2=-0.062
Fold 1: MAE=3.045 | RMSE=3.938 | R2=0.092
Fold 2: MAE=3.169 | RMSE=3.976 | R2=0.268

Extra Trees — Acceptor held-out
Fold 0: MAE=2.503 | RMSE=3.216 | R2=0.558
Fold 1: MAE=2.194 | RMSE=2.868 | R2=0.478
Fold 2: MAE=3.041 | RMSE=3.998 | R2=0.282

Processing only runtime: 0.12 minutes

################################################################################
Structure only
##############################################################

In [ ]:
# ------------------------------------------------------------------
# 8.13
# Aggregate performance across folds
# ------------------------------------------------------------------

wen_panel_summary = (
    wen_panel_results
    .groupby(
        [
            "feature_panel",
            "regime",
        ]
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
    )
    .reset_index()
)


display(
    wen_panel_summary.round(3)
)

,feature_panel,regime,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,Processing only,Acceptor held-out,2.580,0.428,3.361,0.579,0.439,0.142
1,Processing only,D:A pair held-out,2.745,0.351,3.506,0.463,0.364,0.287
2,Processing only,Donor held-out,3.134,0.077,3.954,0.020,0.100,0.165
3,Processing only,Random,1.788,0.066,2.491,0.232,0.698,0.066
4,Structure + processing,Acceptor held-out,2.570,0.197,3.336,0.416,0.451,0.072
5,Structure + processing,D:A pair held-out,2.585,0.300,3.379,0.420,0.445,0.037
6,Structure + processing,Donor held-out,3.157,0.210,3.890,0.342,0.118,0.244
7,Structure + processing,Random,1.027,0.099,1.679,0.218,0.862,0.039
8,Structure only,Acceptor held-out,2.633,0.117,3.419,0.330,0.422,0.052
9,Structure only,D:A pair held-out,2.611,0.428,3.403,0.546,0.439,0.057


In [ ]:
wen_r2_comparison = (
    wen_panel_summary
    .pivot(
        index="regime",
        columns="feature_panel",
        values="R2_mean"
    )
)


display(
    wen_r2_comparison.round(3)
)

feature_panel,Processing only,Structure + processing,Structure only
regime,,,
Acceptor held-out,0.439,0.451,0.422
D:A pair held-out,0.364,0.445,0.439
Donor held-out,0.100,0.118,0.097
Random,0.698,0.862,0.810


In [ ]:
wen_mae_comparison = (
    wen_panel_summary
    .pivot(
        index="regime",
        columns="feature_panel",
        values="MAE_mean"
    )
)


display(
    wen_mae_comparison.round(3)
)

feature_panel,Processing only,Structure + processing,Structure only
regime,,,
Acceptor held-out,2.580,2.570,2.633
D:A pair held-out,2.745,2.585,2.611
Donor held-out,3.134,3.157,3.188
Random,1.788,1.027,1.277


In [ ]:
# ------------------------------------------------------------------
# 8.14
# Incremental value of processing information
# ------------------------------------------------------------------

structure_summary = (
    wen_panel_summary[
        wen_panel_summary[
            "feature_panel"
        ] == "Structure only"
    ][
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
            "R2_mean",
        ]
    ]
    .rename(
        columns={
            "MAE_mean": "structure_MAE",
            "RMSE_mean": "structure_RMSE",
            "R2_mean": "structure_R2",
        }
    )
)


full_summary = (
    wen_panel_summary[
        wen_panel_summary[
            "feature_panel"
        ] == "Structure + processing"
    ][
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
            "R2_mean",
        ]
    ]
    .rename(
        columns={
            "MAE_mean": "full_MAE",
            "RMSE_mean": "full_RMSE",
            "R2_mean": "full_R2",
        }
    )
)


processing_value = (
    structure_summary
    .merge(
        full_summary,
        on="regime",
        validate="one_to_one"
    )
)


processing_value[
    "MAE_reduction_from_processing"
] = (
    processing_value[
        "structure_MAE"
    ]
    -
    processing_value[
        "full_MAE"
    ]
)


processing_value[
    "MAE_percent_reduction"
] = (
    100
    * processing_value[
        "MAE_reduction_from_processing"
    ]
    / processing_value[
        "structure_MAE"
    ]
)


processing_value[
    "R2_gain_from_processing"
] = (
    processing_value[
        "full_R2"
    ]
    -
    processing_value[
        "structure_R2"
    ]
)


display(
    processing_value.round(3)
)

,regime,structure_MAE,structure_RMSE,structure_R2,full_MAE,full_RMSE,full_R2,MAE_reduction_from_processing,MAE_percent_reduction,R2_gain_from_processing
0,Acceptor held-out,2.633,3.419,0.422,2.570,3.336,0.451,0.063,2.386,0.028
1,D:A pair held-out,2.611,3.403,0.439,2.585,3.379,0.445,0.026,1.014,0.005
2,Donor held-out,3.188,3.930,0.097,3.157,3.890,0.118,0.031,0.966,0.021
3,Random,1.277,1.974,0.810,1.027,1.679,0.862,0.249,19.523,0.051


In [ ]:
# ------------------------------------------------------------------
# 8.15
# Fold-level structure vs full comparison
# ------------------------------------------------------------------

structure_folds = (
    wen_panel_results[
        wen_panel_results[
            "feature_panel"
        ] == "Structure only"
    ][
        [
            "regime",
            "fold",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .rename(
        columns={
            "MAE": "structure_MAE",
            "RMSE": "structure_RMSE",
            "R2": "structure_R2",
        }
    )
)


full_folds = (
    wen_panel_results[
        wen_panel_results[
            "feature_panel"
        ] == "Structure + processing"
    ][
        [
            "regime",
            "fold",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .rename(
        columns={
            "MAE": "full_MAE",
            "RMSE": "full_RMSE",
            "R2": "full_R2",
        }
    )
)


wen_processing_fold_effect = (
    structure_folds
    .merge(
        full_folds,
        on=[
            "regime",
            "fold",
        ],
        validate="one_to_one"
    )
)


wen_processing_fold_effect[
    "MAE_reduction"
] = (
    wen_processing_fold_effect[
        "structure_MAE"
    ]
    -
    wen_processing_fold_effect[
        "full_MAE"
    ]
)


wen_processing_fold_effect[
    "R2_gain"
] = (
    wen_processing_fold_effect[
        "full_R2"
    ]
    -
    wen_processing_fold_effect[
        "structure_R2"
    ]
)


display(
    wen_processing_fold_effect.round(3)
)

,regime,fold,structure_MAE,structure_RMSE,structure_R2,full_MAE,full_RMSE,full_R2,MAE_reduction,R2_gain
0,Random,0,1.134,1.763,0.856,0.918,1.458,0.901,0.216,0.046
1,Random,1,1.312,2.078,0.787,1.112,1.893,0.823,0.199,0.036
2,Random,2,1.384,2.081,0.788,1.052,1.688,0.861,0.333,0.072
3,D:A pair held-out,0,2.306,3.120,0.490,2.338,3.131,0.487,-0.032,-0.003
4,D:A pair held-out,1,2.427,3.057,0.450,2.497,3.143,0.419,-0.069,-0.031
5,D:A pair held-out,2,3.100,4.032,0.377,2.919,3.864,0.428,0.181,0.051
6,Donor held-out,0,3.227,3.865,-0.017,3.173,3.774,0.030,0.054,0.048
7,Donor held-out,1,3.399,4.306,-0.085,3.358,4.275,-0.069,0.041,0.016
8,Donor held-out,2,2.938,3.619,0.394,2.940,3.621,0.393,-0.002,-0.001
9,Acceptor held-out,0,2.741,3.761,0.395,2.767,3.791,0.386,-0.026,-0.010


In [ ]:
# ------------------------------------------------------------------
# 8.16
# Save core structure-processing results
# ------------------------------------------------------------------

wen_panel_results.to_csv(
    PROCESSED_DIR
    / "wen_structure_processing_fold_results.csv",
    index=False
)

wen_panel_summary.to_csv(
    PROCESSED_DIR
    / "wen_structure_processing_summary.csv",
    index=False
)

processing_value.to_csv(
    PROCESSED_DIR
    / "wen_processing_incremental_value.csv",
    index=False
)

wen_processing_fold_effect.to_csv(
    PROCESSED_DIR
    / "wen_processing_fold_effect.csv",
    index=False
)

print(
    "Core Wen/Ma structure-processing results saved."
)

Core Wen/Ma structure-processing results saved.


# 9. Why Does Processing Help Mainly Under Random Validation?

Explicit processing information substantially improves prediction under random
row-wise validation but provides only modest incremental benefit under
chemically grouped validation.

A plausible explanation is that random splitting frequently places different
processing conditions belonging to the same donor–acceptor material system in
both training and test sets.

Under this scenario, structure-only models recognize familiar chemistry but
cannot distinguish processing conditions associated with the same molecular
system, whereas structure–processing models can learn condition-dependent
performance variation.

This hypothesis is tested directly by quantifying material-system overlap in
random cross-validation and comparing out-of-fold structure-only and
structure–processing errors according to whether the test material system has
already appeared in training.

In [ ]:
# ------------------------------------------------------------------
# 9.1
# Material-system overlap in random Wen/Ma cross-validation
# ------------------------------------------------------------------

random_pair_overlap_rows = []


for fold in [0, 1, 2]:

    train = wen_cv[
        wen_cv["fold_random"] != fold
    ]

    test = wen_cv[
        wen_cv["fold_random"] == fold
    ]


    train_pairs = set(
        train["nominal_DA_group"]
    )


    pair_seen = (
        test["nominal_DA_group"]
        .isin(train_pairs)
    )


    random_pair_overlap_rows.append({

        "fold":
            fold,

        "test_records":
            len(test),

        "pair_seen_records":
            int(pair_seen.sum()),

        "pair_unseen_records":
            int((~pair_seen).sum()),

        "pair_seen_percent":
            100 * pair_seen.mean(),
    })


wen_random_pair_overlap = pd.DataFrame(
    random_pair_overlap_rows
)


display(
    wen_random_pair_overlap.round(2)
)




print(
    "Mean percentage of random-CV test records "
    "whose D:A pair is already in training:",
    round(
        wen_random_pair_overlap[
            "pair_seen_percent"
        ].mean(),
        2
    ),
    "%"
)

,fold,test_records,pair_seen_records,pair_unseen_records,pair_seen_percent
0,0,332,293,39,88.25
1,1,331,282,49,85.20
2,2,331,273,58,82.48


Mean percentage of random-CV test records whose D:A pair is already in training: 85.31 %


In [ ]:
# ------------------------------------------------------------------
# 9.2
# Random-CV OOF predictions for structure vs full model
# ------------------------------------------------------------------

random_structure_oof = np.full(
    len(wen),
    np.nan,
    dtype=float
)

random_full_oof = np.full(
    len(wen),
    np.nan,
    dtype=float
)


for fold in [0, 1, 2]:

    print(
        f"Random fold {fold}..."
    )

    test_mask = (
        wen_cv["fold_random"]
        .to_numpy()
        == fold
    )

    train_mask = ~test_mask


    structure_model = ExtraTreesRegressor(
        n_estimators=200,
        max_features=0.5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )

    full_model = ExtraTreesRegressor(
        n_estimators=200,
        max_features=0.5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )


    structure_model.fit(
        X_wen_structure[train_mask],
        y_wen[train_mask]
    )

    full_model.fit(
        X_wen_full[train_mask],
        y_wen[train_mask]
    )


    random_structure_oof[
        test_mask
    ] = structure_model.predict(
        X_wen_structure[test_mask]
    )

    random_full_oof[
        test_mask
    ] = full_model.predict(
        X_wen_full[test_mask]
    )


print(
    "Missing structure OOF predictions:",
    np.isnan(
        random_structure_oof
    ).sum()
)

print(
    "Missing full OOF predictions:",
    np.isnan(
        random_full_oof
    ).sum()
)

Random fold 0...
Random fold 1...
Random fold 2...
Missing structure OOF predictions: 0
Missing full OOF predictions: 0


In [ ]:
# ------------------------------------------------------------------
# 9.3
# Training support for each random-CV test material system
# ------------------------------------------------------------------

random_oof_analysis = wen_cv[
    [
        "Name_Donor",
        "Name_Acceptor",
        "nominal_DA_group",
        "fold_random",
        "PCE (%)",
    ]
].copy()


random_oof_analysis[
    "structure_prediction"
] = random_structure_oof

random_oof_analysis[
    "full_prediction"
] = random_full_oof


random_oof_analysis[
    "structure_absolute_error"
] = np.abs(
    random_oof_analysis["PCE (%)"]
    -
    random_oof_analysis[
        "structure_prediction"
    ]
)

random_oof_analysis[
    "full_absolute_error"
] = np.abs(
    random_oof_analysis["PCE (%)"]
    -
    random_oof_analysis[
        "full_prediction"
    ]
)


random_oof_analysis[
    "processing_error_reduction"
] = (
    random_oof_analysis[
        "structure_absolute_error"
    ]
    -
    random_oof_analysis[
        "full_absolute_error"
    ]
)


training_pair_support = np.zeros(
    len(wen),
    dtype=int
)


for fold in [0, 1, 2]:

    train_mask = (
        wen_cv[
            "fold_random"
        ].to_numpy()
        != fold
    )

    test_mask = ~train_mask


    training_counts = (
        wen_cv.loc[
            train_mask,
            "nominal_DA_group"
        ]
        .value_counts()
    )


    training_pair_support[
        test_mask
    ] = (
        wen_cv.loc[
            test_mask,
            "nominal_DA_group"
        ]
        .map(training_counts)
        .fillna(0)
        .astype(int)
        .to_numpy()
    )


random_oof_analysis[
    "training_pair_records"
] = training_pair_support


random_oof_analysis[
    "pair_seen_in_training"
] = (
    random_oof_analysis[
        "training_pair_records"
    ] > 0
)


display(
    random_oof_analysis[
        [
            "Name_Donor",
            "Name_Acceptor",
            "fold_random",
            "training_pair_records",
            "pair_seen_in_training",
            "structure_absolute_error",
            "full_absolute_error",
            "processing_error_reduction",
        ]
    ]
    .head(20)
    .round(3)
)

,Name_Donor,Name_Acceptor,fold_random,training_pair_records,pair_seen_in_training,structure_absolute_error,full_absolute_error,processing_error_reduction
0,J61,BT-IC,0,0,False,0.685,0.595,0.090
1,J71,BT-IC,2,0,False,0.678,0.897,-0.219
2,J71,ITIC,0,0,False,0.956,0.264,0.692
3,J71,BDT-IC,1,0,False,1.729,1.454,0.275
4,J71,TTz1,2,4,True,0.959,0.362,0.597
5,J71,TTz1,0,3,True,0.208,0.990,-0.782
6,J71,TTz1,1,5,True,1.693,0.731,0.961
7,J71,TTz1,0,3,True,0.322,0.405,-0.083
8,J71,TTz1,2,4,True,2.161,1.667,0.494
9,J71,TTz1,0,3,True,0.562,0.178,0.384


In [ ]:
# ------------------------------------------------------------------
# 9.4
# Processing benefit for seen versus unseen material systems
# ------------------------------------------------------------------

seen_unseen_processing_value = (
    random_oof_analysis
    .groupby(
        "pair_seen_in_training"
    )
    .agg(
        records=(
            "PCE (%)",
            "size"
        ),

        structure_MAE=(
            "structure_absolute_error",
            "mean"
        ),

        full_MAE=(
            "full_absolute_error",
            "mean"
        ),

        mean_error_reduction=(
            "processing_error_reduction",
            "mean"
        ),

        median_error_reduction=(
            "processing_error_reduction",
            "median"
        ),
    )
    .reset_index()
)


seen_unseen_processing_value[
    "MAE_percent_reduction"
] = (
    100
    * seen_unseen_processing_value[
        "mean_error_reduction"
    ]
    /
    seen_unseen_processing_value[
        "structure_MAE"
    ]
)


display(
    seen_unseen_processing_value.round(3)
)

,pair_seen_in_training,records,structure_MAE,full_MAE,mean_error_reduction,median_error_reduction,MAE_percent_reduction
0,False,146,2.596,2.548,0.047,0.029,1.824
1,True,848,1.049,0.765,0.284,0.103,27.060


In [ ]:
# ------------------------------------------------------------------
# 9.4
# Processing benefit for seen versus unseen material systems
# ------------------------------------------------------------------

seen_unseen_processing_value = (
    random_oof_analysis
    .groupby(
        "pair_seen_in_training"
    )
    .agg(
        records=(
            "PCE (%)",
            "size"
        ),

        structure_MAE=(
            "structure_absolute_error",
            "mean"
        ),

        full_MAE=(
            "full_absolute_error",
            "mean"
        ),

        mean_error_reduction=(
            "processing_error_reduction",
            "mean"
        ),

        median_error_reduction=(
            "processing_error_reduction",
            "median"
        ),
    )
    .reset_index()
)


seen_unseen_processing_value[
    "MAE_percent_reduction"
] = (
    100
    * seen_unseen_processing_value[
        "mean_error_reduction"
    ]
    /
    seen_unseen_processing_value[
        "structure_MAE"
    ]
)


display(
    seen_unseen_processing_value.round(3)
)

,pair_seen_in_training,records,structure_MAE,full_MAE,mean_error_reduction,median_error_reduction,MAE_percent_reduction
0,False,146,2.596,2.548,0.047,0.029,1.824
1,True,848,1.049,0.765,0.284,0.103,27.060


In [ ]:
# ------------------------------------------------------------------
# 9.5
# Processing value by same-system training support
# ------------------------------------------------------------------

random_oof_analysis[
    "pair_support_bin"
] = pd.cut(
    random_oof_analysis[
        "training_pair_records"
    ],
    bins=[
        -0.5,
        0.5,
        2.5,
        5.5,
        10.5,
        np.inf,
    ],
    labels=[
        "0",
        "1–2",
        "3–5",
        "6–10",
        ">10",
    ],
)


pair_support_effect = (
    random_oof_analysis
    .groupby(
        "pair_support_bin",
        observed=True
    )
    .agg(
        records=(
            "PCE (%)",
            "size"
        ),

        unique_pairs=(
            "nominal_DA_group",
            "nunique"
        ),

        structure_MAE=(
            "structure_absolute_error",
            "mean"
        ),

        full_MAE=(
            "full_absolute_error",
            "mean"
        ),

        mean_error_reduction=(
            "processing_error_reduction",
            "mean"
        ),

        median_error_reduction=(
            "processing_error_reduction",
            "median"
        ),
    )
    .reset_index()
)


pair_support_effect[
    "MAE_percent_reduction"
] = (
    100
    * pair_support_effect[
        "mean_error_reduction"
    ]
    /
    pair_support_effect[
        "structure_MAE"
    ]
)


display(
    pair_support_effect.round(3)
)

,pair_support_bin,records,unique_pairs,structure_MAE,full_MAE,mean_error_reduction,median_error_reduction,MAE_percent_reduction
0,0,146,134,2.596,2.548,0.047,0.029,1.824
1,1–2,127,41,1.161,1.082,0.079,0.029,6.847
2,3–5,280,59,1.190,0.821,0.369,0.091,30.983
3,6–10,395,59,0.954,0.643,0.311,0.148,32.602
4,>10,46,10,0.705,0.605,0.101,0.078,14.282


In [ ]:
# ------------------------------------------------------------------
# 9.6
# Within-system performance variability
# ------------------------------------------------------------------

wen_pair_variability = (
    wen
    .groupby(
        [
            "donor_name_norm",
            "acceptor_name_norm",
        ]
    )
    .agg(
        records=(
            "PCE (%)",
            "size"
        ),

        pce_mean=(
            "PCE (%)",
            "mean"
        ),

        pce_std=(
            "PCE (%)",
            "std"
        ),

        pce_min=(
            "PCE (%)",
            "min"
        ),

        pce_max=(
            "PCE (%)",
            "max"
        ),
    )
    .reset_index()
)


wen_pair_variability[
    "pce_range"
] = (
    wen_pair_variability[
        "pce_max"
    ]
    -
    wen_pair_variability[
        "pce_min"
    ]
)


repeated_pairs = (
    wen_pair_variability[
        wen_pair_variability[
            "records"
        ] > 1
    ]
)


print(
    "D:A systems with >1 processing condition:",
    len(repeated_pairs),
    "/",
    len(wen_pair_variability)
)


print(
    "Records belonging to repeated systems:",
    int(
        repeated_pairs[
            "records"
        ].sum()
    ),
    "/",
    len(wen)
)


display(
    repeated_pairs[
        [
            "records",
            "pce_std",
            "pce_range",
        ]
    ]
    .describe()
    .round(3)
)

D:A systems with >1 processing condition: 123 / 247
Records belonging to repeated systems: 870 / 994


,records,pce_std,pce_range
count,123.000,123.000,123.000
mean,7.073,1.085,2.750
std,4.055,0.994,2.437
min,2.000,0.000,0.000
25%,3.000,0.485,1.220
50%,7.000,0.805,2.220
75%,10.000,1.321,3.475
max,19.000,5.852,12.710


In [ ]:
# ------------------------------------------------------------------
# 9.7
# Save processing-mechanism analysis
# ------------------------------------------------------------------

wen_random_pair_overlap.to_csv(
    PROCESSED_DIR
    / "wen_random_cv_pair_overlap.csv",
    index=False
)

random_oof_analysis.to_csv(
    PROCESSED_DIR
    / "wen_random_oof_processing_analysis.csv",
    index=False
)

seen_unseen_processing_value.to_csv(
    PROCESSED_DIR
    / "wen_processing_value_seen_vs_unseen_pairs.csv",
    index=False
)

pair_support_effect.to_csv(
    PROCESSED_DIR
    / "wen_processing_value_by_pair_support.csv",
    index=False
)

wen_pair_variability.to_csv(
    PROCESSED_DIR
    / "wen_pair_processing_variability.csv",
    index=False
)

print(
    "Processing-mechanism analysis saved."
)

Processing-mechanism analysis saved.


### 9.7 Interpretation of same-system processing support

The benefit of explicit processing information depends strongly on whether the
material system has previously been represented in training.

When a random-CV test record belongs to a completely unseen donor–acceptor
pair, adding processing information reduces MAE by only 1.8%. The benefit
remains modest when only one or two same-system training records are
available, but rises sharply once three or more processing conditions for the
same material pair are represented in training.

The largest reductions occur for systems with 3–5 and 6–10 same-pair training
records, where MAE decreases by approximately 31% and 33%, respectively.

The reduction is smaller for the >10-support category. This category contains
only 46 test observations from 10 material systems and already exhibits a low
structure-only MAE, limiting the additional error that processing information
can remove.

Thus, the relationship is not strictly monotonic with training support.
Rather, the results indicate that processing information is most valuable
after a moderate amount of material-system familiarity has been established.
For chemically unseen systems, processing descriptors alone provide little
additional predictive power.

In [ ]:
# ------------------------------------------------------------------
# 9.8A
# Aggregate processing benefit at pair-fold level
# ------------------------------------------------------------------

pair_fold_effect = (
    random_oof_analysis
    .groupby(
        [
            "fold_random",
            "nominal_DA_group",
            "pair_seen_in_training",
            "training_pair_records",
        ],
        dropna=False
    )
    .agg(
        test_records=(
            "PCE (%)",
            "size"
        ),

        structure_MAE=(
            "structure_absolute_error",
            "mean"
        ),

        full_MAE=(
            "full_absolute_error",
            "mean"
        ),

        mean_error_reduction=(
            "processing_error_reduction",
            "mean"
        ),
    )
    .reset_index()
)


print(
    "Pair-fold clusters:",
    len(pair_fold_effect)
)

display(
    pair_fold_effect.head(20).round(3)
)



# ------------------------------------------------------------------
# 9.8B
# Cluster bootstrap: seen vs unseen D:A systems
# ------------------------------------------------------------------

seen_effect = (
    pair_fold_effect.loc[
        pair_fold_effect[
            "pair_seen_in_training"
        ],
        "mean_error_reduction"
    ]
    .to_numpy()
)

unseen_effect = (
    pair_fold_effect.loc[
        ~pair_fold_effect[
            "pair_seen_in_training"
        ],
        "mean_error_reduction"
    ]
    .to_numpy()
)


observed_seen = seen_effect.mean()
observed_unseen = unseen_effect.mean()

observed_difference = (
    observed_seen
    - observed_unseen
)


rng = np.random.default_rng(42)

n_boot = 10000

bootstrap_differences = np.empty(
    n_boot,
    dtype=float
)


for i in range(n_boot):

    seen_sample = rng.choice(
        seen_effect,
        size=len(seen_effect),
        replace=True
    )

    unseen_sample = rng.choice(
        unseen_effect,
        size=len(unseen_effect),
        replace=True
    )

    bootstrap_differences[i] = (
        seen_sample.mean()
        - unseen_sample.mean()
    )


ci_low, ci_high = np.percentile(
    bootstrap_differences,
    [2.5, 97.5]
)


bootstrap_summary = pd.DataFrame({
    "quantity": [
        "seen-pair mean error reduction",
        "unseen-pair mean error reduction",
        "seen minus unseen",
        "95% CI lower",
        "95% CI upper",
    ],

    "value": [
        observed_seen,
        observed_unseen,
        observed_difference,
        ci_low,
        ci_high,
    ],
})


display(
    bootstrap_summary.round(4)
)



print(
    "Seen pair-fold clusters:",
    len(seen_effect)
)

print(
    "Unseen pair-fold clusters:",
    len(unseen_effect)
)

Pair-fold clusters: 436


,fold_random,nominal_DA_group,pair_seen_in_training,training_pair_records,test_records,structure_MAE,full_MAE,mean_error_reduction
0,0,"('c-2f', 'n3')",True,5,4,0.702,0.417,0.285
1,0,"('d18', 'tt-naph1')",True,5,1,0.934,0.716,0.218
2,0,"('dio-pbdb-t', 'y6')",True,1,1,0.182,1.420,-1.238
3,0,"('idt-bfo', 'p3ht')",False,0,2,1.418,1.307,0.111
4,0,"('idt-bfo', 'pbdb-t')",True,4,7,2.330,1.920,0.410
5,0,"('idt-bfo', 'pce-10')",True,1,1,0.439,0.454,-0.014
6,0,"('idt-bof', 'pbdb-t')",True,2,1,1.690,1.232,0.459
7,0,"('idt-bof', 'pce-10')",True,1,1,0.596,0.609,-0.013
8,0,"('j61', 'bt-ic')",False,0,1,0.685,0.595,0.090
9,0,"('j71', 'itic')",False,0,1,0.956,0.264,0.692


,quantity,value
0,seen-pair mean error reduction,0.3090
1,unseen-pair mean error reduction,0.0438
2,seen minus unseen,0.2652
3,95% CI lower,0.1454
4,95% CI upper,0.3930


Seen pair-fold clusters: 302
Unseen pair-fold clusters: 134


### 9.9 Unique-material-system bootstrap

The pair-fold bootstrap indicates a substantially larger processing benefit
for material systems already represented in training.

Because an individual donor–acceptor pair can contribute observations to more
than one random-CV fold, a stricter robustness analysis resamples complete
donor–acceptor systems rather than pair-fold combinations.

All test observations belonging to a sampled material system are retained
together during each bootstrap replicate. This preserves dependence among
observations originating from the same nominal chemistry.

In [ ]:
# ------------------------------------------------------------------
# 9.9A 
# Unique-D:A-pair bootstrap
# ------------------------------------------------------------------

# Summarize each material system once
pair_boot_data = (
    random_oof_analysis
    .groupby(
        "nominal_DA_group",
        dropna=False
    )
    .agg(
        seen_error_sum=(
            "processing_error_reduction",
            lambda x: x[
                random_oof_analysis.loc[
                    x.index,
                    "pair_seen_in_training"
                ]
            ].sum()
        ),

        seen_n=(
            "pair_seen_in_training",
            "sum"
        ),

        total_error_sum=(
            "processing_error_reduction",
            "sum"
        ),

        total_n=(
            "processing_error_reduction",
            "size"
        ),
    )
    .reset_index()
)


pair_boot_data[
    "unseen_error_sum"
] = (
    pair_boot_data[
        "total_error_sum"
    ]
    -
    pair_boot_data[
        "seen_error_sum"
    ]
)

pair_boot_data[
    "unseen_n"
] = (
    pair_boot_data[
        "total_n"
    ]
    -
    pair_boot_data[
        "seen_n"
    ]
)


seen_sum = pair_boot_data[
    "seen_error_sum"
].to_numpy(dtype=float)

seen_n = pair_boot_data[
    "seen_n"
].to_numpy(dtype=float)

unseen_sum = pair_boot_data[
    "unseen_error_sum"
].to_numpy(dtype=float)

unseen_n = pair_boot_data[
    "unseen_n"
].to_numpy(dtype=float)


n_pairs = len(pair_boot_data)

rng = np.random.default_rng(42)

n_boot = 10000

bootstrap_differences = []


for _ in range(n_boot):

    idx = rng.integers(
        0,
        n_pairs,
        size=n_pairs
    )

    boot_seen_n = seen_n[idx].sum()
    boot_unseen_n = unseen_n[idx].sum()

    if (
        boot_seen_n == 0
        or boot_unseen_n == 0
    ):
        continue

    seen_mean = (
        seen_sum[idx].sum()
        / boot_seen_n
    )

    unseen_mean = (
        unseen_sum[idx].sum()
        / boot_unseen_n
    )

    bootstrap_differences.append(
        seen_mean - unseen_mean
    )


bootstrap_differences = np.asarray(
    bootstrap_differences
)


ci_low, ci_high = np.percentile(
    bootstrap_differences,
    [2.5, 97.5]
)


observed_seen = (
    random_oof_analysis.loc[
        random_oof_analysis[
            "pair_seen_in_training"
        ],
        "processing_error_reduction"
    ].mean()
)

observed_unseen = (
    random_oof_analysis.loc[
        ~random_oof_analysis[
            "pair_seen_in_training"
        ],
        "processing_error_reduction"
    ].mean()
)


unique_pair_bootstrap_summary = pd.DataFrame({

    "quantity": [
        "seen-system mean error reduction",
        "unseen-system mean error reduction",
        "seen minus unseen",
        "95% CI lower",
        "95% CI upper",
        "successful bootstrap replicates",
    ],

    "value": [
        observed_seen,
        observed_unseen,
        observed_seen - observed_unseen,
        ci_low,
        ci_high,
        len(bootstrap_differences),
    ],
})


display(
    unique_pair_bootstrap_summary.round(4)
)

,quantity,value
0,seen-system mean error reduction,0.2839
1,unseen-system mean error reduction,0.0473
2,seen minus unseen,0.2366
3,95% CI lower,0.0854
4,95% CI upper,0.3991
5,successful bootstrap replicates,10000.0000


### 9.10 Processing-familiarity conclusion

Random row-wise validation places 85.31% of test observations in
donor–acceptor material systems already represented in the corresponding
training data.

Under these familiar-system conditions, explicit processing information
substantially improves prediction. At the observation level, adding processing
features reduces MAE by 27.06% for previously observed D:A systems but by only
1.82% for completely unseen systems.

The processing advantage is largest once several same-system processing
conditions are available in training, with approximately 31–33% MAE reduction
for systems represented by 3–10 training conditions.

A cluster bootstrap using D:A-pair × fold units yields a positive difference
between familiar- and unfamiliar-system processing benefit. A stricter
bootstrap in which complete unique D:A systems are resampled together confirms
the result: the mean reduction in absolute error is 0.2839 PCE points for
familiar systems and 0.0473 points for unseen systems, corresponding to a
difference of 0.2366 points with a 95% bootstrap confidence interval of
0.0854–0.3991.

Thus, processing variables contain substantial predictive information, but
their principal benefit is condition-level refinement within chemistry already
represented during training. Processing descriptors alone provide little
additional ability to extrapolate toward completely unseen donor–acceptor
systems.

In [ ]:
unique_pair_bootstrap_summary.to_csv(
    PROCESSED_DIR
    / "wen_unique_pair_bootstrap_processing_benefit.csv",
    index=False
)

print("Section 9 complete and bootstrap result saved.")

Section 9 complete and bootstrap result saved.


# 10. Model-Family Robustness

The structure–processing results obtained with Extra Trees are tested using a
fundamentally different predictive model.

Standardized L2-regularized Ridge regression is applied to the same frozen
processing-only, structure-only, and combined structure–processing feature
panels under exactly the same four validation regimes.

Feature scaling is performed separately within each training fold using a
scikit-learn pipeline, preventing information from the held-out fold from
entering preprocessing.

The purpose of this analysis is not hyperparameter optimization. It tests
whether the qualitative conclusion that processing information is most useful
within familiar material systems is robust to model family.

In [ ]:
# ------------------------------------------------------------------
# 10.1
# Standardized Ridge across all Wen/Ma panels and validation regimes
# ------------------------------------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import time


ridge_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(
            alpha=1.0,
            solver="lsqr"
        )
    ),
])


wen_ridge_panel_frames = []


for panel_name, X_panel in wen_feature_panels.items():

    print("\n" + "#" * 80)
    print("RIDGE —", panel_name)
    print("#" * 80)

    start_time = time.perf_counter()


    panel_results = (
        evaluate_model_across_regimes(
            model=ridge_pipeline,
            model_name="Standardized Ridge",
            X=X_panel,
            y=y_wen,
            cv_table=wen_cv,
            regimes=wen_validation_regimes,
        )
    )


    panel_results[
        "feature_panel"
    ] = panel_name


    wen_ridge_panel_frames.append(
        panel_results
    )


    elapsed = (
        time.perf_counter()
        - start_time
    ) / 60


    print(
        f"\n{panel_name} runtime:",
        round(elapsed, 2),
        "minutes"
    )


wen_ridge_results = pd.concat(
    wen_ridge_panel_frames,
    ignore_index=True
)


################################################################################
RIDGE — Processing only
################################################################################

Standardized Ridge — Random
Fold 0: MAE=3.052 | RMSE=3.676 | R2=0.373
Fold 1: MAE=2.951 | RMSE=3.696 | R2=0.324
Fold 2: MAE=3.000 | RMSE=3.636 | R2=0.354

Standardized Ridge — D:A pair held-out
Fold 0: MAE=2.950 | RMSE=3.739 | R2=0.268
Fold 1: MAE=3.661 | RMSE=4.462 | R2=-0.171
Fold 2: MAE=3.186 | RMSE=3.858 | R2=0.430

Standardized Ridge — Donor held-out
Fold 0: MAE=3.546 | RMSE=4.365 | R2=-0.297
Fold 1: MAE=3.272 | RMSE=4.240 | R2=-0.052
Fold 2: MAE=3.819 | RMSE=4.855 | R2=-0.091

Standardized Ridge — Acceptor held-out
Fold 0: MAE=3.317 | RMSE=3.952 | R2=0.332
Fold 1: MAE=2.842 | RMSE=3.804 | R2=0.082
Fold 2: MAE=3.539 | RMSE=4.201 | R2=0.207

Processing only runtime: 0.57 minutes

################################################################################
RIDGE — Structure only
###############

In [ ]:
# ------------------------------------------------------------------
# 10.2
# Ridge performance summary
# ------------------------------------------------------------------

wen_ridge_summary = (
    wen_ridge_results
    .groupby(
        [
            "feature_panel",
            "regime",
        ]
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
    )
    .reset_index()
)


display(
    wen_ridge_summary.round(3)
)

,feature_panel,regime,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,Processing only,Acceptor held-out,3.233,0.356,3.986,0.201,0.207,0.125
1,Processing only,D:A pair held-out,3.266,0.362,4.020,0.388,0.176,0.311
2,Processing only,Donor held-out,3.546,0.274,4.487,0.325,-0.147,0.132
3,Processing only,Random,3.001,0.051,3.670,0.031,0.351,0.025
4,Structure + processing,Acceptor held-out,4.352,0.525,6.040,1.100,-0.787,0.335
5,Structure + processing,D:A pair held-out,3.997,0.574,5.324,0.923,-0.423,0.522
6,Structure + processing,Donor held-out,4.138,0.674,5.460,0.893,-0.691,0.330
7,Structure + processing,Random,1.353,0.235,2.464,0.663,0.690,0.159
8,Structure only,Acceptor held-out,4.081,0.207,5.755,0.623,-0.634,0.154
9,Structure only,D:A pair held-out,4.084,0.742,5.393,0.896,-0.413,0.216


In [ ]:
# ------------------------------------------------------------------
# 10.3
# Ridge R2 comparison
# ------------------------------------------------------------------

wen_ridge_r2 = (
    wen_ridge_summary
    .pivot(
        index="regime",
        columns="feature_panel",
        values="R2_mean"
    )
)


display(
    wen_ridge_r2.round(3)
)

feature_panel,Processing only,Structure + processing,Structure only
regime,,,
Acceptor held-out,0.207,-0.787,-0.634
D:A pair held-out,0.176,-0.423,-0.413
Donor held-out,-0.147,-0.691,-0.441
Random,0.351,0.690,0.657


In [ ]:
# ------------------------------------------------------------------
# 10.4
# Ridge MAE comparison
# ------------------------------------------------------------------

wen_ridge_mae = (
    wen_ridge_summary
    .pivot(
        index="regime",
        columns="feature_panel",
        values="MAE_mean"
    )
)


display(
    wen_ridge_mae.round(3)
)

feature_panel,Processing only,Structure + processing,Structure only
regime,,,
Acceptor held-out,3.233,4.352,4.081
D:A pair held-out,3.266,3.997,4.084
Donor held-out,3.546,4.138,3.950
Random,3.001,1.353,1.459


In [ ]:
# ------------------------------------------------------------------
# 10.5
# Incremental processing value under Ridge
# ------------------------------------------------------------------

ridge_structure_summary = (
    wen_ridge_summary[
        wen_ridge_summary[
            "feature_panel"
        ] == "Structure only"
    ][
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
            "R2_mean",
        ]
    ]
    .rename(
        columns={
            "MAE_mean": "structure_MAE",
            "RMSE_mean": "structure_RMSE",
            "R2_mean": "structure_R2",
        }
    )
)


ridge_full_summary = (
    wen_ridge_summary[
        wen_ridge_summary[
            "feature_panel"
        ] == "Structure + processing"
    ][
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
            "R2_mean",
        ]
    ]
    .rename(
        columns={
            "MAE_mean": "full_MAE",
            "RMSE_mean": "full_RMSE",
            "R2_mean": "full_R2",
        }
    )
)


ridge_processing_value = (
    ridge_structure_summary
    .merge(
        ridge_full_summary,
        on="regime",
        validate="one_to_one"
    )
)


ridge_processing_value[
    "MAE_reduction_from_processing"
] = (
    ridge_processing_value[
        "structure_MAE"
    ]
    -
    ridge_processing_value[
        "full_MAE"
    ]
)


ridge_processing_value[
    "MAE_percent_reduction"
] = (
    100
    * ridge_processing_value[
        "MAE_reduction_from_processing"
    ]
    /
    ridge_processing_value[
        "structure_MAE"
    ]
)


ridge_processing_value[
    "R2_gain_from_processing"
] = (
    ridge_processing_value[
        "full_R2"
    ]
    -
    ridge_processing_value[
        "structure_R2"
    ]
)


display(
    ridge_processing_value.round(3)
)

,regime,structure_MAE,structure_RMSE,structure_R2,full_MAE,full_RMSE,full_R2,MAE_reduction_from_processing,MAE_percent_reduction,R2_gain_from_processing
0,Acceptor held-out,4.081,5.755,-0.634,4.352,6.040,-0.787,-0.271,-6.642,-0.153
1,D:A pair held-out,4.084,5.393,-0.413,3.997,5.324,-0.423,0.087,2.140,-0.010
2,Donor held-out,3.950,5.042,-0.441,4.138,5.460,-0.691,-0.188,-4.750,-0.251
3,Random,1.459,2.592,0.657,1.353,2.464,0.690,0.106,7.268,0.033


In [ ]:
# ------------------------------------------------------------------
# 10.6
# Extra Trees vs Ridge processing-effect comparison
# ------------------------------------------------------------------

extra_processing_effect = (
    processing_value[
        [
            "regime",
            "MAE_percent_reduction",
            "R2_gain_from_processing",
        ]
    ]
    .rename(
        columns={
            "MAE_percent_reduction":
                "ExtraTrees_MAE_percent_reduction",

            "R2_gain_from_processing":
                "ExtraTrees_R2_gain",
        }
    )
)


ridge_processing_effect = (
    ridge_processing_value[
        [
            "regime",
            "MAE_percent_reduction",
            "R2_gain_from_processing",
        ]
    ]
    .rename(
        columns={
            "MAE_percent_reduction":
                "Ridge_MAE_percent_reduction",

            "R2_gain_from_processing":
                "Ridge_R2_gain",
        }
    )
)


model_family_processing_effect = (
    extra_processing_effect
    .merge(
        ridge_processing_effect,
        on="regime",
        validate="one_to_one"
    )
)


display(
    model_family_processing_effect.round(3)
)

,regime,ExtraTrees_MAE_percent_reduction,ExtraTrees_R2_gain,Ridge_MAE_percent_reduction,Ridge_R2_gain
0,Acceptor held-out,2.386,0.028,-6.642,-0.153
1,D:A pair held-out,1.014,0.005,2.140,-0.010
2,Donor held-out,0.966,0.021,-4.750,-0.251
3,Random,19.523,0.051,7.268,0.033


In [ ]:
# ------------------------------------------------------------------
# 10.7
# Fold-level Ridge structure vs full comparison
# ------------------------------------------------------------------

ridge_structure_folds = (
    wen_ridge_results[
        wen_ridge_results[
            "feature_panel"
        ] == "Structure only"
    ][
        [
            "regime",
            "fold",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .rename(
        columns={
            "MAE": "structure_MAE",
            "RMSE": "structure_RMSE",
            "R2": "structure_R2",
        }
    )
)


ridge_full_folds = (
    wen_ridge_results[
        wen_ridge_results[
            "feature_panel"
        ] == "Structure + processing"
    ][
        [
            "regime",
            "fold",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .rename(
        columns={
            "MAE": "full_MAE",
            "RMSE": "full_RMSE",
            "R2": "full_R2",
        }
    )
)


ridge_fold_effect = (
    ridge_structure_folds
    .merge(
        ridge_full_folds,
        on=[
            "regime",
            "fold",
        ],
        validate="one_to_one"
    )
)


ridge_fold_effect[
    "MAE_reduction"
] = (
    ridge_fold_effect[
        "structure_MAE"
    ]
    -
    ridge_fold_effect[
        "full_MAE"
    ]
)


ridge_fold_effect[
    "R2_gain"
] = (
    ridge_fold_effect[
        "full_R2"
    ]
    -
    ridge_fold_effect[
        "structure_R2"
    ]
)


display(
    ridge_fold_effect.round(3)
)

,regime,fold,structure_MAE,structure_RMSE,structure_R2,full_MAE,full_RMSE,full_R2,MAE_reduction,R2_gain
0,Random,0,1.194,1.855,0.840,1.090,1.755,0.857,0.104,0.017
1,Random,1,1.511,2.660,0.650,1.426,2.569,0.674,0.084,0.024
2,Random,2,1.672,3.260,0.481,1.542,3.069,0.540,0.130,0.059
3,D:A pair held-out,0,3.847,4.721,-0.168,3.410,4.259,0.050,0.437,0.218
4,D:A pair held-out,1,3.490,5.047,-0.497,4.022,5.808,-0.983,-0.533,-0.485
5,D:A pair held-out,2,4.915,6.411,-0.575,4.558,5.904,-0.336,0.358,0.239
6,Donor held-out,0,3.610,4.742,-0.531,3.991,5.182,-0.829,-0.381,-0.298
7,Donor held-out,1,3.657,4.783,-0.339,3.550,4.740,-0.315,0.107,0.024
8,Donor held-out,2,4.584,5.601,-0.452,4.873,6.459,-0.931,-0.289,-0.479
9,Acceptor held-out,0,4.301,6.403,-0.753,4.579,6.440,-0.773,-0.279,-0.020


In [ ]:
# ------------------------------------------------------------------
# 10.8
# Save model-family robustness results
# ------------------------------------------------------------------

wen_ridge_results.to_csv(
    PROCESSED_DIR
    / "wen_ridge_panel_fold_results.csv",
    index=False
)

wen_ridge_summary.to_csv(
    PROCESSED_DIR
    / "wen_ridge_panel_summary.csv",
    index=False
)

ridge_processing_value.to_csv(
    PROCESSED_DIR
    / "wen_ridge_processing_incremental_value.csv",
    index=False
)

model_family_processing_effect.to_csv(
    PROCESSED_DIR
    / "wen_processing_effect_model_family_comparison.csv",
    index=False
)

ridge_fold_effect.to_csv(
    PROCESSED_DIR
    / "wen_ridge_processing_fold_effect.csv",
    index=False
)

print("Model-family robustness results saved.")

Model-family robustness results saved.


### 10.9 Nested regularization sensitivity

Fixed-alpha Ridge regression performs poorly for high-dimensional structure
representations under chemically grouped validation.

Because approximately 2,000 structural variables are modeled from only
~660 training observations in each outer fold, the initially selected
regularization strength (`alpha=1`) may be insufficient.

A nested regularization analysis is therefore performed for the
structure-only and structure–processing panels.

For every outer validation fold, Ridge regularization strength is selected
using only the corresponding training data. Inner validation preserves the
same grouping principle as the outer experiment: donor–acceptor pair, donor,
or acceptor groups remain intact during grouped hyperparameter selection.

This analysis distinguishes a genuine limitation of linear structure–
processing modeling from an artifact of an arbitrary regularization strength.

In [ ]:
# ------------------------------------------------------------------
# 10.9
# Expanded Ridge regularization grid
# ------------------------------------------------------------------

ridge_alpha_grid = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
    10000.0,
    100000.0,
    1000000.0,
]

print("Expanded alpha grid:")
print(ridge_alpha_grid)

Expanded alpha grid:
[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0]


In [ ]:
# ------------------------------------------------------------------
# 10.9A
# Nested group-aware Ridge tuning
# ------------------------------------------------------------------

from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    KFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


ridge_alpha_grid = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
    10000.0,
]


inner_group_columns = {
    "Random":
        None,

    "D:A pair held-out":
        "nominal_DA_group",

    "Donor held-out":
        "donor_name_norm",

    "Acceptor held-out":
        "acceptor_name_norm",
}


def evaluate_nested_ridge(
    X,
    panel_name
):

    result_rows = []


    for regime_name, fold_col in (
        wen_validation_regimes.items()
    ):

        print("\n" + "=" * 80)
        print(panel_name, "—", regime_name)
        print("=" * 80)


        group_col = inner_group_columns[
            regime_name
        ]


        for fold in [0, 1, 2]:

            test_mask = (
                wen_cv[
                    fold_col
                ].to_numpy()
                == fold
            )

            train_mask = ~test_mask


            X_train = X[train_mask]
            X_test = X[test_mask]

            y_train = y_wen[train_mask]
            y_test = y_wen[test_mask]


            pipeline = Pipeline([
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "ridge",
                    Ridge(
                        solver="lsqr"
                    )
                ),
            ])


            # --------------------------------------------------
            # Inner validation design
            # --------------------------------------------------

            if group_col is None:

                inner_cv = KFold(
                    n_splits=3,
                    shuffle=True,
                    random_state=42
                )

                fit_kwargs = {}

            else:

                groups_train = (
                    wen_cv.loc[
                        train_mask,
                        group_col
                    ]
                    .to_numpy()
                )

                inner_cv = GroupKFold(
                    n_splits=3
                )

                fit_kwargs = {
                    "groups":
                        groups_train
                }


            search = GridSearchCV(
                estimator=pipeline,

                param_grid={
                    "ridge__alpha":
                        ridge_alpha_grid
                },

                scoring=
                    "neg_mean_absolute_error",

                cv=inner_cv,

                n_jobs=-1,

                refit=True,
            )


            search.fit(
                X_train,
                y_train,
                **fit_kwargs
            )


            prediction = search.predict(
                X_test
            )


            mae = mean_absolute_error(
                y_test,
                prediction
            )

            rmse = root_mean_squared_error(
                y_test,
                prediction
            )

            r2 = r2_score(
                y_test,
                prediction
            )


            best_alpha = (
                search.best_params_[
                    "ridge__alpha"
                ]
            )


            result_rows.append({

                "feature_panel":
                    panel_name,

                "regime":
                    regime_name,

                "fold":
                    fold,

                "best_alpha":
                    best_alpha,

                "train_n":
                    len(y_train),

                "test_n":
                    len(y_test),

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2,
            })


            print(
                f"Fold {fold}: "
                f"alpha={best_alpha:g} | "
                f"MAE={mae:.3f} | "
                f"RMSE={rmse:.3f} | "
                f"R2={r2:.3f}"
            )


    return pd.DataFrame(
        result_rows
    )

In [ ]:
# ------------------------------------------------------------------
# 10.9B
# Nested Ridge — structure only
# ------------------------------------------------------------------

nested_ridge_structure = (
    evaluate_nested_ridge(
        X=X_wen_structure,
        panel_name="Structure only",
    )
)


Structure only — Random
Fold 0: alpha=100 | MAE=1.165 | RMSE=1.789 | R2=0.852
Fold 1: alpha=100 | MAE=1.465 | RMSE=2.386 | R2=0.719
Fold 2: alpha=100 | MAE=1.482 | RMSE=2.259 | R2=0.751

Structure only — D:A pair held-out
Fold 0: alpha=1000 | MAE=2.256 | RMSE=3.126 | R2=0.488
Fold 1: alpha=1000 | MAE=2.371 | RMSE=3.197 | R2=0.399
Fold 2: alpha=10000 | MAE=3.176 | RMSE=4.043 | R2=0.374

Structure only — Donor held-out
Fold 0: alpha=10000 | MAE=3.619 | RMSE=4.288 | R2=-0.252
Fold 1: alpha=10000 | MAE=3.412 | RMSE=4.293 | R2=-0.079
Fold 2: alpha=10 | MAE=4.174 | RMSE=4.801 | R2=-0.067

Structure only — Acceptor held-out
Fold 0: alpha=10000 | MAE=2.493 | RMSE=3.672 | R2=0.424
Fold 1: alpha=100 | MAE=2.711 | RMSE=3.339 | R2=0.293
Fold 2: alpha=10000 | MAE=2.899 | RMSE=3.632 | R2=0.407


In [ ]:
# ------------------------------------------------------------------
# 10.9C
# Nested Ridge — structure + processing
# ------------------------------------------------------------------

nested_ridge_full = (
    evaluate_nested_ridge(
        X=X_wen_full,
        panel_name="Structure + processing",
    )
)


Structure + processing — Random
Fold 0: alpha=100 | MAE=1.087 | RMSE=1.675 | R2=0.870
Fold 1: alpha=10 | MAE=1.423 | RMSE=2.455 | R2=0.702
Fold 2: alpha=100 | MAE=1.322 | RMSE=2.073 | R2=0.790

Structure + processing — D:A pair held-out
Fold 0: alpha=1000 | MAE=2.191 | RMSE=3.088 | R2=0.501
Fold 1: alpha=1000 | MAE=2.371 | RMSE=3.181 | R2=0.405
Fold 2: alpha=10000 | MAE=3.138 | RMSE=4.002 | R2=0.386

Structure + processing — Donor held-out
Fold 0: alpha=10000 | MAE=3.557 | RMSE=4.209 | R2=-0.207
Fold 1: alpha=10000 | MAE=3.358 | RMSE=4.243 | R2=-0.054
Fold 2: alpha=100 | MAE=3.646 | RMSE=4.216 | R2=0.177

Structure + processing — Acceptor held-out
Fold 0: alpha=10000 | MAE=2.461 | RMSE=3.629 | R2=0.437
Fold 1: alpha=100 | MAE=2.600 | RMSE=3.172 | R2=0.362
Fold 2: alpha=10000 | MAE=2.884 | RMSE=3.613 | R2=0.414


In [ ]:
# ------------------------------------------------------------------
# 10.10
# Nested Ridge summary
# ------------------------------------------------------------------

nested_ridge_results = pd.concat(
    [
        nested_ridge_structure,
        nested_ridge_full,
    ],
    ignore_index=True
)


nested_ridge_summary = (
    nested_ridge_results
    .groupby(
        [
            "feature_panel",
            "regime",
        ]
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),

        median_best_alpha=(
            "best_alpha",
            "median"
        ),
    )
    .reset_index()
)


display(
    nested_ridge_summary.round(3)
)

,feature_panel,regime,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,median_best_alpha
0,Structure + processing,Acceptor held-out,2.648,0.216,3.471,0.260,0.404,0.039,10000.0
1,Structure + processing,D:A pair held-out,2.567,0.503,3.424,0.503,0.431,0.061,1000.0
2,Structure + processing,Donor held-out,3.520,0.147,4.223,0.018,-0.028,0.193,10000.0
3,Structure + processing,Random,1.277,0.172,2.068,0.390,0.787,0.084,100.0
4,Structure only,Acceptor held-out,2.701,0.203,3.548,0.182,0.374,0.071,10000.0
5,Structure only,D:A pair held-out,2.601,0.501,3.455,0.510,0.420,0.060,1000.0
6,Structure only,Donor held-out,3.735,0.394,4.461,0.295,-0.132,0.104,10000.0
7,Structure only,Random,1.371,0.178,2.145,0.314,0.774,0.069,100.0


In [ ]:
# ------------------------------------------------------------------
# 10.10B
# Selected regularization strengths
# ------------------------------------------------------------------

display(
    nested_ridge_results[
        [
            "feature_panel",
            "regime",
            "fold",
            "best_alpha",
            "MAE",
            "R2",
        ]
    ]
    .sort_values(
        [
            "feature_panel",
            "regime",
            "fold",
        ]
    )
    .round(3)
)

,feature_panel,regime,fold,best_alpha,MAE,R2
21,Structure + processing,Acceptor held-out,0,10000.0,2.461,0.437
22,Structure + processing,Acceptor held-out,1,100.0,2.600,0.362
23,Structure + processing,Acceptor held-out,2,10000.0,2.884,0.414
15,Structure + processing,D:A pair held-out,0,1000.0,2.191,0.501
16,Structure + processing,D:A pair held-out,1,1000.0,2.371,0.405
17,Structure + processing,D:A pair held-out,2,10000.0,3.138,0.386
18,Structure + processing,Donor held-out,0,10000.0,3.557,-0.207
19,Structure + processing,Donor held-out,1,10000.0,3.358,-0.054
20,Structure + processing,Donor held-out,2,100.0,3.646,0.177
12,Structure + processing,Random,0,100.0,1.087,0.870


In [ ]:
# ------------------------------------------------------------------
# 10.11
# Processing contribution under nested Ridge
# ------------------------------------------------------------------

nested_structure_summary = (
    nested_ridge_summary[
        nested_ridge_summary[
            "feature_panel"
        ] == "Structure only"
    ][
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
            "R2_mean",
        ]
    ]
    .rename(
        columns={
            "MAE_mean":
                "structure_MAE",

            "RMSE_mean":
                "structure_RMSE",

            "R2_mean":
                "structure_R2",
        }
    )
)


nested_full_summary = (
    nested_ridge_summary[
        nested_ridge_summary[
            "feature_panel"
        ]
        == "Structure + processing"
    ][
        [
            "regime",
            "MAE_mean",
            "RMSE_mean",
            "R2_mean",
        ]
    ]
    .rename(
        columns={
            "MAE_mean":
                "full_MAE",

            "RMSE_mean":
                "full_RMSE",

            "R2_mean":
                "full_R2",
        }
    )
)


nested_ridge_processing_value = (
    nested_structure_summary
    .merge(
        nested_full_summary,
        on="regime",
        validate="one_to_one"
    )
)


nested_ridge_processing_value[
    "MAE_reduction_from_processing"
] = (
    nested_ridge_processing_value[
        "structure_MAE"
    ]
    -
    nested_ridge_processing_value[
        "full_MAE"
    ]
)


nested_ridge_processing_value[
    "MAE_percent_reduction"
] = (
    100
    * nested_ridge_processing_value[
        "MAE_reduction_from_processing"
    ]
    /
    nested_ridge_processing_value[
        "structure_MAE"
    ]
)


nested_ridge_processing_value[
    "R2_gain_from_processing"
] = (
    nested_ridge_processing_value[
        "full_R2"
    ]
    -
    nested_ridge_processing_value[
        "structure_R2"
    ]
)


display(
    nested_ridge_processing_value.round(3)
)

,regime,structure_MAE,structure_RMSE,structure_R2,full_MAE,full_RMSE,full_R2,MAE_reduction_from_processing,MAE_percent_reduction,R2_gain_from_processing
0,Acceptor held-out,2.701,3.548,0.374,2.648,3.471,0.404,0.053,1.958,0.030
1,D:A pair held-out,2.601,3.455,0.420,2.567,3.424,0.431,0.034,1.321,0.010
2,Donor held-out,3.735,4.461,-0.132,3.520,4.223,-0.028,0.215,5.746,0.105
3,Random,1.371,2.145,0.774,1.277,2.068,0.787,0.094,6.849,0.014


In [ ]:
# ------------------------------------------------------------------
# 10.12
# Extra Trees vs properly regularized Ridge
# ------------------------------------------------------------------

final_model_family_effect = (
    processing_value[
        [
            "regime",
            "MAE_percent_reduction",
            "R2_gain_from_processing",
        ]
    ]
    .rename(
        columns={
            "MAE_percent_reduction":
                "ExtraTrees_MAE_reduction_pct",

            "R2_gain_from_processing":
                "ExtraTrees_R2_gain",
        }
    )
    .merge(
        nested_ridge_processing_value[
            [
                "regime",
                "MAE_percent_reduction",
                "R2_gain_from_processing",
            ]
        ]
        .rename(
            columns={
                "MAE_percent_reduction":
                    "NestedRidge_MAE_reduction_pct",

                "R2_gain_from_processing":
                    "NestedRidge_R2_gain",
            }
        ),
        on="regime",
        validate="one_to_one"
    )
)


display(
    final_model_family_effect.round(3)
)

,regime,ExtraTrees_MAE_reduction_pct,ExtraTrees_R2_gain,NestedRidge_MAE_reduction_pct,NestedRidge_R2_gain
0,Acceptor held-out,2.386,0.028,1.958,0.030
1,D:A pair held-out,1.014,0.005,1.321,0.010
2,Donor held-out,0.966,0.021,5.746,0.105
3,Random,19.523,0.051,6.849,0.014


In [ ]:
nested_ridge_results.to_csv(
    PROCESSED_DIR
    / "wen_nested_ridge_fold_results.csv",
    index=False
)

nested_ridge_summary.to_csv(
    PROCESSED_DIR
    / "wen_nested_ridge_summary.csv",
    index=False
)

nested_ridge_processing_value.to_csv(
    PROCESSED_DIR
    / "wen_nested_ridge_processing_value.csv",
    index=False
)

print(
    "Nested Ridge robustness analysis saved."
)

Nested Ridge robustness analysis saved.


### 10.12A Regularization-boundary conclusion

The initial nested Ridge analysis frequently selected `alpha = 10,000` under
chemically grouped validation. The regularization grid was therefore expanded
to include `100,000` and `1,000,000`.

Neither expanded value was selected in any outer fold, and the resulting
predictive metrics remained unchanged. The nested Ridge results are therefore
not artifacts of an insufficient upper regularization boundary.

### 10.12B Model-family robustness conclusion

Nested regularization substantially improves the high-dimensional Ridge
baseline relative to the arbitrary fixed-alpha model.

Extra Trees and properly regularized Ridge differ in absolute predictive
performance, but both reproduce the principal validation-dependent pattern:
the incremental value of processing information is greatest under random
row-wise validation and considerably smaller under chemically grouped
validation.

The magnitude of the processing benefit is model dependent. Nested Ridge
shows a larger donor-held-out improvement than Extra Trees, but the resulting
combined donor-held-out model remains poorly predictive, indicating that
processing information does not resolve unseen-donor extrapolation.

The central structure–processing conclusion is therefore qualitative rather
than model specific: explicit processing information is most effective when
the underlying chemistry is already represented in the training domain.

# 11. Predictive-Uncertainty Screening

Chemically grouped validation reveals substantial variation in predictive
difficulty that is not fully explained by molecular similarity, target-domain
shift, or model family.

This motivates a predictive-uncertainty analysis.

As an initial screening experiment, dispersion among trees in the Extra Trees
ensemble is used as a model-based uncertainty indicator. The analysis uses the
combined structure–processing representation and the same frozen validation
regimes used previously.

Tree-ensemble dispersion is not treated as a calibrated prediction interval.
Instead, this stage tests whether larger model disagreement is systematically
associated with larger out-of-fold prediction error.

Only if ensemble uncertainty contains useful error-ranking information will a
subsequent calibrated uncertainty method be justified.

In [ ]:
# ------------------------------------------------------------------
# 11.1
# OOF prediction + ensemble-dispersion uncertainty
# ------------------------------------------------------------------

from scipy.stats import pearsonr, spearmanr
import time


uncertainty_rows = []


for regime_name, fold_col in (
    wen_validation_regimes.items()
):

    print("\n" + "=" * 80)
    print("UNCERTAINTY —", regime_name)
    print("=" * 80)


    for fold in [0, 1, 2]:

        start_time = time.perf_counter()


        test_mask = (
            wen_cv[
                fold_col
            ].to_numpy()
            == fold
        )

        train_mask = ~test_mask


        model = ExtraTreesRegressor(
            n_estimators=200,
            max_features=0.5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        )


        model.fit(
            X_wen_full[train_mask],
            y_wen[train_mask]
        )


        # ----------------------------------------------------------
        # Predictions from every individual tree
        # ----------------------------------------------------------

        tree_predictions = np.vstack([
            tree.predict(
                X_wen_full[test_mask]
            )
            for tree in model.estimators_
        ])


        mean_prediction = (
            tree_predictions.mean(
                axis=0
            )
        )

        prediction_std = (
            tree_predictions.std(
                axis=0,
                ddof=1
            )
        )


        test_indices = np.flatnonzero(
            test_mask
        )


        for local_i, row_i in enumerate(
            test_indices
        ):

            observed = y_wen[row_i]

            predicted = (
                mean_prediction[
                    local_i
                ]
            )

            uncertainty = (
                prediction_std[
                    local_i
                ]
            )


            uncertainty_rows.append({

                "regime":
                    regime_name,

                "fold":
                    fold,

                "row_index":
                    int(row_i),

                "Name_Donor":
                    wen.iloc[
                        row_i
                    ]["Name_Donor"],

                "Name_Acceptor":
                    wen.iloc[
                        row_i
                    ]["Name_Acceptor"],

                "nominal_DA_group":
                    wen.iloc[
                        row_i
                    ]["nominal_DA_group"],

                "observed_pce":
                    observed,

                "predicted_pce":
                    predicted,

                "absolute_error":
                    abs(
                        observed
                        - predicted
                    ),

                "ensemble_std":
                    uncertainty,
            })


        elapsed = (
            time.perf_counter()
            - start_time
        )


        print(
            f"Fold {fold}: "
            f"{test_mask.sum()} records | "
            f"{elapsed:.1f} s"
        )


wen_uncertainty_oof = pd.DataFrame(
    uncertainty_rows
)


print("\nOOF uncertainty records:")
print(len(wen_uncertainty_oof))

print(
    "Expected:",
    len(wen)
    * len(
        wen_validation_regimes
    )
)

print(
    "Missing uncertainty values:",
    wen_uncertainty_oof[
        "ensemble_std"
    ].isna().sum()
)


UNCERTAINTY — Random
Fold 0: 332 records | 106.7 s
Fold 1: 331 records | 77.3 s
Fold 2: 331 records | 77.6 s

UNCERTAINTY — D:A pair held-out
Fold 0: 332 records | 53.8 s
Fold 1: 331 records | 28.1 s
Fold 2: 331 records | 34.7 s

UNCERTAINTY — Donor held-out
Fold 0: 332 records | 49.0 s
Fold 1: 331 records | 17.0 s
Fold 2: 331 records | 15.3 s

UNCERTAINTY — Acceptor held-out
Fold 0: 332 records | 15.3 s
Fold 1: 331 records | 17.4 s
Fold 2: 331 records | 15.1 s

OOF uncertainty records:
3976
Expected: 3976
Missing uncertainty values: 0


In [ ]:
# ------------------------------------------------------------------
# 11.2
# Association between ensemble dispersion and absolute error
# ------------------------------------------------------------------

uncertainty_association_rows = []


for regime_name in (
    wen_validation_regimes.keys()
):

    subset = (
        wen_uncertainty_oof[
            wen_uncertainty_oof[
                "regime"
            ] == regime_name
        ]
        .copy()
    )


    pearson_result = pearsonr(
        subset[
            "ensemble_std"
        ],
        subset[
            "absolute_error"
        ]
    )


    spearman_result = spearmanr(
        subset[
            "ensemble_std"
        ],
        subset[
            "absolute_error"
        ]
    )


    uncertainty_association_rows.append({

        "regime":
            regime_name,

        "records":
            len(subset),

        "mean_uncertainty":
            subset[
                "ensemble_std"
            ].mean(),

        "mean_absolute_error":
            subset[
                "absolute_error"
            ].mean(),

        "pearson_r":
            pearson_result.statistic,

        "pearson_p":
            pearson_result.pvalue,

        "spearman_rho":
            spearman_result.statistic,

        "spearman_p":
            spearman_result.pvalue,
    })


uncertainty_association = pd.DataFrame(
    uncertainty_association_rows
)


display(
    uncertainty_association.round(4)
)

,regime,records,mean_uncertainty,mean_absolute_error,pearson_r,pearson_p,spearman_rho,spearman_p
0,Random,994,0.8943,1.0272,0.6026,0.0000,0.4616,0.0000
1,D:A pair held-out,994,2.6471,2.5845,0.2403,0.0000,0.2766,0.0000
2,Donor held-out,994,3.0306,3.1571,-0.0190,0.5493,0.0466,0.1422
3,Acceptor held-out,994,2.6097,2.5700,0.2679,0.0000,0.2462,0.0000


In [ ]:
# ------------------------------------------------------------------
# 11.3
# Error across within-regime uncertainty quartiles
# ------------------------------------------------------------------

quartile_frames = []


for regime_name in (
    wen_validation_regimes.keys()
):

    subset = (
        wen_uncertainty_oof[
            wen_uncertainty_oof[
                "regime"
            ] == regime_name
        ]
        .copy()
    )


    subset[
        "uncertainty_quartile"
    ] = pd.qcut(
        subset[
            "ensemble_std"
        ],
        q=4,
        labels=[
            "Q1 lowest",
            "Q2",
            "Q3",
            "Q4 highest",
        ],
        duplicates="drop",
    )


    summary = (
        subset
        .groupby(
            "uncertainty_quartile",
            observed=True
        )
        .agg(
            records=(
                "absolute_error",
                "size"
            ),

            mean_uncertainty=(
                "ensemble_std",
                "mean"
            ),

            MAE=(
                "absolute_error",
                "mean"
            ),

            median_absolute_error=(
                "absolute_error",
                "median"
            ),
        )
        .reset_index()
    )


    summary[
        "regime"
    ] = regime_name


    quartile_frames.append(
        summary
    )


uncertainty_quartile_summary = (
    pd.concat(
        quartile_frames,
        ignore_index=True
    )
)


display(
    uncertainty_quartile_summary[
        [
            "regime",
            "uncertainty_quartile",
            "records",
            "mean_uncertainty",
            "MAE",
            "median_absolute_error",
        ]
    ].round(3)
)

,regime,uncertainty_quartile,records,mean_uncertainty,MAE,median_absolute_error
0,Random,Q1 lowest,249,0.172,0.452,0.279
1,Random,Q2,248,0.368,0.693,0.544
2,Random,Q3,248,0.755,0.864,0.593
3,Random,Q4 highest,249,2.279,2.098,1.580
4,D:A pair held-out,Q1 lowest,249,1.159,1.998,1.527
5,D:A pair held-out,Q2,248,2.322,1.875,1.340
6,D:A pair held-out,Q3,248,3.079,2.797,2.375
7,D:A pair held-out,Q4 highest,249,4.029,3.667,3.135
8,Donor held-out,Q1 lowest,249,1.566,3.087,2.178
9,Donor held-out,Q2,248,2.891,3.441,3.353


In [ ]:
# ------------------------------------------------------------------
# 11.4
# Risk enrichment in highest-uncertainty predictions
# ------------------------------------------------------------------

uncertainty_risk_rows = []


for regime_name in (
    wen_validation_regimes.keys()
):

    subset = (
        wen_uncertainty_oof[
            wen_uncertainty_oof[
                "regime"
            ] == regime_name
        ]
        .copy()
    )


    uncertainty_threshold = (
        subset[
            "ensemble_std"
        ].quantile(0.80)
    )


    high_uncertainty = (
        subset[
            subset[
                "ensemble_std"
            ] >= uncertainty_threshold
        ]
    )

    remaining = (
        subset[
            subset[
                "ensemble_std"
            ] < uncertainty_threshold
        ]
    )


    high_mae = (
        high_uncertainty[
            "absolute_error"
        ].mean()
    )

    remaining_mae = (
        remaining[
            "absolute_error"
        ].mean()
    )


    uncertainty_risk_rows.append({

        "regime":
            regime_name,

        "high_uncertainty_records":
            len(high_uncertainty),

        "high_uncertainty_MAE":
            high_mae,

        "remaining_MAE":
            remaining_mae,

        "MAE_risk_ratio":
            (
                high_mae
                / remaining_mae
            ),
    })


uncertainty_risk = pd.DataFrame(
    uncertainty_risk_rows
)


display(
    uncertainty_risk.round(3)
)

,regime,high_uncertainty_records,high_uncertainty_MAE,remaining_MAE,MAE_risk_ratio
0,Random,199,2.339,0.699,3.346
1,D:A pair held-out,199,3.773,2.287,1.650
2,Donor held-out,200,3.310,3.119,1.061
3,Acceptor held-out,199,3.195,2.414,1.324


In [ ]:
# ------------------------------------------------------------------
# 11.5
# Capture of largest prediction errors
# ------------------------------------------------------------------

error_capture_rows = []


for regime_name in (
    wen_validation_regimes.keys()
):

    subset = (
        wen_uncertainty_oof[
            wen_uncertainty_oof[
                "regime"
            ] == regime_name
        ]
        .copy()
    )


    n_top = max(
        1,
        int(
            np.ceil(
                0.20
                * len(subset)
            )
        )
    )


    high_uncertainty_indices = set(
        subset.nlargest(
            n_top,
            "ensemble_std"
        ).index
    )


    high_error_indices = set(
        subset.nlargest(
            n_top,
            "absolute_error"
        ).index
    )


    captured = len(
        high_uncertainty_indices
        &
        high_error_indices
    )


    error_capture_rows.append({

        "regime":
            regime_name,

        "top_fraction":
            0.20,

        "top_error_records":
            n_top,

        "captured_by_high_uncertainty":
            captured,

        "capture_percent":
            100
            * captured
            / n_top,

        "random_expected_percent":
            20.0,
    })


uncertainty_error_capture = (
    pd.DataFrame(
        error_capture_rows
    )
)


display(
    uncertainty_error_capture.round(2)
)

,regime,top_fraction,top_error_records,captured_by_high_uncertainty,capture_percent,random_expected_percent
0,Random,0.2,199,112,56.28,20.0
1,D:A pair held-out,0.2,199,71,35.68,20.0
2,Donor held-out,0.2,199,32,16.08,20.0
3,Acceptor held-out,0.2,199,55,27.64,20.0


In [ ]:
# ------------------------------------------------------------------
# 11.6
# Save uncertainty screening
# ------------------------------------------------------------------

wen_uncertainty_oof.to_csv(
    PROCESSED_DIR
    / "wen_extra_trees_uncertainty_oof.csv",
    index=False
)

uncertainty_association.to_csv(
    PROCESSED_DIR
    / "wen_uncertainty_error_association.csv",
    index=False
)

uncertainty_quartile_summary.to_csv(
    PROCESSED_DIR
    / "wen_uncertainty_quartile_summary.csv",
    index=False
)

uncertainty_risk.to_csv(
    PROCESSED_DIR
    / "wen_uncertainty_risk_enrichment.csv",
    index=False
)

uncertainty_error_capture.to_csv(
    PROCESSED_DIR
    / "wen_uncertainty_error_capture.csv",
    index=False
)

print(
    "Uncertainty screening saved."
)

Uncertainty screening saved.


### 11.7 Uncertainty-screening conclusion

Dispersion among Extra Trees ensemble members contains substantial information
about prediction reliability under random row-wise validation.

Random-CV ensemble uncertainty correlates strongly with absolute prediction
error, and the highest-uncertainty 20% of observations exhibit more than
three times the MAE of the remaining predictions while capturing 56.3% of the
largest 20% of prediction errors.

The uncertainty signal weakens as validation becomes chemically independent.
Moderate error-ranking ability remains under donor–acceptor-pair and
acceptor-held-out validation, but ensemble dispersion provides essentially no
useful error ranking under donor-held-out validation. In that regime,
uncertainty–error correlation is not statistically significant, the
high-uncertainty risk ratio approaches unity, and the highest-uncertainty 20%
captures fewer large errors than expected from random ranking.

Thus, raw ensemble dispersion should not be interpreted as a universally
reliable confidence measure. Chemical-domain extrapolation can simultaneously
degrade predictive accuracy and the model's ability to recognize its own
errors.

In [ ]:
# ------------------------------------------------------------------
# 12.1
# Finite-sample conformal quantile
# ------------------------------------------------------------------

def conformal_quantile(
    scores,
    alpha=0.10
):

    scores = np.asarray(
        scores,
        dtype=float
    )

    n = len(scores)

    level = min(
        1.0,
        np.ceil(
            (n + 1)
            * (1 - alpha)
        )
        / n
    )

    return np.quantile(
        scores,
        level,
        method="higher"
    )

In [ ]:
# ------------------------------------------------------------------
# 12.2
# Internal calibration split
# ------------------------------------------------------------------

calibration_group_columns = {

    "Random":
        None,

    "D:A pair held-out":
        "nominal_DA_group",

    "Donor held-out":
        "donor_name_norm",

    "Acceptor held-out":
        "acceptor_name_norm",
}


def make_internal_calibration_split(
    outer_train_indices,
    regime_name,
    fold
):

    outer_train_indices = np.asarray(
        outer_train_indices
    )


    group_col = (
        calibration_group_columns[
            regime_name
        ]
    )


    # ----------------------------------------------------------
    # Random regime:
    # one of five balanced random partitions becomes calibration
    # ----------------------------------------------------------

    if group_col is None:

        inner = KFold(
            n_splits=5,
            shuffle=True,
            random_state=1000 + fold
        )

        proper_rel, calibration_rel = next(
            inner.split(
                outer_train_indices
            )
        )


    # ----------------------------------------------------------
    # Chemical regimes:
    # preserve complete groups
    # ----------------------------------------------------------

    else:

        groups = (
            wen_cv.iloc[
                outer_train_indices
            ][group_col]
            .reset_index(
                drop=True
            )
        )


        assignments, _ = (
            make_balanced_group_folds(
                groups,
                n_splits=5,
                random_state=1000 + fold,
            )
        )


        calibration_rel = np.flatnonzero(
            assignments == 0
        )

        proper_rel = np.flatnonzero(
            assignments != 0
        )


    proper_indices = (
        outer_train_indices[
            proper_rel
        ]
    )

    calibration_indices = (
        outer_train_indices[
            calibration_rel
        ]
    )


    return (
        proper_indices,
        calibration_indices
    )

In [ ]:
# ------------------------------------------------------------------
# 12.3
# 90% calibrated prediction intervals
# ------------------------------------------------------------------

TARGET_COVERAGE = 0.90
ALPHA = 1.0 - TARGET_COVERAGE

interval_rows = []
interval_fold_rows = []


for regime_name, fold_col in (
    wen_validation_regimes.items()
):

    print("\n" + "=" * 80)
    print(
        "INTERVAL CALIBRATION —",
        regime_name
    )
    print("=" * 80)


    for fold in [0, 1, 2]:

        outer_test_mask = (
            wen_cv[
                fold_col
            ].to_numpy()
            == fold
        )


        outer_test_indices = (
            np.flatnonzero(
                outer_test_mask
            )
        )

        outer_train_indices = (
            np.flatnonzero(
                ~outer_test_mask
            )
        )


        (
            proper_indices,
            calibration_indices
        ) = make_internal_calibration_split(
            outer_train_indices,
            regime_name,
            fold,
        )


        model = ExtraTreesRegressor(
            n_estimators=200,
            max_features=0.5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        )


        model.fit(
            X_wen_full[
                proper_indices
            ],
            y_wen[
                proper_indices
            ]
        )


        # ------------------------------------------------------
        # Ensemble predictions — calibration
        # ------------------------------------------------------

        calibration_tree_predictions = (
            np.vstack([
                tree.predict(
                    X_wen_full[
                        calibration_indices
                    ]
                )
                for tree
                in model.estimators_
            ])
        )


        calibration_mean = (
            calibration_tree_predictions
            .mean(axis=0)
        )

        calibration_std = (
            calibration_tree_predictions
            .std(
                axis=0,
                ddof=1
            )
        )


        # ------------------------------------------------------
        # Ensemble predictions — outer test
        # ------------------------------------------------------

        test_tree_predictions = (
            np.vstack([
                tree.predict(
                    X_wen_full[
                        outer_test_indices
                    ]
                )
                for tree
                in model.estimators_
            ])
        )


        test_mean = (
            test_tree_predictions
            .mean(axis=0)
        )

        test_std = (
            test_tree_predictions
            .std(
                axis=0,
                ddof=1
            )
        )


        # ------------------------------------------------------
        # Constant-width calibration
        # ------------------------------------------------------

        calibration_residuals = np.abs(
            y_wen[
                calibration_indices
            ]
            -
            calibration_mean
        )


        q_constant = conformal_quantile(
            calibration_residuals,
            alpha=ALPHA
        )


        constant_lower = (
            test_mean
            - q_constant
        )

        constant_upper = (
            test_mean
            + q_constant
        )


        # ------------------------------------------------------
        # Ensemble-adaptive calibration
        # ------------------------------------------------------

        EPS = 1e-6

        calibration_scale = np.maximum(
            calibration_std,
            EPS
        )

        test_scale = np.maximum(
            test_std,
            EPS
        )


        normalized_scores = (
            calibration_residuals
            /
            calibration_scale
        )


        q_adaptive = conformal_quantile(
            normalized_scores,
            alpha=ALPHA
        )


        adaptive_half_width = (
            q_adaptive
            * test_scale
        )


        adaptive_lower = (
            test_mean
            - adaptive_half_width
        )

        adaptive_upper = (
            test_mean
            + adaptive_half_width
        )


        observed = y_wen[
            outer_test_indices
        ]


        constant_covered = (
            (observed >= constant_lower)
            &
            (observed <= constant_upper)
        )


        adaptive_covered = (
            (observed >= adaptive_lower)
            &
            (observed <= adaptive_upper)
        )


        # ------------------------------------------------------
        # Observation-level storage
        # ------------------------------------------------------

        for j, row_i in enumerate(
            outer_test_indices
        ):

            interval_rows.append({

                "regime":
                    regime_name,

                "fold":
                    fold,

                "row_index":
                    int(row_i),

                "observed_pce":
                    observed[j],

                "prediction":
                    test_mean[j],

                "absolute_error":
                    abs(
                        observed[j]
                        - test_mean[j]
                    ),

                "ensemble_std":
                    test_std[j],

                "constant_lower":
                    constant_lower[j],

                "constant_upper":
                    constant_upper[j],

                "constant_width":
                    constant_upper[j]
                    - constant_lower[j],

                "constant_covered":
                    constant_covered[j],

                "adaptive_lower":
                    adaptive_lower[j],

                "adaptive_upper":
                    adaptive_upper[j],

                "adaptive_width":
                    adaptive_upper[j]
                    - adaptive_lower[j],

                "adaptive_covered":
                    adaptive_covered[j],
            })


        interval_fold_rows.append({

            "regime":
                regime_name,

            "fold":
                fold,

            "proper_train_n":
                len(proper_indices),

            "calibration_n":
                len(calibration_indices),

            "test_n":
                len(outer_test_indices),

            "constant_coverage":
                constant_covered.mean(),

            "constant_mean_width":
                (
                    constant_upper
                    - constant_lower
                ).mean(),

            "adaptive_coverage":
                adaptive_covered.mean(),

            "adaptive_mean_width":
                (
                    adaptive_upper
                    - adaptive_lower
                ).mean(),

            "q_constant":
                q_constant,

            "q_adaptive":
                q_adaptive,
        })


        print(
            f"Fold {fold}: "
            f"constant coverage="
            f"{constant_covered.mean():.3f} | "
            f"adaptive coverage="
            f"{adaptive_covered.mean():.3f}"
        )


wen_interval_oof = pd.DataFrame(
    interval_rows
)

wen_interval_fold_summary = pd.DataFrame(
    interval_fold_rows
)


INTERVAL CALIBRATION — Random
Fold 0: constant coverage=0.904 | adaptive coverage=0.952
Fold 1: constant coverage=0.870 | adaptive coverage=0.915
Fold 2: constant coverage=0.903 | adaptive coverage=0.943

INTERVAL CALIBRATION — D:A pair held-out
Fold 0: constant coverage=0.955 | adaptive coverage=0.913
Fold 1: constant coverage=0.934 | adaptive coverage=0.789
Fold 2: constant coverage=0.900 | adaptive coverage=0.819

INTERVAL CALIBRATION — Donor held-out
Fold 0: constant coverage=0.792 | adaptive coverage=0.723
Fold 1: constant coverage=0.870 | adaptive coverage=0.888
Fold 2: constant coverage=0.918 | adaptive coverage=0.943

INTERVAL CALIBRATION — Acceptor held-out
Fold 0: constant coverage=0.949 | adaptive coverage=0.858
Fold 1: constant coverage=0.988 | adaptive coverage=0.991
Fold 2: constant coverage=0.982 | adaptive coverage=0.949


In [ ]:
# ------------------------------------------------------------------
# 12.4
# Overall interval performance
# ------------------------------------------------------------------

interval_summary_rows = []


for regime_name in (
    wen_validation_regimes.keys()
):

    subset = (
        wen_interval_oof[
            wen_interval_oof[
                "regime"
            ] == regime_name
        ]
    )


    for method in [
        "constant",
        "adaptive",
    ]:

        coverage = (
            subset[
                f"{method}_covered"
            ].mean()
        )

        mean_width = (
            subset[
                f"{method}_width"
            ].mean()
        )

        median_width = (
            subset[
                f"{method}_width"
            ].median()
        )


        interval_summary_rows.append({

            "regime":
                regime_name,

            "method":
                method,

            "nominal_coverage":
                TARGET_COVERAGE,

            "empirical_coverage":
                coverage,

            "coverage_error":
                coverage
                - TARGET_COVERAGE,

            "mean_interval_width":
                mean_width,

            "median_interval_width":
                median_width,
        })


wen_interval_summary = pd.DataFrame(
    interval_summary_rows
)


display(
    wen_interval_summary.round(3)
)

,regime,method,nominal_coverage,empirical_coverage,coverage_error,mean_interval_width,median_interval_width
0,Random,constant,0.9,0.892,-0.008,5.003,4.946
1,Random,adaptive,0.9,0.937,0.037,8.412,4.989
2,D:A pair held-out,constant,0.9,0.930,0.030,12.811,13.022
3,D:A pair held-out,adaptive,0.9,0.840,-0.060,11.773,11.749
4,Donor held-out,constant,0.9,0.860,-0.040,11.655,12.494
5,Donor held-out,adaptive,0.9,0.851,-0.049,12.626,13.146
6,Acceptor held-out,constant,0.9,0.973,0.073,16.621,15.862
7,Acceptor held-out,adaptive,0.9,0.933,0.033,15.398,13.368


In [ ]:
# ------------------------------------------------------------------
# 12.5
# Adaptive interval efficiency relative to constant width
# ------------------------------------------------------------------

constant_summary = (
    wen_interval_summary[
        wen_interval_summary[
            "method"
        ] == "constant"
    ]
    .set_index("regime")
)


adaptive_summary = (
    wen_interval_summary[
        wen_interval_summary[
            "method"
        ] == "adaptive"
    ]
    .set_index("regime")
)


interval_method_comparison = pd.DataFrame({

    "constant_coverage":
        constant_summary[
            "empirical_coverage"
        ],

    "adaptive_coverage":
        adaptive_summary[
            "empirical_coverage"
        ],

    "constant_mean_width":
        constant_summary[
            "mean_interval_width"
        ],

    "adaptive_mean_width":
        adaptive_summary[
            "mean_interval_width"
        ],
})


interval_method_comparison[
    "adaptive_width_change_percent"
] = (
    100
    * (
        interval_method_comparison[
            "adaptive_mean_width"
        ]
        -
        interval_method_comparison[
            "constant_mean_width"
        ]
    )
    /
    interval_method_comparison[
        "constant_mean_width"
    ]
)


display(
    interval_method_comparison.round(3)
)

,constant_coverage,adaptive_coverage,constant_mean_width,adaptive_mean_width,adaptive_width_change_percent
regime,,,,,
Random,0.892,0.937,5.003,8.412,68.138
D:A pair held-out,0.930,0.840,12.811,11.773,-8.105
Donor held-out,0.860,0.851,11.655,12.626,8.334
Acceptor held-out,0.973,0.933,16.621,15.398,-7.355


### 12.6 Prediction-interval conclusion

Prediction-interval calibration further demonstrates that uncertainty
reliability depends on the validation domain.

Constant-width split-conformal intervals achieve coverage close to the
nominal 90% level under random validation, whereas coverage becomes more
variable under chemically independent validation. In particular,
donor-held-out coverage falls to 86.0% overall and varies substantially across
individual donor folds.

Scaling interval width using Extra Trees ensemble dispersion does not provide
a universally superior solution. Under random validation, the adaptive
interval increases empirical coverage to 93.7% but requires intervals that
are approximately 68% wider than the constant-width alternative. Under
donor-held-out validation, adaptive intervals are wider while achieving
slightly lower coverage. Pair-held-out adaptive intervals are narrower but
substantially under-cover the nominal target.

Only acceptor-held-out validation shows a modest combination of reduced width
and acceptable aggregate coverage, although individual-fold variability
remains.

Together with the preceding error-ranking analysis, these results demonstrate
that ensemble disagreement can provide useful uncertainty information within
or near the training domain but should not be interpreted as a universally
reliable uncertainty scale for chemically novel materials.

Severe chemical-domain shift can therefore degrade both predictive accuracy
and the reliability of the model's own uncertainty estimates.

In [ ]:
# ------------------------------------------------------------------
# 12.7
# Save calibrated uncertainty analysis
# ------------------------------------------------------------------

wen_interval_oof.to_csv(
    PROCESSED_DIR
    / "wen_calibrated_interval_oof.csv",
    index=False
)

wen_interval_fold_summary.to_csv(
    PROCESSED_DIR
    / "wen_interval_fold_summary.csv",
    index=False
)

wen_interval_summary.to_csv(
    PROCESSED_DIR
    / "wen_interval_summary.csv",
    index=False
)

interval_method_comparison.to_csv(
    PROCESSED_DIR
    / "wen_interval_method_comparison.csv",
    index=False
)

print("Calibrated uncertainty analysis saved.")

Calibrated uncertainty analysis saved.


# 13. Cross-Dataset Transfer-Learning Feasibility

The original study concept included transfer of structure–performance knowledge
between the large OPV-DB corpus and the smaller processing-rich
Wen–Zhang–Ma benchmark.

Transfer learning is only scientifically defensible if both datasets can be
represented in a common molecular feature space without introducing severe
chemical-selection bias.

The preceding audit identified molecular-representation inconsistencies
between the datasets, including invalid Wen–Zhang–Ma acceptor SMILES for
several important modern non-fullerene acceptor systems.

Before fitting a transfer model, the final 994-condition Wen–Zhang–Ma cohort
is therefore evaluated for compatibility with an independently generated
RDKit molecular representation.

If common-representation coverage is strongly selective with respect to
chemistry or photovoltaic performance, naïve cross-dataset transfer will not
be pursued.

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.error")

In [ ]:
# ------------------------------------------------------------------
# 13.1
# RDKit compatibility of final Wen/Ma cohort
# ------------------------------------------------------------------

from rdkit import Chem


def rdkit_parseable(smiles):

    if pd.isna(smiles):
        return False

    smiles = str(smiles).strip()

    if not smiles:
        return False

    return (
        Chem.MolFromSmiles(smiles)
        is not None
    )


wen_transfer_audit = (
    wen[
        [
            "Name_Donor",
            "Smiles_Donor",
            "Name_Acceptor",
            "Smiles_Acceptor",
            "PCE (%)",
            "donor_name_norm",
            "acceptor_name_norm",
            "nominal_DA_group",
        ]
    ]
    .copy()
)


wen_transfer_audit[
    "donor_rdkit_parseable"
] = (
    wen_transfer_audit[
        "Smiles_Donor"
    ]
    .apply(rdkit_parseable)
)


wen_transfer_audit[
    "acceptor_rdkit_parseable"
] = (
    wen_transfer_audit[
        "Smiles_Acceptor"
    ]
    .apply(rdkit_parseable)
)


wen_transfer_audit[
    "both_rdkit_parseable"
] = (
    wen_transfer_audit[
        "donor_rdkit_parseable"
    ]
    &
    wen_transfer_audit[
        "acceptor_rdkit_parseable"
    ]
)


transfer_coverage_summary = pd.DataFrame({

    "criterion": [
        "donor parseable",
        "acceptor parseable",
        "both components parseable",
    ],

    "records": [
        int(
            wen_transfer_audit[
                "donor_rdkit_parseable"
            ].sum()
        ),

        int(
            wen_transfer_audit[
                "acceptor_rdkit_parseable"
            ].sum()
        ),

        int(
            wen_transfer_audit[
                "both_rdkit_parseable"
            ].sum()
        ),
    ],
})


transfer_coverage_summary[
    "percent_of_994"
] = (
    100
    * transfer_coverage_summary[
        "records"
    ]
    / len(wen_transfer_audit)
)


display(
    transfer_coverage_summary.round(2)
)

,criterion,records,percent_of_994
0,donor parseable,994,100.00
1,acceptor parseable,731,73.54
2,both components parseable,731,73.54


In [ ]:
# ------------------------------------------------------------------
# 13.2
# Compare retained and excluded performance distributions
# ------------------------------------------------------------------

transfer_subset_comparison = (
    wen_transfer_audit
    .groupby(
        "both_rdkit_parseable"
    )
    .agg(
        records=(
            "PCE (%)",
            "size"
        ),

        unique_donors=(
            "donor_name_norm",
            "nunique"
        ),

        unique_acceptors=(
            "acceptor_name_norm",
            "nunique"
        ),

        unique_pairs=(
            "nominal_DA_group",
            "nunique"
        ),

        pce_mean=(
            "PCE (%)",
            "mean"
        ),

        pce_median=(
            "PCE (%)",
            "median"
        ),

        pce_std=(
            "PCE (%)",
            "std"
        ),

        pce_min=(
            "PCE (%)",
            "min"
        ),

        pce_max=(
            "PCE (%)",
            "max"
        ),
    )
    .reset_index()
)


display(
    transfer_subset_comparison.round(3)
)

,both_rdkit_parseable,records,unique_donors,unique_acceptors,unique_pairs,pce_mean,pce_median,pce_std,pce_min,pce_max
0,False,263,36,21,57,12.023,13.20,4.776,0.23,18.51
1,True,731,35,160,190,9.226,9.96,4.244,0.10,19.06


In [ ]:
# ------------------------------------------------------------------
# 13.3
# Most common acceptors lost from common RDKit representation
# ------------------------------------------------------------------

excluded_acceptors = (
    wen_transfer_audit[
        ~wen_transfer_audit[
            "both_rdkit_parseable"
        ]
    ]
    .groupby(
        "Name_Acceptor"
    )
    .agg(
        records=(
            "PCE (%)",
            "size"
        ),

        pce_mean=(
            "PCE (%)",
            "mean"
        ),

        pce_median=(
            "PCE (%)",
            "median"
        ),

        pce_min=(
            "PCE (%)",
            "min"
        ),

        pce_max=(
            "PCE (%)",
            "max"
        ),
    )
    .reset_index()
    .sort_values(
        "records",
        ascending=False
    )
)


print(
    "Unique excluded acceptor systems:",
    len(excluded_acceptors)
)


display(
    excluded_acceptors
    .head(30)
    .round(3)
)

Unique excluded acceptor systems: 21


,Name_Acceptor,records,pce_mean,pce_median,pce_min,pce_max
18,Y6,134,12.031,12.955,0.23,17.62
11,L8-BO,28,10.301,10.885,4.24,17.68
12,MQ6,13,14.835,15.000,13.12,16.39
13,N3,12,13.320,13.795,7.76,17.61
19,m-TEH,12,17.750,17.675,17.18,18.51
7,C7BTP-BO-2Cl-2F,11,16.164,15.900,15.00,18.00
16,S-SubPc-PDI,11,3.964,4.060,2.93,4.53
10,E-SubPc-PDI,11,1.488,1.650,0.81,1.78
17,Se46,8,18.179,18.225,17.66,18.46
1,BT-LIC,5,12.052,12.150,10.40,13.20


In [ ]:
# ------------------------------------------------------------------
# 13.4
# High-PCE representation bias
# ------------------------------------------------------------------

performance_thresholds = [
    10,
    12,
    15,
    17,
]


high_pce_coverage_rows = []


for threshold in performance_thresholds:

    high_pce = (
        wen_transfer_audit[
            wen_transfer_audit[
                "PCE (%)"
            ] >= threshold
        ]
    )


    retained = (
        high_pce[
            "both_rdkit_parseable"
        ].sum()
    )


    high_pce_coverage_rows.append({

        "PCE_threshold":
            threshold,

        "high_PCE_records":
            len(high_pce),

        "transfer_compatible_records":
            int(retained),

        "percent_transfer_compatible":
            (
                100 * retained
                / len(high_pce)
                if len(high_pce)
                else np.nan
            ),
    })


high_pce_transfer_coverage = pd.DataFrame(
    high_pce_coverage_rows
)


display(
    high_pce_transfer_coverage.round(2)
)

,PCE_threshold,high_PCE_records,transfer_compatible_records,percent_transfer_compatible
0,10,569,365,64.15
1,12,391,220,56.27
2,15,121,46,38.02
3,17,56,21,37.50


### 13.5 Transfer-learning feasibility conclusion

A naïve cross-dataset transfer-learning experiment is not scientifically
justified using an independently generated RDKit molecular representation.

Although all 994 Wen–Zhang–Ma donor structures are RDKit-compatible, only
731 of 994 acceptor-containing device records (73.54%) can be represented
without modification. Exclusion is strongly non-random with respect to both
chemistry and photovoltaic performance.

The excluded 263 observations exhibit a substantially higher mean and median
PCE than the transfer-compatible subset and contain several prominent modern
non-fullerene acceptor systems, including Y6, L8-BO, MQ6, N3, and multiple
BTP-derived materials.

Representation coverage deteriorates further toward the high-performance
frontier. Only 38.02% of records with PCE >= 15% and 37.50% of records with
PCE >= 17% are compatible with the common RDKit representation.

Consequently, restricting transfer learning to RDKit-compatible observations
would preferentially remove modern high-performing OPV chemistry and create a
chemically selected benchmark.

Transfer learning is therefore not pursued merely to satisfy the original
study design. Instead, cross-dataset representation incompatibility is
retained as an explicit methodological finding and limitation. A valid future
transfer benchmark would require independently verified molecular
representations for the currently incompatible acceptor systems.

In [ ]:
# ------------------------------------------------------------------
# 13.6
# Save transfer-learning feasibility audit
# ------------------------------------------------------------------

wen_transfer_audit.to_csv(
    PROCESSED_DIR
    / "wen_transfer_compatibility_audit.csv",
    index=False
)

transfer_coverage_summary.to_csv(
    PROCESSED_DIR
    / "wen_transfer_coverage_summary.csv",
    index=False
)

transfer_subset_comparison.to_csv(
    PROCESSED_DIR
    / "wen_transfer_subset_bias.csv",
    index=False
)

excluded_acceptors.to_csv(
    PROCESSED_DIR
    / "wen_transfer_excluded_acceptors.csv",
    index=False
)

high_pce_transfer_coverage.to_csv(
    PROCESSED_DIR
    / "wen_transfer_high_pce_coverage.csv",
    index=False
)

print("Transfer-learning feasibility audit saved.")

Transfer-learning feasibility audit saved.


# 14. Physical Drivers of Processing-Aware Prediction

The preceding experiments establish that explicit processing information
improves photovoltaic-efficiency prediction, particularly within material
systems already represented during training.

The final analysis asks which categories of processing information carry this
predictive signal.

Because several processing variables form chemically related descriptor
blocks, individual-column impurity importance is not used. Instead, physically
related variables are treated as feature groups and jointly permuted in held-
out observations.

Importance is quantified by the increase in out-of-fold MAE after permutation.
A larger positive MAE increase indicates that the trained model relies more
strongly on that processing factor.

Permutation is performed only on test-fold features; models are never refitted
on permuted data.

In [ ]:
# ------------------------------------------------------------------
# 14.1
# Inspect frozen processing feature definitions
# ------------------------------------------------------------------

print(
    "Number of frozen processing features:",
    len(processing_features)
)

for i, feature in enumerate(
    processing_features,
    start=1
):
    print(
        f"{i:2d}. {feature}"
    )

Number of frozen processing features: 20
 1. D_A_Weight_Ratio
 2. Blend_Concentration (mg/ml)
 3. Solvent_DipoleMoment (Debye)
 4. Solvent_EnergyGap (eV)
 5. Solvent_Polarizability (a.u.)
 6. Additive_MeltingPoint (℃)
 7. Additive_BoilingPoint  (℃)
 8. Additive_Density (g/cm3)
 9. Additive_MolecularWeight
10. Additive_DipoleMoment  (Debye)
11. Additive_EnergyGap (eV)
12. Additive_Polarizability (a.u.)
13. Additive_Volume_Ratio (vol%)
14. Spin_Coating_Rate (rpm)
15. Annealing_Temperature(℃)
16. Annealing_Time_Physical
17. Active_Layer_Thickness (nm)
18. Additive_Present
19. Thermal_Annealing_Present
20. Annealing_Duration_Unresolved


In [ ]:
# ------------------------------------------------------------------
# 14.2
# Physically grouped processing variables
# ------------------------------------------------------------------

processing_groups = {

    "D:A ratio": [
        f for f in processing_features
        if f == "D_A_Weight_Ratio"
    ],

    "Blend concentration": [
        f for f in processing_features
        if f.startswith("Blend_Concentration")
    ],

    "Solvent chemistry": [
        f for f in processing_features
        if f.startswith("Solvent_")
    ],

    "Additive protocol": [
        f for f in processing_features
        if f.startswith("Additive_")
    ],

    "Spin coating": [
        f for f in processing_features
        if f.startswith("Spin_Coating")
    ],

    "Annealing protocol": [
        f for f in processing_features
        if (
            f.startswith("Annealing_")
            or f == "Thermal_Annealing_Present"
        )
    ],

    "Active-layer thickness": [
        f for f in processing_features
        if f.startswith("Active_Layer_Thickness")
    ],
}


for group_name, features in processing_groups.items():

    print("\n" + group_name)

    for feature in features:
        print("   ", feature)


D:A ratio
    D_A_Weight_Ratio

Blend concentration
    Blend_Concentration (mg/ml)

Solvent chemistry
    Solvent_DipoleMoment (Debye)
    Solvent_EnergyGap (eV)
    Solvent_Polarizability (a.u.)

Additive protocol
    Additive_MeltingPoint (℃)
    Additive_BoilingPoint  (℃)
    Additive_Density (g/cm3)
    Additive_MolecularWeight
    Additive_DipoleMoment  (Debye)
    Additive_EnergyGap (eV)
    Additive_Polarizability (a.u.)
    Additive_Volume_Ratio (vol%)
    Additive_Present

Spin coating
    Spin_Coating_Rate (rpm)

Annealing protocol
    Annealing_Temperature(℃)
    Annealing_Time_Physical
    Thermal_Annealing_Present
    Annealing_Duration_Unresolved

Active-layer thickness
    Active_Layer_Thickness (nm)


In [ ]:
# ------------------------------------------------------------------
# 14.3
# Validate processing-group definitions
# ------------------------------------------------------------------

grouped_features = [
    feature
    for features in processing_groups.values()
    for feature in features
]


missing_from_groups = sorted(
    set(processing_features)
    -
    set(grouped_features)
)


duplicated_between_groups = sorted({
    feature
    for feature in grouped_features
    if grouped_features.count(feature) > 1
})


print(
    "Frozen processing features:",
    len(processing_features)
)

print(
    "Grouped feature assignments:",
    len(grouped_features)
)

print(
    "Unique grouped features:",
    len(set(grouped_features))
)

print(
    "Missing from groups:",
    missing_from_groups
)

print(
    "Duplicated between groups:",
    duplicated_between_groups
)

Frozen processing features: 20
Grouped feature assignments: 20
Unique grouped features: 20
Missing from groups: []
Duplicated between groups: []


## 14.4 Grouped Permutation Importance

Physical processing factors are evaluated by jointly permuting all variables
belonging to a factor within each held-out test fold.

Columns belonging to the same physical factor receive the same row
permutation, preserving their internal descriptor relationships while
breaking their correspondence with the device outcome.

Importance is quantified as the increase in held-out MAE relative to the
unpermuted prediction.

The procedure is repeated ten times per processing group and fold. Models are
trained only once per fold; permutation affects test features only.

In [ ]:
# ------------------------------------------------------------------
# 14.4
# Grouped out-of-fold permutation importance
# ------------------------------------------------------------------

import time


full_feature_index = {
    feature: i
    for i, feature in enumerate(full_features)
}


group_column_indices = {

    group_name: [
        full_feature_index[feature]
        for feature in features
    ]

    for group_name, features
    in processing_groups.items()
}


N_PERMUTATIONS = 10

rng = np.random.default_rng(42)

permutation_rows = []


start_total = time.perf_counter()


for regime_name, fold_col in (
    wen_validation_regimes.items()
):

    print("\n" + "=" * 80)
    print(
        "PROCESSING IMPORTANCE —",
        regime_name
    )
    print("=" * 80)


    for fold in [0, 1, 2]:

        fold_start = time.perf_counter()


        test_mask = (
            wen_cv[
                fold_col
            ].to_numpy()
            == fold
        )

        train_mask = ~test_mask


        X_train = X_wen_full[
            train_mask
        ]

        X_test = X_wen_full[
            test_mask
        ]

        y_train = y_wen[
            train_mask
        ]

        y_test = y_wen[
            test_mask
        ]


        model = ExtraTreesRegressor(
            n_estimators=200,
            max_features=0.5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        )


        model.fit(
            X_train,
            y_train
        )


        baseline_prediction = model.predict(
            X_test
        )


        baseline_mae = mean_absolute_error(
            y_test,
            baseline_prediction
        )


        # ------------------------------------------------------
        # Permute each physical group
        # ------------------------------------------------------

        for (
            group_name,
            column_indices
        ) in group_column_indices.items():


            for repetition in range(
                N_PERMUTATIONS
            ):

                permutation = rng.permutation(
                    len(y_test)
                )


                X_permuted = X_test.copy()


                X_permuted[
                    :,
                    column_indices
                ] = (
                    X_test[
                        permutation
                    ][
                        :,
                        column_indices
                    ]
                )


                permuted_prediction = (
                    model.predict(
                        X_permuted
                    )
                )


                permuted_mae = (
                    mean_absolute_error(
                        y_test,
                        permuted_prediction
                    )
                )


                permutation_rows.append({

                    "regime":
                        regime_name,

                    "fold":
                        fold,

                    "processing_group":
                        group_name,

                    "repetition":
                        repetition,

                    "baseline_MAE":
                        baseline_mae,

                    "permuted_MAE":
                        permuted_mae,

                    "MAE_increase":
                        (
                            permuted_mae
                            - baseline_mae
                        ),

                    "MAE_percent_increase":
                        (
                            100
                            * (
                                permuted_mae
                                - baseline_mae
                            )
                            / baseline_mae
                        ),
                })


        elapsed = (
            time.perf_counter()
            - fold_start
        )


        print(
            f"Fold {fold}: "
            f"baseline MAE={baseline_mae:.3f} | "
            f"{elapsed:.1f} s"
        )


grouped_permutation_results = pd.DataFrame(
    permutation_rows
)


total_minutes = (
    time.perf_counter()
    - start_total
) / 60


print(
    "\nTotal grouped-permutation runtime:",
    round(total_minutes, 2),
    "minutes"
)

print(
    "Permutation evaluations:",
    len(grouped_permutation_results)
)


PROCESSING IMPORTANCE — Random
Fold 0: baseline MAE=0.918 | 83.2 s
Fold 1: baseline MAE=1.112 | 58.4 s
Fold 2: baseline MAE=1.052 | 76.6 s

PROCESSING IMPORTANCE — D:A pair held-out
Fold 0: baseline MAE=2.338 | 38.8 s
Fold 1: baseline MAE=2.497 | 30.6 s
Fold 2: baseline MAE=2.919 | 14.7 s

PROCESSING IMPORTANCE — Donor held-out
Fold 0: baseline MAE=3.173 | 14.2 s
Fold 1: baseline MAE=3.358 | 14.2 s
Fold 2: baseline MAE=2.940 | 19.7 s

PROCESSING IMPORTANCE — Acceptor held-out
Fold 0: baseline MAE=2.767 | 51.7 s
Fold 1: baseline MAE=2.374 | 14.1 s
Fold 2: baseline MAE=2.569 | 14.3 s

Total grouped-permutation runtime: 7.17 minutes
Permutation evaluations: 840


In [ ]:
# ------------------------------------------------------------------
# 14.5
# Aggregate grouped permutation importance
# ------------------------------------------------------------------

processing_importance_summary = (
    grouped_permutation_results
    .groupby(
        [
            "regime",
            "processing_group",
        ]
    )
    .agg(
        mean_MAE_increase=(
            "MAE_increase",
            "mean"
        ),

        std_MAE_increase=(
            "MAE_increase",
            "std"
        ),

        median_MAE_increase=(
            "MAE_increase",
            "median"
        ),

        mean_MAE_percent_increase=(
            "MAE_percent_increase",
            "mean"
        ),

        positive_fraction=(
            "MAE_increase",
            lambda x:
                np.mean(
                    np.asarray(x) > 0
                )
        ),
    )
    .reset_index()
)


display(
    processing_importance_summary
    .sort_values(
        [
            "regime",
            "mean_MAE_increase",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .round(4)
)

,regime,processing_group,mean_MAE_increase,std_MAE_increase,median_MAE_increase,mean_MAE_percent_increase,positive_fraction
2,Acceptor held-out,Annealing protocol,0.0502,0.0235,0.0504,1.9551,1.0000
1,Acceptor held-out,Additive protocol,0.0455,0.0349,0.0531,1.7995,0.8333
5,Acceptor held-out,Solvent chemistry,0.0145,0.0313,0.0280,0.4997,0.6333
6,Acceptor held-out,Spin coating,0.0075,0.0047,0.0079,0.2890,1.0000
4,Acceptor held-out,D:A ratio,0.0005,0.0035,-0.0002,0.0248,0.5000
0,Acceptor held-out,Active-layer thickness,-0.0007,0.0009,-0.0008,-0.0284,0.2333
3,Acceptor held-out,Blend concentration,-0.0010,0.0040,-0.0024,-0.0472,0.3333
12,D:A pair held-out,Solvent chemistry,0.0902,0.0883,0.0387,3.6003,1.0000
9,D:A pair held-out,Annealing protocol,0.0648,0.0231,0.0648,2.5613,1.0000
8,D:A pair held-out,Additive protocol,0.0207,0.0462,0.0053,0.6624,0.6000


In [ ]:
# ------------------------------------------------------------------
# 14.6
# Processing-factor rankings
# ------------------------------------------------------------------

processing_importance_summary[
    "importance_rank"
] = (
    processing_importance_summary
    .groupby(
        "regime"
    )[
        "mean_MAE_increase"
    ]
    .rank(
        ascending=False,
        method="dense"
    )
)


processing_importance_ranked = (
    processing_importance_summary
    .sort_values(
        [
            "regime",
            "importance_rank",
        ]
    )
)


display(
    processing_importance_ranked[
        [
            "regime",
            "importance_rank",
            "processing_group",
            "mean_MAE_increase",
            "mean_MAE_percent_increase",
            "positive_fraction",
        ]
    ]
    .round(3)
)

,regime,importance_rank,processing_group,mean_MAE_increase,mean_MAE_percent_increase,positive_fraction
2,Acceptor held-out,1.0,Annealing protocol,0.050,1.955,1.000
1,Acceptor held-out,2.0,Additive protocol,0.046,1.799,0.833
5,Acceptor held-out,3.0,Solvent chemistry,0.015,0.500,0.633
6,Acceptor held-out,4.0,Spin coating,0.008,0.289,1.000
4,Acceptor held-out,5.0,D:A ratio,0.000,0.025,0.500
0,Acceptor held-out,6.0,Active-layer thickness,-0.001,-0.028,0.233
3,Acceptor held-out,7.0,Blend concentration,-0.001,-0.047,0.333
12,D:A pair held-out,1.0,Solvent chemistry,0.090,3.600,1.000
9,D:A pair held-out,2.0,Annealing protocol,0.065,2.561,1.000
8,D:A pair held-out,3.0,Additive protocol,0.021,0.662,0.600


In [ ]:
# ------------------------------------------------------------------
# 14.7
# Cross-regime processing importance matrix
# ------------------------------------------------------------------

processing_importance_matrix = (
    processing_importance_summary
    .pivot(
        index="processing_group",
        columns="regime",
        values="mean_MAE_increase"
    )
)


display(
    processing_importance_matrix.round(3)
)

regime,Acceptor held-out,D:A pair held-out,Donor held-out,Random
processing_group,,,,
Active-layer thickness,-0.001,0.001,-0.000,0.002
Additive protocol,0.046,0.021,0.015,0.138
Annealing protocol,0.050,0.065,0.052,0.196
Blend concentration,-0.001,0.002,-0.000,0.006
D:A ratio,0.000,-0.002,-0.001,0.005
Solvent chemistry,0.015,0.090,0.058,0.067
Spin coating,0.008,-0.002,0.002,0.005


In [ ]:
# ------------------------------------------------------------------
# 14.8
# Save grouped processing importance
# ------------------------------------------------------------------

grouped_permutation_results.to_csv(
    PROCESSED_DIR
    / "wen_grouped_processing_permutation_raw.csv",
    index=False
)

processing_importance_summary.to_csv(
    PROCESSED_DIR
    / "wen_grouped_processing_importance_summary.csv",
    index=False
)

processing_importance_ranked.to_csv(
    PROCESSED_DIR
    / "wen_grouped_processing_importance_ranked.csv",
    index=False
)

print(
    "Grouped processing importance saved."
)

Grouped processing importance saved.


### 14.9 Processing-driver conclusion

Grouped permutation analysis shows that the predictive value of processing
information is strongly validation dependent.

Under random row-wise validation, annealing protocol is the dominant
processing factor: joint permutation increases MAE by approximately 19%.
Additive protocol is the second-largest contributor, increasing MAE by
approximately 14%, followed by solvent chemistry at approximately 6%.

These large processing effects contract substantially when evaluation is
chemically independent. Under donor–acceptor-pair and donor holdout, solvent
chemistry becomes the highest-ranked processing block, although its absolute
contribution remains modest. Annealing information also retains a positive
contribution across chemically grouped regimes.

Blend concentration, D:A ratio, spin-coating rate, and active-layer thickness
show little independent permutation importance once molecular structure and
the remaining processing variables are available to the model.

The results therefore distinguish condition-specific from more transferable
processing information. Annealing and additive descriptors carry substantial
predictive information within familiar chemistry, whereas solvent chemistry
and, to a lesser extent, annealing retain modest predictive value across
chemical domains.

These permutation effects quantify model reliance rather than causal physical
effects and should not be interpreted as experimental estimates of processing
causality.